In [ ]:
# Read the file and check the changes
source_path = "/path/to/tangram/mapping_utils.py"

with open(source_path, 'r') as f:
    content = f.read()

# Check ray import
if 'import ray' in content:
    print("✓ ray import found")
else:
    print("✗ ray import NOT found")

# Check hardcoded path is gone
if '/nfs/path/to/output/' in content:
    print("✗ hardcoded path still present")
else:
    print("✓ hardcoded path removed")

# Check new init
if 'ray.is_initialized()' in content:
    print("✓ ray.is_initialized() found")
else:
    print("✗ ray.is_initialized() NOT found")

# Show the relevant lines around line 624
lines = content.split('\n')
for i, line in enumerate(lines):
    if 'ray' in line.lower() and ('init' in line.lower() or 'import' in line.lower()):
        print(f"Line {i+1}: {line}")

In [ ]:
import scipy
scipy.inf = float('inf')

source_path = "/path/to/tangram/mapping_utils.py"

with open(source_path, 'r') as f:
    content = f.read()

# Fix the broken indentation
content = content.replace(
    "        if not ray.is_initialized():\n        ray.init(_temp_dir='/tmp/ray_tg')",
    "        if not ray.is_initialized():\n            ray.init(_temp_dir='/tmp/ray_tg')"
)

with open(source_path, 'w') as f:
    f.write(content)

# Verify lines around 624
lines = content.split('\n')
for i, line in enumerate(lines[620:630], start=621):
    print(f"Line {i}: {repr(line)}")

In [ ]:
import refined_tangram as tg
print("success!")

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import squidpy as sq
from scvi.external import Tangram

import os, re
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings("ignore") 

import scipy.sparse as sparse
from scipy.io import mmread
from scipy.stats import pearsonr, pointbiserialr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

import seaborn as sns
sc.logging.print_header()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
plt.rcParams["font.family"] = "Arial"
sns.set_style("white")

import random

import scipy.sparse as sp

from joblib import Parallel, delayed

import anndata as ad

from sklearn.neighbors import KDTree


In [ ]:
# Sections to use: "uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2", 
#                  "myelin_d1_1", "myelin_d3_4", "myelin_d4_3", "myelin_d5_1", "myelin_d7_11", "myelin_d9_9", "myelin_d14_11", 
#                  "liver_d1_2", "liver_d3_4", "liver_d4_1", "liver_d5_6", "liver_d7_14", "liver_d9_11", "liver_d14_11"


In [ ]:
save_dir = "/path/to/output/"
proj_dir = "/path/to/project/"

# ── Load all objects ──────────────────────────────────────────
print("Loading objects...")
# ad_sc         = sc.read_h5ad(f"{save_dir}fibroblasts_sc.h5ad")
# immune_only_sp  = sc.read_h5ad("/path/to/your/file.h5ad")
adata_subset  = sc.read_h5ad(f"{proj_dir}adata_subset_unified.h5ad")   # latest version with combined_tangram
# ad_map        = sc.read_h5ad(f"{save_dir}ad_map.h5ad")
# ad_atlas      = sc.read_h5ad(f"{save_dir}MZ_RNA.h5ad")
# adata_crush   = sc.read_h5ad(f"{save_dir}adata_crush_neighbors.h5ad")
# ad_map_immune = sc.read_h5ad(f"{save_dir}ad_map_immune.h5ad")
MZ_immune_cells = sc.read_h5ad("/path/to/your/file.h5ad")

print("Done")


In [ ]:
immune_only_sp  = sc.read_h5ad("/path/to/your/file.h5ad")


# Reconstruct the scRNAseq data

In [ ]:
import scanpy as sc
import scipy.io
import pandas as pd
import os

base = os.path.expanduser("~/Desktop/multiome/d9/tangram_export/")

counts   = scipy.io.mmread(base + "counts.mtx").T.tocsr()
barcodes = pd.read_csv(base + "barcodes.csv", index_col=0)
features = pd.read_csv(base + "features.csv", index_col=0)
metadata = pd.read_csv(base + "metadata.csv", index_col=0)

ad_sc = sc.AnnData(X=counts)
ad_sc.obs_names = barcodes.index.values   # use index directly
ad_sc.var_names = features.index.values   # use index directly
ad_sc.obs       = metadata

print(ad_sc)
ad_sc.write_h5ad(base + "fibroblasts_sc.h5ad")

# Import spatial data - subset of fibroblasts

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import tangram as tg
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KDTree

save_dir = "/path/to/output/"
proj_dir = "/path/to/project/"

# ── Load all objects ──────────────────────────────────────────
print("Loading objects...")
ad_sc         = sc.read_h5ad(f"{save_dir}fibroblasts_sc.h5ad")
immune_only_sp  = sc.read_h5ad("/path/to/your/file.h5ad")
adata_subset  = sc.read_h5ad(f"{proj_dir}adata_subset_unified.h5ad")   # latest version with combined_tangram
adata_crush   = sc.read_h5ad(f"{save_dir}adata_crush_neighbors.h5ad")
ad_map        = sc.read_h5ad(f"{save_dir}ad_map.h5ad")
ad_atlas      = sc.read_h5ad(f"{save_dir}MZ_RNA.h5ad")

print(f"ad_sc        : {ad_sc.shape}")
print(f"immune_only_sp : {immune_only_sp.shape}")
print(f"adata_subset : {adata_subset.shape}")
print(f"adata_crush  : {adata_crush.shape}")
print(f"ad_atlas     : {ad_atlas.shape}")

# ── Load precomputed results ──────────────────────────────────
neighborhood_df = pd.read_csv(f"{save_dir}neighborhood_raw.csv", index_col=0)
enrichment_df   = pd.read_csv(f"{save_dir}neighborhood_enrichment.csv", index_col=0)

# ── Reload key variables ──────────────────────────────────────
cell_pops         = sorted(ad_sc.obs["cell_pop"].unique().tolist())
exclude_celltypes = ["Fibroblasts"]
n_neighbors       = 10
tangram_threshold = 0.03

print("cell_pops:", cell_pops)

custom_palette = {
    "PvFib":         "#F0A03D",
    "Sox9+":         "#3F8758",
    "MyoFib_1":      "#C8C8F9",
    "Ly6a+":         "#31378A",
    "MyoFib_2":      "#4369DD",
    "Activated_Fib": "#EA8062",
    "Cxcl14+":       "#882017",
    "Mc5r+":         "#E83628",
}

# ── Rebuild adata_crush_unified ───────────────────────────────
crush_conditions = [c for c in adata_subset.obs["inj_type_per_day"].unique()
                    if c.startswith("crush") or c == "uninjured_uninjured"]
adata_crush_unified = adata_subset[
    adata_subset.obs["inj_type_per_day"].isin(crush_conditions)
].copy()
print(f"adata_crush_unified: {adata_crush_unified.shape[0]} spots")

# ── Rebuild combined_tangram_broad if missing ─────────────────
if "combined_tangram_broad" not in adata_crush_unified.obs.columns:
    print("Rebuilding combined_tangram_broad...")
    adata_crush_unified.obs["combined_tangram_broad"] = (
        adata_crush_unified.obs["combined_tangram"]
        .astype(str)
        .str.replace("_\d+$", "", regex=True)
        .str.replace("Neurons_D|Neurons_V", "Neurons", regex=True)
    )
    fibro_mask = adata_crush_unified.obs["fibro_subtype"].notna()
    adata_crush_unified.obs.loc[fibro_mask, "combined_tangram_broad"] = \
        adata_crush_unified.obs.loc[fibro_mask, "fibro_subtype"]

# ── Rebuild broad_types ───────────────────────────────────────
broad_types = sorted(set(
    adata_crush_unified.obs["combined_tangram_broad"]
    .unique().tolist()
) - set(cell_pops) - {"Fibroblasts", "nan"})
print(f"Broad types: {broad_types}")

# ── Rebuild KDTrees ───────────────────────────────────────────
print("\nBuilding KDTrees...")
tree         = KDTree(adata_crush.obsm["spatial"])
tree_unified = KDTree(adata_crush_unified.obsm["spatial"])
print(f"tree         : {adata_crush.shape[0]} spots")
print(f"tree_unified : {adata_crush_unified.shape[0]} spots")

# ── Phase map ─────────────────────────────────────────────────
phase_map = {
    "uninjured": "0_Uninjured",
    "d1": "1_Acute",    "d2": "1_Acute",    "d3": "1_Acute",
    "d4": "2_Subacute", "d5": "2_Subacute", "d7": "2_Subacute",
    "d9": "3_Late",     "d14": "3_Late",
}
phase_order  = ["0_Uninjured", "1_Acute", "2_Subacute", "3_Late"]
phase_labels = ["Uninjured", "Acute", "Subacute", "Late"]

print("\nAll objects loaded and ready!")

#
Run Tangram + check if the datasets are normalized

In [ ]:
import tangram as tg
import scanpy as sc
import pandas as pd
import scipy.io
import os

# Normalize (skip if already done)
sc.pp.normalize_total(ad_sc)
sc.pp.log1p(ad_sc)

# Choose the normalized assay 
immune_only_sp.X = immune_only_sp.layers["normalized"]

# Find shared genes between MERSCOPE panel and scRNAseq
tg.pp_adatas(ad_sc, immune_only_sp, genes=None)
print(f"Shared genes: {len(ad_sc.uns['overlap_genes'])}")


In [ ]:
# Run mapping (ad_map is the mapping matrix)
ad_map = tg.map_cells_to_space(
    adata_sc=ad_sc,
    adata_sp=immune_only_sp,
    mode="cells",
    density_prior="rna_count_based",
    num_epochs=300,
    device="cpu", 
    verbose=True
)

ad_map.write_h5ad(f"{save_dir}ad_map.h5ad")
print("ad_map saved!")

In [ ]:
# Project genes onto space (ad_ge is the projected gene expression on spatial coords)
ad_ge = tg.project_genes(adata_map=ad_map, adata_sc=ad_sc)


In [ ]:
print(ad_sc.obs.columns.tolist())  # find your cell type/subtype column


In [ ]:
print(ad_sc.obs["cell_pop"].value_counts())

In [ ]:
print(immune_only_sp.obsm.keys())  # spatial coords are often stored here


# Data visualization

## Per slide and per population

In [ ]:
def plot_tangram_section(adata_sp, ad_sc, ad_map, batch_id, annotation="cell_pop",
                          ncols=4, cmap="Reds", s=2, figsize_per_panel=(4, 4),
                          dpi=150, threshold=None, shared_scale=False, 
                          populations=None, save=None):
    import numpy as np
    import matplotlib.pyplot as plt

    tg.project_cell_annotations(ad_map, adata_sp, annotation=annotation)

    mask = adata_sp.obs["batch"] == batch_id
    adata_sub = adata_sp[mask].copy()
    df = adata_sub.obsm["tangram_ct_pred"]

    all_pops  = sorted(ad_sc.obs[annotation].unique().tolist())
    cell_pops = populations if populations is not None else all_pops
    cell_pops = [p for p in cell_pops if p in df.columns]
    if len(cell_pops) == 0:
        raise ValueError(f"None of {populations} found. Available: {all_pops}")

    ncols = min(ncols, len(cell_pops))
    nrows = int(np.ceil(len(cell_pops) / ncols))

    # Dynamic figure size based on grid and per-panel size
    figw = figsize_per_panel[0] * ncols
    figh = figsize_per_panel[1] * nrows
    fig, axs = plt.subplots(nrows, ncols, figsize=(figw, figh))

    if len(cell_pops) == 1:
        axs_flat = [axs]
    else:
        axs_flat = axs.flatten() if hasattr(axs, "flatten") else [axs]

    if shared_scale:
        all_scores = df[cell_pops].values.flatten()
        if threshold is not None:
            all_scores = all_scores[all_scores >= threshold]
        global_vmin = np.nanquantile(all_scores, 0.02)
        global_vmax = np.nanquantile(all_scores, 0.98)

    for i, pop in enumerate(cell_pops):
        scores = df[pop].values.copy()
        x = adata_sub.obs["x"].values
        y = adata_sub.obs["y"].values

        if threshold is not None:
            scores_plot = np.where(scores >= threshold, scores, np.nan)
        else:
            scores_plot = scores

        if shared_scale:
            vmin, vmax = global_vmin, global_vmax
        else:
            vmin = np.nanquantile(scores_plot, 0.02)
            vmax = np.nanquantile(scores_plot, 0.98)

        axs_flat[i].scatter(x, y, c="lightgrey", s=s, zorder=1)
        sc_plot = axs_flat[i].scatter(x, y, c=scores_plot, cmap=cmap, s=s,
                                       vmin=vmin, vmax=vmax, zorder=2)

        title = f"{pop}\n(thresh={threshold})" if threshold is not None else pop
        axs_flat[i].set_title(title, fontsize=11)
        axs_flat[i].axis("off")

        cbar = plt.colorbar(sc_plot, ax=axs_flat[i], fraction=0.046, pad=0.04)
        cbar.set_label("mapping score", fontsize=8)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_ticks([vmin, (vmin + vmax) / 2, vmax])
        cbar.set_ticklabels([f"{vmin:.3f}", f"{(vmin+vmax)/2:.3f}", f"{vmax:.3f}"])

    for j in range(i + 1, len(axs_flat)):
        axs_flat[j].axis("off")

    scale_label = "shared scale" if shared_scale else "individual scale"
    fig.suptitle(f"Section: {batch_id} ({scale_label})", fontsize=16, y=1.02)
    plt.tight_layout()

    if save:
        plt.savefig(save, dpi=dpi, bbox_inches="tight")
        print(f"Saved to {save}")
    plt.show()

In [ ]:
# ---- Only change these ----
batch             = "crush_d9_1"
threshold         = 0.003
shared_scale      = True
populations       = ["Ly6a+"]
dot_size          = 10 # size of each spot
figsize_per_panel = (8, 5)     # (width, height) per subplot panel
# ---------------------------

plot_tangram_section(
    adata_sp=immune_only_sp,
    ad_sc=ad_sc,
    ad_map=ad_map,
    batch_id=batch,
    threshold=threshold,
    shared_scale=shared_scale,
    populations=populations,
    s=dot_size,
    figsize_per_panel=figsize_per_panel,
    save=f"/path/to/output/"
)

## Time line with all the time point and all populations

In [ ]:
print(immune_only_sp.obs["inj_type_per_day"].unique().tolist())


In [ ]:
# ---- Only change these ----
selected_groups = ["uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2"]
# ---------------------------

def plot_timeline_per_population(adata_sp, ad_sc, ad_map, groups, annotation="cell_pop", 
                                  cmap="Reds", s=2, dpi=150, threshold=None, 
                                  shared_scale=True, save=None):
    """One plot per population, columns = timepoints"""
    import numpy as np
    import matplotlib.pyplot as plt

    tg.project_cell_annotations(ad_map, adata_sp, annotation=annotation)
    cell_pops = sorted(ad_sc.obs[annotation].unique().tolist())

    # Compute global scale if needed
    if shared_scale:
        all_scores = []
        for grp in groups:
            mask = adata_sp.obs["batch"] == grp
            df = adata_sp[mask].obsm["tangram_ct_pred"]
            vals = df[cell_pops].values.flatten()
            if threshold is not None:
                vals = vals[vals >= threshold]
            all_scores.append(vals)
        all_scores = np.concatenate(all_scores)
        global_vmin = np.nanquantile(all_scores, 0.02)
        global_vmax = np.nanquantile(all_scores, 0.98)

    nrows = len(cell_pops)
    ncols = len(groups)
    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    # ensure 2D array
    if nrows == 1: axs = axs[np.newaxis, :]
    if ncols == 1: axs = axs[:, np.newaxis]

    for row, pop in enumerate(cell_pops):
        for col, grp in enumerate(groups):
            mask = adata_sp.obs["batch"] == grp
            adata_sub = adata_sp[mask]
            df = adata_sub.obsm["tangram_ct_pred"]
            scores = df[pop].values.copy()
            x = adata_sub.obs["x"].values
            y = adata_sub.obs["y"].values

            if threshold is not None:
                scores_plot = np.where(scores >= threshold, scores, np.nan)
            else:
                scores_plot = scores

            if shared_scale:
                vmin, vmax = global_vmin, global_vmax
            else:
                vmin = np.nanquantile(scores_plot, 0.02)
                vmax = np.nanquantile(scores_plot, 0.98)

            axs[row, col].scatter(x, y, c="lightgrey", s=s, zorder=1)
            sc_plot = axs[row, col].scatter(x, y, c=scores_plot, cmap=cmap, s=s,
                                             vmin=vmin, vmax=vmax, zorder=2)
            axs[row, col].axis("off")

            # Labels
            if col == 0:
                axs[row, col].set_ylabel(pop, fontsize=10, rotation=90, labelpad=5)
                axs[row, col].axis("on")
                axs[row, col].set_yticks([])
                axs[row, col].set_xticks([])
                for spine in axs[row, col].spines.values():
                    spine.set_visible(False)
            if row == 0:
                axs[row, col].set_title(grp, fontsize=9, rotation=45, ha="right")

            # Colorbar only on last column
            if col == ncols - 1:
                cbar = plt.colorbar(sc_plot, ax=axs[row, col], fraction=0.046, pad=0.04)
                cbar.ax.tick_params(labelsize=6)
                cbar.set_ticks([vmin, vmax])
                cbar.set_ticklabels([f"{vmin:.3f}", f"{vmax:.3f}"])

    scale_label = "shared scale" if shared_scale else "individual scale"
    fig.suptitle(f"Fibroblast subtypes over time ({scale_label})", fontsize=14, y=1.01)
    plt.tight_layout()

    if save:
        plt.savefig(save, dpi=dpi, bbox_inches="tight")
        print(f"Saved to {save}")
    plt.show()


In [ ]:
# ---- Only change these ----
selected_groups = ["uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2"]
threshold    = 0.003
shared_scale = True
# ---------------------------

# Option A — one plot per population
plot_timeline_per_population(
    adata_sp=immune_only_sp, ad_sc=ad_sc, ad_map=ad_map,
    groups=selected_groups, threshold=threshold, shared_scale=shared_scale,
    save=f"/path/to/output/"
)

## Proportion of cells per time points

In [ ]:
custom_palette = {
    "PvFib":         "#F0A03D",
    "Sox9+":         "#3F8758",
    "MyoFib_1":      "#C8C8F9",
    "Ly6a+":         "#31378A",
    "MyoFib_2":      "#4369DD",
    "Activated_Fib": "#EA8062",
    "Cxcl14+":       "#882017",
    "Mc5r+":         "#E83628",
}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def compute_mean_scores(adata_sp, ad_sc, ad_map, groups, annotation="cell_pop", threshold=None):
    """Compute mean mapping score per population per group"""
    tg.project_cell_annotations(ad_map, adata_sp, annotation=annotation)
    cell_pops = sorted(ad_sc.obs[annotation].unique().tolist())
    
    records = []
    for grp in groups:
        mask = adata_sp.obs["inj_type_per_day"] == grp
        df = adata_sp[mask].obsm["tangram_ct_pred"]
        for pop in cell_pops:
            scores = df[pop].values
            if threshold is not None:
                scores = scores[scores >= threshold]
            records.append({
                "group": grp,
                "population": pop,
                "mean_score": np.mean(scores) if len(scores) > 0 else 0,
                "injury": grp.rsplit("_d", 1)[0],   # e.g. "crush"
                "day": int(grp.rsplit("_d", 1)[1])  # e.g. 5
            })
    
    return pd.DataFrame(records)


def plot_summary(adata_sp, ad_sc, ad_map, groups, annotation="cell_pop",
                 threshold=None, mode="both", split_injury=True,
                 palette=None, dpi=150, save=None):
    """
    mode        : "lineplot", "heatmap", or "both"
    split_injury: True  = one plot per injury type
                  False = all injury types together
    """
    df = compute_mean_scores(adata_sp, ad_sc, ad_map, groups, annotation, threshold)
    cell_pops = sorted(df["population"].unique().tolist())
    injuries  = sorted(df["injury"].unique().tolist())

    if palette is None:
        palette = dict(zip(cell_pops, sns.color_palette("tab10", len(cell_pops))))

    groups_to_plot = injuries if split_injury else ["all"]

    # ── LINEPLOT ──────────────────────────────────────────────────────────────
    if mode in ("lineplot", "both"):
        ncols = len(groups_to_plot)
        fig, axs = plt.subplots(1, ncols, figsize=(ncols * 6, 5), sharey=True)
        if ncols == 1:
            axs = [axs]

        for ax, inj in zip(axs, groups_to_plot):
            if inj == "all":
                sub = df.copy()
                sub["label"] = sub["group"]
            else:
                sub = df[df["injury"] == inj].copy()
                sub["label"] = sub["day"]

            for pop in cell_pops:
                pop_data = sub[sub["population"] == pop].sort_values("day" if inj != "all" else "group")
                ax.plot(
                    pop_data["label"],
                    pop_data["mean_score"],
                    marker="o", label=pop,
                    color=palette[pop], linewidth=2
                )

            ax.set_title(inj if inj != "all" else "All conditions", fontsize=13)
            ax.set_xlabel("Day post injury", fontsize=11)
            ax.set_ylabel("Mean mapping score", fontsize=11)
            ax.tick_params(axis="x", rotation=45)
            ax.spines[["top", "right"]].set_visible(False)

        # Single shared legend
        handles = [plt.Line2D([0], [0], color=palette[p], marker="o", label=p) for p in cell_pops]
        fig.legend(handles=handles, title="Population", bbox_to_anchor=(1.01, 0.5),
                   loc="center left", fontsize=9)
        title = f"Fibroblast subtype abundance over time"
        if threshold: title += f" (thresh={threshold})"
        fig.suptitle(title, fontsize=14, y=1.02)
        plt.tight_layout()

        if save:
            path = save.replace(".png", "_lineplot.png")
            plt.savefig(path, dpi=dpi, bbox_inches="tight")
            print(f"Saved to {path}")
        plt.show()

    # ── HEATMAP ───────────────────────────────────────────────────────────────
    if mode in ("heatmap", "both"):
        ncols = len(groups_to_plot)
        fig, axs = plt.subplots(1, ncols, figsize=(ncols * 6, 5))
        if ncols == 1:
            axs = [axs]

        for ax, inj in zip(axs, groups_to_plot):
            if inj == "all":
                sub = df.copy()
                col_label = "group"
            else:
                sub = df[df["injury"] == inj].copy()
                col_label = "day"

            pivot = sub.pivot_table(index="population", columns=col_label,
                                    values="mean_score").sort_index()
            pivot = pivot[sorted(pivot.columns)]  # sort timepoints

            sns.heatmap(
                pivot, ax=ax, cmap="Reds",
                annot=True, fmt=".3f", annot_kws={"size": 7},
                linewidths=0.5, cbar_kws={"label": "mean mapping score"}
            )
            ax.set_title(inj if inj != "all" else "All conditions", fontsize=13)
            ax.set_xlabel("Day post injury", fontsize=11)
            ax.set_ylabel("Population", fontsize=11)
            ax.tick_params(axis="x", rotation=45)
            ax.tick_params(axis="y", rotation=0)

        title = "Fibroblast subtype abundance over time"
        if threshold: title += f" (thresh={threshold})"
        fig.suptitle(title, fontsize=14, y=1.02)
        plt.tight_layout()

        if save:
            path = save.replace(".png", "_heatmap.png")
            plt.savefig(path, dpi=dpi, bbox_inches="tight")
            print(f"Saved to {path}")
        plt.show()

In [ ]:
# ---- Only change these ----
selected_groups = ["crush_d1", "crush_d2", "crush_d3", "crush_d4", "crush_d5", 
                   "crush_d7", "crush_d9", "crush_d14"]
threshold    = 0.003
mode         = "both"         # "lineplot", "heatmap", or "both"
split_injury = True           # True = one panel per injury, False = all together
# ---------------------------

plot_summary(
    adata_sp=immune_only_sp, ad_sc=ad_sc, ad_map=ad_map,
    groups=selected_groups,
    threshold=threshold,
    mode=mode,
    split_injury=split_injury,
    palette=custom_palette,
    save="/path/to/output/"
)

In [ ]:
# ---- Only change these ----
selected_groups = ["crush_d1", "crush_d2", "crush_d3", "crush_d4", "crush_d5",
                   "crush_d7", "crush_d9", "crush_d14", "uninjured_uninjured"]

phase_map = {
    "uninjured_uninjured": "0_Uninjured",
    "crush_d1":  "1_Acute",
    "crush_d2":  "1_Acute",
    "crush_d3":  "1_Acute",
    "crush_d4":  "2_Subacute",
    "crush_d5":  "2_Subacute",
    "crush_d7":  "2_Subacute",
    "crush_d9":  "3_Late",
    "crush_d14": "3_Late",
}

threshold    = 0
mode         = "both"
split_injury = True
# ---------------------------
def plot_summary_phased(adata_sp, ad_sc, ad_map, groups, phase_map,
                         annotation="cell_pop", threshold=None,
                         mode="both", split_injury=True,
                         palette=None, dpi=150, save=None):

    df = compute_mean_scores_phased(adata_sp, ad_sc, ad_map, groups, phase_map,
                                     annotation, threshold)
    cell_pops = sorted(df["population"].unique().tolist())
    injuries  = sorted(df["injury"].unique().tolist())

    if palette is None:
        palette = dict(zip(cell_pops, sns.color_palette("tab10", len(cell_pops))))

    groups_to_plot = injuries if split_injury else ["all"]

    # ── LINEPLOT ──────────────────────────────────────────────────────────────
    if mode in ("lineplot", "both"):
        ncols = len(groups_to_plot)
        fig, axs = plt.subplots(1, ncols, figsize=(ncols * 6, 5), sharey=True)
        if ncols == 1: axs = [axs]

        for ax, inj in zip(axs, groups_to_plot):
            sub = df if inj == "all" else df[df["injury"] == inj]

            # Use only phases present for this injury
            available_phases = sorted(sub["phase"].unique().tolist())
            available_labels = [p.split("_", 1)[1] for p in available_phases]

            for pop in cell_pops:
                pop_data = sub[sub["population"] == pop].sort_values("phase")
                if len(pop_data) == 0:
                    continue
                ax.plot(
                    available_labels,
                    pop_data["mean_score"].values,
                    marker="o", label=pop,
                    color=palette[pop], linewidth=2
                )

            ax.set_title(inj if inj != "all" else "All conditions", fontsize=13)
            ax.set_xlabel("Phase", fontsize=11)
            ax.set_ylabel("Mean mapping score", fontsize=11)
            ax.tick_params(axis="x", rotation=30)
            ax.spines[["top", "right"]].set_visible(False)

        handles = [plt.Line2D([0], [0], color=palette[p], marker="o", label=p) for p in cell_pops]
        fig.legend(handles=handles, title="Population", bbox_to_anchor=(1.01, 0.5),
                   loc="center left", fontsize=9)
        title = "Fibroblast subtype abundance by phase"
        if threshold: title += f" (thresh={threshold})"
        fig.suptitle(title, fontsize=14, y=1.02)
        plt.tight_layout()

        if save:
            path = save.replace(".png", "_lineplot.png")
            plt.savefig(path, dpi=dpi, bbox_inches="tight")
            print(f"Saved to {path}")
        plt.show()

    # ── HEATMAP ───────────────────────────────────────────────────────────────
    if mode in ("heatmap", "both"):
        ncols = len(groups_to_plot)
        fig, axs = plt.subplots(1, ncols, figsize=(ncols * 5, 5))
        if ncols == 1: axs = [axs]

        for ax, inj in zip(axs, groups_to_plot):
            sub = df if inj == "all" else df[df["injury"] == inj]

            pivot = sub.pivot_table(index="population", columns="phase",
                                    values="mean_score").sort_index()
            pivot = pivot[sorted(pivot.columns)]
            pivot.columns = [c.split("_", 1)[1] for c in pivot.columns]

            sns.heatmap(
                pivot, ax=ax, cmap="Reds",
                annot=True, fmt=".3f", annot_kws={"size": 8},
                linewidths=0.5, cbar_kws={"label": "mean mapping score"}
            )
            ax.set_title(inj if inj != "all" else "All conditions", fontsize=13)
            ax.set_xlabel("Phase", fontsize=11)
            ax.set_ylabel("Population", fontsize=11)
            ax.tick_params(axis="x", rotation=30)
            ax.tick_params(axis="y", rotation=0)

        title = "Fibroblast subtype abundance by phase"
        if threshold: title += f" (thresh={threshold})"
        fig.suptitle(title, fontsize=14, y=1.02)
        plt.tight_layout()

        if save:
            path = save.replace(".png", "_heatmap.pdf")
            plt.savefig(path, dpi=dpi, bbox_inches="tight")
            print(f"Saved to {path}")
        plt.show()

# Run
plot_summary_phased(
    adata_sp=immune_only_sp, ad_sc=ad_sc, ad_map=ad_map,
    groups=selected_groups,
    phase_map=phase_map,
    threshold=threshold,
    mode=mode,
    split_injury=split_injury,
    palette=custom_palette,
    save="/path/to/output/"
)

In [ ]:
def plot_proportions(adata_sp, ad_sc, ad_map, groups, phase_map=None,
                     annotation="cell_pop", threshold=None, use_phases=False,
                     split_injury=True, palette=None, dpi=150, save=None):

    df = compute_proportions(adata_sp, ad_sc, ad_map, groups, phase_map,
                              annotation, threshold, use_phases)
    cell_pops = sorted(df["population"].unique().tolist())
    injuries  = sorted(df["injury"].unique().tolist())

    if palette is None:
        palette = dict(zip(cell_pops, sns.color_palette("tab10", len(cell_pops))))

    groups_to_plot = injuries if split_injury else ["all"]

    ncols = len(groups_to_plot)
    fig, axs = plt.subplots(1, ncols, figsize=(ncols * 7, 5), sharey=True)
    if ncols == 1: axs = [axs]

    for ax, inj in zip(axs, groups_to_plot):
        sub = df if inj == "all" else df[df["injury"] == inj]

        label_order = sub.sort_values("label")["label"].unique().tolist()

        pivot = sub.pivot_table(index="label", columns="population",
                                values="proportion").reindex(label_order)
        pivot.columns.name = None
        available_pops = [p for p in cell_pops if p in pivot.columns]
        pivot = pivot[available_pops]

        # Renormalize rows to 100% in case of missing pops
        pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100

        bottom = np.zeros(len(pivot))
        for pop in available_pops:
            ax.bar(
                range(len(pivot)),
                pivot[pop].fillna(0).values,
                bottom=bottom,
                color=palette[pop],
                label=pop,
                width=0.7,
                edgecolor="white",
                linewidth=0.5
            )
            bottom += pivot[pop].fillna(0).values

        ax.set_xticks(range(len(pivot)))
        ax.set_xticklabels(
            [str(l).split("_", 1)[1] if "_" in str(l) else str(l) for l in label_order],
            rotation=45, ha="right", fontsize=9
        )
        ax.set_title(inj if inj != "all" else "All conditions", fontsize=13)
        ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
        ax.set_ylabel("% of fibroblast population", fontsize=11)
        ax.set_ylim(0, 100)
        ax.spines[["top", "right"]].set_visible(False)

    handles = [plt.Rectangle((0, 0), 1, 1, color=palette[p], label=p) for p in cell_pops]
    fig.legend(handles=handles, title="Population", bbox_to_anchor=(1.01, 0.5),
               loc="center left", fontsize=9)

    title = "Fibroblast subtype proportions"
    title += " by phase" if use_phases else " per timepoint"
    if threshold: title += f" (thresh={threshold})"
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()

    if save:
        plt.savefig(save, dpi=dpi, bbox_inches="tight")
        print(f"Saved to {save}")
    plt.show()

In [ ]:
# ---- Only change these ----
threshold    = 0.003
split_injury = True

# Per individual timepoint
plot_proportions(
    adata_sp=immune_only_sp, ad_sc=ad_sc, ad_map=ad_map,
    groups=selected_groups,
    phase_map=phase_map,
    threshold=threshold,
    use_phases=False,       # False = per timepoint
    split_injury=split_injury,
    palette=custom_palette,
    save="/path/to/output/"
)

# Per phase (pooled)
plot_proportions(
    adata_sp=immune_only_sp, ad_sc=ad_sc, ad_map=ad_map,
    groups=selected_groups,
    phase_map=phase_map,
    threshold=threshold,
    use_phases=True,        # True = pooled by phase
    split_injury=split_injury,
    palette=custom_palette,
    save="/path/to/output/"
)

# Spatial neighbourhood for fibroblast subclusters

## Data transfert to global object

In [ ]:
adata_subset = sc.read_h5ad("/path/to/your/file.h5ad")

In [ ]:
import pandas as pd
import numpy as np

# Make sure tangram predictions are projected
tg.project_cell_annotations(ad_map, immune_only_sp, annotation="cell_pop")
cell_pops = sorted(ad_sc.obs["cell_pop"].unique().tolist())

# Get the prediction dataframe from fibro object
tangram_pred = immune_only_sp.obsm["tangram_ct_pred"].copy()
tangram_pred.index = immune_only_sp.obs_names  # make sure index = barcodes

# Initialize columns in global object with NaN (non-fibroblast spots get NaN)
for pop in cell_pops:
    adata_subset.obs[f"tangram_{pop}"] = np.nan

# Transfer by barcode matching
adata_subset.obs.loc[tangram_pred.index, [f"tangram_{pop}" for pop in cell_pops]] = \
    tangram_pred[cell_pops].values

# Verify
print(adata_subset.obs[[f"tangram_{pop}" for pop in cell_pops]].describe())
print(f"\nNon-NaN fibro spots: {adata_subset.obs['tangram_MyoFib_1'].notna().sum()}")  # should be 47834
print(f"NaN non-fibro spots: {adata_subset.obs['tangram_MyoFib_1'].isna().sum()}")    # should be 846686-47834

In [ ]:
# Check available cell types
print(adata_subset.obs["celltype"].value_counts())

# Check your tangram columns are there
tangram_cols = [f"tangram_{pop}" for pop in cell_pops]
print("\nTangram columns in adata_subset:")
print(adata_subset.obs[tangram_cols].notna().sum())

# Check which conditions are available
print("\nAvailable conditions:")
print(adata_subset.obs["inj_type_per_day"].value_counts())

# Check connectivities graph is there
print("\nobsp keys:", list(adata_subset.obsp.keys()))

## Spatial neighbourhood using kmeans - Need to be re-run after loading all the files

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

# Step 1 — subset to uninjured + crush only
crush_conditions = [c for c in adata_subset.obs["inj_type_per_day"].unique() 
                    if c.startswith("crush") or c == "uninjured_uninjured"]
print("Keeping conditions:", crush_conditions)

adata_crush = adata_subset[adata_subset.obs["inj_type_per_day"].isin(crush_conditions)].copy()
print(f"Spots after filtering: {adata_crush.shape[0]}")

# Step 2 — identify fibroblast spots with a dominant subtype
tangram_cols = [f"tangram_{pop}" for pop in cell_pops]

# For each fibroblast spot, assign the subtype with highest tangram score
fibro_mask = adata_crush.obs["celltype"] == "Fibroblasts"
fibro_idx  = np.where(fibro_mask)[0]

tangram_df = adata_crush.obs[tangram_cols].copy()
tangram_df.columns = cell_pops

# Assign dominant subtype (only for fibroblast spots)
adata_crush.obs["fibro_subtype"] = np.nan
dominant = tangram_df.loc[fibro_mask].idxmax(axis=1)
adata_crush.obs.loc[fibro_mask, "fibro_subtype"] = dominant.values

print("\nFibroblast subtype distribution:")
print(adata_crush.obs["fibro_subtype"].value_counts())

adata_crush.write("/path/to/output/")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy import sparse

# ============================================================
# PARAMETERS — only change these
# ============================================================
crush_conditions    = ["uninjured_uninjured", "crush_d1", "crush_d2", "crush_d3",
                       "crush_d4", "crush_d5", "crush_d7", "crush_d9", "crush_d14"]

tangram_threshold   = 0.003     # min score to assign a subtype (None = always assign dominant subtype, 0.05 = discard uncertain spots)
n_neighbors         = 10       # number of spatial neighbors
use_rep             = "spatial" # "spatial" = physical distance, "X_pca" = expression-based
normalized          = True     # True = % relative to background, False = raw %
exclude_celltypes   = ["Fibroblasts"]  # cell types to exclude from neighbor counting

save_dir = "/path/to/output/"
# ============================================================

# ── STEP 1 — Subset to selected conditions ───────────────────
adata_crush = adata_subset[
    adata_subset.obs["inj_type_per_day"].isin(crush_conditions)
].copy()
print(f"Spots after filtering: {adata_crush.shape[0]}")
print(adata_crush.obs["inj_type_per_day"].value_counts())

In [ ]:
from sklearn.neighbors import KDTree
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── STEP 2 — Build KDTree on spatial coordinates ─────────────
print("Extracting spatial coordinates...")
coords       = adata_crush.obsm["spatial"]
fibro_mask   = (adata_crush.obs["celltype"] == "Fibroblasts").values
fibro_coords = coords[fibro_mask]

print(f"Total spots       : {coords.shape[0]}")
print(f"Fibroblast spots  : {fibro_mask.sum()}")
print(f"Building KDTree on {coords.shape[0]} spots...")

tree = KDTree(coords)

print(f"Querying {n_neighbors} neighbors for each fibroblast spot...")
distances, indices = tree.query(fibro_coords, k=n_neighbors + 1)  # +1 includes self
# Remove self (first column)
indices = indices[:, 1:]

print(f"Done! Neighbor index shape: {indices.shape}")  # (n_fibro_spots x n_neighbors)


In [ ]:
# ── STEP 3 — Assign dominant fibroblast subtype ───────────────
print("\nAssigning fibroblast subtypes...")
tangram_cols = [f"tangram_{pop}" for pop in cell_pops]
tangram_df   = adata_crush.obs[tangram_cols].copy()
tangram_df.columns = cell_pops

tangram_fibro   = tangram_df.loc[fibro_mask].copy()
dominant_scores = tangram_fibro.max(axis=1)
dominant        = tangram_fibro.idxmax(axis=1)

if tangram_threshold is not None:
    print(f"Applying tangram threshold: {tangram_threshold}")
    n_before = dominant.notna().sum()
    dominant[dominant_scores < tangram_threshold] = np.nan
    n_after  = dominant.notna().sum()
    print(f"  Spots before threshold : {n_before}")
    print(f"  Spots after threshold  : {n_after}")
    print(f"  Spots discarded        : {n_before - n_after}")
else:
    print("No threshold — winner-takes-all assignment")

adata_crush.obs["fibro_subtype"] = np.nan
adata_crush.obs.loc[fibro_mask, "fibro_subtype"] = dominant.values

print("\nFibroblast subtype distribution:")
print(adata_crush.obs["fibro_subtype"].value_counts())


In [ ]:
# ── STEP 4 — Neighborhood enrichment using KDTree indices ────
print("\nComputing neighborhood enrichment...")

target_celltypes = [ct for ct in adata_crush.obs["celltype"].unique()
                    if ct not in exclude_celltypes]

# Background proportions
background = adata_crush.obs["celltype"].value_counts(normalize=True) * 100
background = background[target_celltypes]

# Get celltype array for fast indexing
celltype_array  = adata_crush.obs["celltype"].values
subtype_array   = adata_crush.obs["fibro_subtype"].values

# Map fibroblast spot index in adata_crush to its row in fibro_coords/indices
fibro_global_idx = np.where(fibro_mask)[0]  # global indices of fibro spots

results = {}

for pop in cell_pops:
    # Which fibroblast spots belong to this subtype
    pop_fibro_mask = subtype_array[fibro_global_idx] == pop
    pop_fibro_rows = np.where(pop_fibro_mask)[0]  # rows in indices array

    if len(pop_fibro_rows) == 0:
        print(f"  {pop}: no spots — skipping")
        continue

    print(f"  {pop}: {len(pop_fibro_rows)} spots...")

    # Get all neighbor global indices for this subtype
    neighbor_global_idx = indices[pop_fibro_rows].flatten()  # (n_spots x n_neighbors)

    # Get cell types of all neighbors
    neighbor_cts = celltype_array[neighbor_global_idx]

    # Count
    unique, counts = np.unique(neighbor_cts, return_counts=True)
    count_dict = dict(zip(unique, counts))

    total = sum(count_dict.get(ct, 0) for ct in target_celltypes)
    if total > 0:
        results[pop] = {ct: count_dict.get(ct, 0) / total * 100
                        for ct in target_celltypes}
    else:
        results[pop] = {ct: 0 for ct in target_celltypes}

neighborhood_df = pd.DataFrame(results).T[target_celltypes]

print("\nRaw neighborhood proportions (%):")
print(neighborhood_df.round(2))

# Normalize by background
if normalized:
    print("\nNormalizing by background...")
    enrichment_df = neighborhood_df.div(background, axis=1)
    plot_df    = enrichment_df
    cbar_label = "Enrichment over background\n(observed % / expected %)"
    fmt        = ".2f"
    cmap       = "RdBu_r"
    center     = 1.0
    print(enrichment_df.round(3))
else:
    plot_df    = neighborhood_df
    cbar_label = "% of neighbors"
    fmt        = ".1f"
    cmap       = "YlOrRd"
    center     = None

In [ ]:
# ── STEP 5 — Plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    plot_df,
    ax=ax,
    cmap=cmap,
    center=center,
    annot=True,
    fmt=fmt,
    linewidths=0.5,
    cbar_kws={"label": cbar_label},
    annot_kws={"size": 9}
)

norm_label   = "normalized" if normalized else "raw"
thresh_label = f"thresh={tangram_threshold}" if tangram_threshold else "no_thresh"

ax.set_title(
    f"Spatial neighborhood enrichment ({norm_label}, {thresh_label})\n"
    f"Fibroblast subtypes → other cell types | spatial KDTree, k={n_neighbors}",
    fontsize=12
)
ax.set_xlabel("Neighboring cell type", fontsize=11)
ax.set_ylabel("Fibroblast subtype", fontsize=11)
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
save_path = f"{save_dir}neighborhood_{norm_label}_{thresh_label}_k{n_neighbors}.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"\nPlot saved to {save_path}")
plt.show()


In [ ]:
# ── STEP 6 — Save ─────────────────────────────────────────────
print("\nSaving adata_crush...")
adata_crush.write_h5ad(f"{save_dir}adata_crush_neighbors.h5ad")
neighborhood_df.to_csv(f"{save_dir}neighborhood_raw.csv")
if normalized:
    enrichment_df.to_csv(f"{save_dir}neighborhood_enrichment.csv")
print("All saved!")

In [ ]:
# ── STEP 5 — Plot ─────────────────────────────────────────────

# Always plot raw proportions here
plot_df    = neighborhood_df
cbar_label = "% of neighbors"
fmt        = ".1f"
cmap       = "YlOrRd"
center     = None

norm_label   = "raw"
thresh_label = f"thresh={tangram_threshold}" if tangram_threshold else "no_thresh"

fig, ax = plt.subplots(figsize=(12, 5))

sns.heatmap(
    plot_df,
    ax=ax,
    cmap=cmap,
    center=center,
    annot=True,
    fmt=fmt,
    linewidths=0.5,
    cbar_kws={"label": cbar_label},
    annot_kws={"size": 9}
)

ax.set_title(
    f"Spatial neighborhood enrichment ({norm_label}, {thresh_label})\n"
    f"Fibroblast subtypes → other cell types | spatial KDTree, k={n_neighbors}",
    fontsize=12
)

# Cell type labels at the bottom
ax.set_xlabel("")
ax.set_ylabel("Fibroblast subtype", fontsize=11)
ax.tick_params(axis="y", rotation=0)

# Move x labels to bottom and horizontal
ax.xaxis.set_ticks_position("bottom")
ax.xaxis.set_label_position("bottom")
ax.set_xticklabels(
    plot_df.columns.tolist(),
    rotation=0,          # horizontal
    ha="center",
    fontsize=10
)

plt.tight_layout()
save_path = f"{save_dir}neighborhood_{norm_label}_{thresh_label}_k{n_neighbors}.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"\nPlot saved to {save_path}")
plt.show()

In [ ]:
# ── STEP 5b — One heatmap per timepoint ───────────────────────

def plot_neighborhood_per_timepoint(adata_crush, neighborhood_results, 
                                     timepoint_col="day",
                                     normalized=False, n_neighbors=10,
                                     tangram_threshold=0.05,
                                     cmap="YlOrRd", save_dir=None):
    
    timepoints = sorted(adata_crush.obs[timepoint_col].unique().tolist())
    print(f"Timepoints found: {timepoints}")

    # Background proportions per timepoint
    target_celltypes = [ct for ct in adata_crush.obs["celltype"].unique()
                        if ct not in exclude_celltypes]

    # Spatial coords and arrays
    coords        = adata_crush.obsm["spatial"]
    celltype_array = adata_crush.obs["celltype"].values
    subtype_array  = adata_crush.obs["fibro_subtype"].values
    timepoint_array = adata_crush.obs[timepoint_col].values

    # Rebuild KDTree once on full dataset
    print("Building KDTree...")
    tree = KDTree(coords)

    for tp in timepoints:
        print(f"\nProcessing {tp}...")

        # Mask for this timepoint + fibroblasts
        tp_mask    = timepoint_array == tp
        tp_fibro_mask = tp_mask & (celltype_array == "Fibroblasts")
        tp_fibro_idx  = np.where(tp_fibro_mask)[0]

        if len(tp_fibro_idx) == 0:
            print(f"  No fibroblasts for {tp} — skipping")
            continue

        fibro_coords_tp = coords[tp_fibro_idx]

        # Query neighbors within full dataset
        _, indices_tp = tree.query(fibro_coords_tp, k=n_neighbors + 1)
        indices_tp = indices_tp[:, 1:]  # remove self

        # Background for this timepoint
        tp_celltypes = celltype_array[tp_mask]
        background_tp = pd.Series(tp_celltypes).value_counts(normalize=True) * 100
        background_tp = background_tp.reindex(target_celltypes, fill_value=0)

        results_tp = {}
        for pop in cell_pops:
            pop_mask_tp = subtype_array[tp_fibro_idx] == pop
            pop_rows    = np.where(pop_mask_tp)[0]

            if len(pop_rows) == 0:
                results_tp[pop] = {ct: 0 for ct in target_celltypes}
                continue

            neighbor_idx = indices_tp[pop_rows].flatten()
            neighbor_cts = celltype_array[neighbor_idx]

            unique, counts = np.unique(neighbor_cts, return_counts=True)
            count_dict = dict(zip(unique, counts))

            total = sum(count_dict.get(ct, 0) for ct in target_celltypes)
            if total > 0:
                results_tp[pop] = {ct: count_dict.get(ct, 0) / total * 100
                                   for ct in target_celltypes}
            else:
                results_tp[pop] = {ct: 0 for ct in target_celltypes}

        df_tp = pd.DataFrame(results_tp).T[target_celltypes]

        if normalized:
            plot_df_tp = df_tp.div(background_tp, axis=1)
            cbar_label = "Enrichment over background"
            fmt        = ".2f"
            center     = 1.0
            norm_label = "normalized"
        else:
            plot_df_tp = df_tp
            cbar_label = "% of neighbors"
            fmt        = ".1f"
            center     = None
            norm_label = "raw"

        # Plot
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.heatmap(
            plot_df_tp,
            ax=ax,
            cmap=cmap,
            center=center,
            annot=True,
            fmt=fmt,
            linewidths=0.5,
            cbar_kws={"label": cbar_label},
            annot_kws={"size": 9}
        )

        thresh_label = f"thresh={tangram_threshold}" if tangram_threshold else "no_thresh"
        ax.set_title(
            f"{tp} — Neighborhood enrichment ({norm_label}, {thresh_label}, k={n_neighbors})",
            fontsize=12
        )
        ax.set_ylabel("Fibroblast subtype", fontsize=11)
        ax.set_xlabel("")
        ax.tick_params(axis="y", rotation=0)
        ax.xaxis.set_ticks_position("bottom")
        ax.xaxis.set_label_position("bottom")
        ax.set_xticklabels(plot_df_tp.columns.tolist(), rotation=0, ha="center", fontsize=10)

        plt.tight_layout()

        if save_dir:
            save_path = f"{save_dir}neighborhood_{tp}_{norm_label}_k{n_neighbors}.png"
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            print(f"  Saved to {save_path}")

        plt.show()
        plt.close()


In [ ]:
# ── Run ───────────────────────────────────────────────────────
plot_neighborhood_per_timepoint(
    adata_crush=adata_crush,
    neighborhood_results=neighborhood_df,
    timepoint_col="day",
    normalized=False,       # True = enrichment, False = raw %
    n_neighbors=n_neighbors,
    tangram_threshold=tangram_threshold,
    save_dir=save_dir
)

In [ ]:
# ---- Only change these ----
mode       = "lineplot"
normalized = False
use_phases = True   # False = per timepoint, True = binned phases

phase_map = {
    "uninjured": "0_Uninjured",
    "d1": "1_Acute",  "d2": "1_Acute",  "d3": "1_Acute",
    "d4": "2_Subacute", "d5": "2_Subacute", "d7": "2_Subacute",
    "d9": "3_Late",   "d14": "3_Late",
}
phase_order  = ["0_Uninjured", "1_Acute", "2_Subacute", "3_Late"]
phase_labels = ["Uninjured", "Acute", "Subacute", "Late"]
# ---------------------------


def plot_neighborhood_over_time(adata_crush, tree, cell_pops,
                                 timepoint_col="day",
                                 normalized=False, n_neighbors=10,
                                 tangram_threshold=0.003,
                                 mode="heatmap",
                                 use_phases=False,
                                 phase_map=None,
                                 phase_order=None,
                                 phase_labels=None,
                                 exclude_celltypes=["Fibroblasts"],
                                 custom_palette=None,
                                 save_dir=None, dpi=150):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ── Timepoint ordering ────────────────────────────────────
    day_order = ["uninjured", "d1", "d2", "d3", "d4", "d5", "d7", "d9", "d14"]

    def sort_timepoint(tp):
        return day_order.index(tp) if tp in day_order else 99

    def clean_label(tp):
        return tp

    timepoints = sorted(adata_crush.obs[timepoint_col].unique().tolist(),
                        key=sort_timepoint)
    print(f"Timepoint order: {timepoints}")

    # ── Setup ─────────────────────────────────────────────────
    print("Reusing existing KDTree...")
    coords           = adata_crush.obsm["spatial"]
    celltype_array   = adata_crush.obs["celltype"].values
    subtype_array    = adata_crush.obs["fibro_subtype"].values
    timepoint_array  = adata_crush.obs[timepoint_col].values
    target_celltypes = [ct for ct in adata_crush.obs["celltype"].unique()
                        if ct not in exclude_celltypes]

    # ── Compute raw scores per timepoint ─────────────────────
    print("Computing scores across timepoints...")
    all_results = {}  # {pop: {tp: {ct: score}}}

    for tp in timepoints:
        tp_mask       = timepoint_array == tp
        tp_fibro_mask = tp_mask & (celltype_array == "Fibroblasts")
        tp_fibro_idx  = np.where(tp_fibro_mask)[0]

        if len(tp_fibro_idx) == 0:
            print(f"  {tp}: no fibroblasts — skipping")
            continue

        fibro_coords_tp = coords[tp_fibro_idx]
        _, indices_tp   = tree.query(fibro_coords_tp, k=n_neighbors + 1)
        indices_tp      = indices_tp[:, 1:]

        background_tp = pd.Series(celltype_array[tp_mask]).value_counts(normalize=True) * 100
        background_tp = background_tp.reindex(target_celltypes, fill_value=0)

        for pop in cell_pops:
            pop_mask_tp = subtype_array[tp_fibro_idx] == pop
            pop_rows    = np.where(pop_mask_tp)[0]

            if pop not in all_results:
                all_results[pop] = {}

            if len(pop_rows) == 0:
                all_results[pop][tp] = {ct: np.nan for ct in target_celltypes}
                continue

            neighbor_idx   = indices_tp[pop_rows].flatten()
            neighbor_cts   = celltype_array[neighbor_idx]
            unique, counts = np.unique(neighbor_cts, return_counts=True)
            count_dict     = dict(zip(unique, counts))

            total = sum(count_dict.get(ct, 0) for ct in target_celltypes)
            if total > 0:
                raw = {ct: count_dict.get(ct, 0) / total * 100 for ct in target_celltypes}
            else:
                raw = {ct: 0 for ct in target_celltypes}

            if normalized:
                all_results[pop][tp] = {
                    ct: raw[ct] / background_tp[ct] if background_tp[ct] > 0 else np.nan
                    for ct in target_celltypes
                }
            else:
                all_results[pop][tp] = raw

    # ── Aggregate into phases if requested ───────────────────
    def aggregate_phases(results_pop, phase_map, phase_order):
        """Average scores across timepoints within each phase"""
        phase_results = {}
        for phase in phase_order:
            tps_in_phase = [tp for tp, ph in phase_map.items() if ph == phase
                            and tp in results_pop]
            if not tps_in_phase:
                continue
            # Average across timepoints in this phase
            phase_data = {}
            for ct in target_celltypes:
                vals = [results_pop[tp][ct] for tp in tps_in_phase
                        if not np.isnan(results_pop[tp].get(ct, np.nan))]
                phase_data[ct] = np.mean(vals) if vals else np.nan
            phase_results[phase] = phase_data
        return phase_results

    # ── Palette ───────────────────────────────────────────────
    if custom_palette is None:
        ct_palette = dict(zip(target_celltypes,
                              sns.color_palette("tab10", len(target_celltypes))))
    else:
        ct_palette = custom_palette

    norm_label   = "normalized" if normalized else "raw"
    thresh_label = f"thresh={tangram_threshold}" if tangram_threshold else "no_thresh"

    # ── Plot one figure per fibroblast subtype ────────────────
    for pop in cell_pops:
        if pop not in all_results:
            continue

        if use_phases and phase_map is not None:
            # Aggregate into phases
            phase_results = aggregate_phases(all_results[pop], phase_map, phase_order)
            df_pop   = pd.DataFrame(phase_results).T
            df_pop   = df_pop.reindex(phase_order).dropna(how="all")
            df_pop   = df_pop[target_celltypes]
            x_labels = phase_labels[:len(df_pop)]
        else:
            df_pop   = pd.DataFrame(all_results[pop]).T
            df_pop   = df_pop.reindex(timepoints).dropna(how="all")
            df_pop   = df_pop[target_celltypes]
            x_labels = [clean_label(tp) for tp in df_pop.index]

        # ── HEATMAP ───────────────────────────────────────────
        if mode == "heatmap":
            fig, ax = plt.subplots(figsize=(len(df_pop) * 1.5 + 2, 5))
            cmap   = "RdBu_r" if normalized else "YlOrRd"
            center = 1.0 if normalized else None
            fmt    = ".2f" if normalized else ".1f"

            sns.heatmap(
                df_pop.T, ax=ax, cmap=cmap, center=center,
                annot=True, fmt=fmt, linewidths=0.5,
                cbar_kws={"label": "Enrichment" if normalized else "% neighbors"},
                annot_kws={"size": 8}
            )
            ax.set_xticklabels(x_labels, rotation=0, ha="center", fontsize=10)
            ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("Cell type", fontsize=11)

        # ── LINEPLOT ──────────────────────────────────────────
        elif mode == "lineplot":
            fig, ax = plt.subplots(figsize=(len(df_pop) * 1.5 + 2, 5))

            for ct in target_celltypes:
                ax.plot(x_labels, df_pop[ct].values,
                        marker="o", label=ct,
                        color=ct_palette.get(ct), linewidth=2)

            if normalized:
                ax.axhline(y=1.0, color="grey", linestyle="--",
                           linewidth=1, label="expected (=1)")
            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("Enrichment" if normalized else "% of neighbors", fontsize=11)
            ax.tick_params(axis="x", rotation=0)
            ax.spines[["top", "right"]].set_visible(False)
            ax.legend(title="Cell type", bbox_to_anchor=(1.01, 0.5),
                      loc="center left", fontsize=9)

        # ── BUBBLE ────────────────────────────────────────────
        elif mode == "bubble":
            fig, ax = plt.subplots(figsize=(len(df_pop) * 1.5 + 2, 5))

            for i, ct in enumerate(target_celltypes):
                for j, tp in enumerate(df_pop.index):
                    val = df_pop.loc[tp, ct]
                    if np.isnan(val):
                        continue
                    size  = abs(val) * 50 if normalized else val * 10
                    color = ct_palette.get(ct, "grey")
                    ax.scatter(j, i, s=size, color=color,
                               alpha=0.8, edgecolors="white", linewidth=0.5)
                    ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                            fontsize=6, color="white" if size > 200 else "black")

            ax.set_xticks(range(len(x_labels)))
            ax.set_xticklabels(x_labels, rotation=0, ha="center", fontsize=10)
            ax.set_yticks(range(len(target_celltypes)))
            ax.set_yticklabels(target_celltypes, fontsize=9)
            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("Cell type", fontsize=11)
            ax.grid(True, alpha=0.2)

            for ref_val in ([1, 2, 3] if normalized else [10, 25, 50]):
                ref_size = ref_val * 50 if normalized else ref_val * 10
                ax.scatter([], [], s=ref_size, color="grey", alpha=0.6,
                           label=f"{ref_val}{'x' if normalized else '%'}")
            ax.legend(title="Score", bbox_to_anchor=(1.01, 0.5),
                      loc="center left", fontsize=9)

        # ── STACKED BAR ───────────────────────────────────────
        elif mode == "stacked_bar":
            fig, ax = plt.subplots(figsize=(len(df_pop) * 1.5 + 2, 5))
            bottom = np.zeros(len(df_pop))

            for ct in target_celltypes:
                vals = df_pop[ct].fillna(0).values
                ax.bar(x_labels, vals, bottom=bottom,
                       color=ct_palette.get(ct, "grey"),
                       label=ct, edgecolor="white", linewidth=0.5)
                bottom += vals

            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("% of neighbors", fontsize=11)
            ax.tick_params(axis="x", rotation=0)
            ax.spines[["top", "right"]].set_visible(False)
            ax.legend(title="Cell type", bbox_to_anchor=(1.01, 0.5),
                      loc="center left", fontsize=9)

        time_label = "phases" if use_phases else "timepoints"
        fig.suptitle(f"{pop} — {mode} ({norm_label}, k={n_neighbors}, {thresh_label})",
                     fontsize=13, y=1.02)
        plt.tight_layout()

        if save_dir:
            save_path = (f"{save_dir}neighborhood_overtime_{pop}_"
                         f"{mode}_{norm_label}_{time_label}_k{n_neighbors}.png")
            plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
            print(f"Saved: {save_path}")

        plt.show()
        plt.close()


# ── Run ───────────────────────────────────────────────────────
# ---- Only change these ----
mode       = "lineplot"
normalized = False
use_phases = True

phase_map = {
    "uninjured": "0_Uninjured",
    "d1": "1_Acute",   "d2": "1_Acute",   "d3": "1_Acute",
    "d4": "2_Subacute", "d5": "2_Subacute", "d7": "2_Subacute",
    "d9": "3_Late",    "d14": "3_Late",
}
phase_order  = ["0_Uninjured", "1_Acute", "2_Subacute", "3_Late"]
phase_labels = ["Uninjured", "Acute", "Subacute", "Late"]
# ---------------------------

plot_neighborhood_over_time(
    adata_crush=adata_crush,
    tree=tree,
    cell_pops=cell_pops,
    timepoint_col="day",
    normalized=normalized,
    n_neighbors=n_neighbors,
    tangram_threshold=tangram_threshold,
    mode=mode,
    use_phases=use_phases,
    phase_map=phase_map,
    phase_order=phase_order,
    phase_labels=phase_labels,
    exclude_celltypes=exclude_celltypes,
    custom_palette=None,
    save_dir=save_dir
)

## Projection of all others cells - Use of Margherita dataset

In [ ]:
import scanpy as sc
import scipy.io
import pandas as pd
import numpy as np
import os

base = os.path.expanduser("~/Desktop/multiome/d9/tangram_export/")

# Load
counts   = scipy.io.mmread(base + "counts.mtx").T.tocsr()
barcodes = pd.read_csv(base + "barcodes.csv")
features = pd.read_csv(base + "features.csv")
metadata = pd.read_csv(base + "metadata.csv", index_col=0)

# Build AnnData
ad_atlas = sc.AnnData(X=counts)
ad_atlas.obs_names = barcodes["barcode"].values
ad_atlas.var_names = features["gene"].values
ad_atlas.obs       = metadata

# Verify metadata transferred
print(ad_atlas)
print(ad_atlas.obs.columns.tolist())
print(ad_atlas.obs.head())

# Normalize
sc.pp.normalize_total(ad_atlas, target_sum=1e4)
sc.pp.log1p(ad_atlas)

import pandas as pd
import numpy as np

# Load UMAP
umap = pd.read_csv(base + "umap.csv", index_col=0)
print(umap.head())
print(umap.shape)  # should be (n_cells x 2)

# Add to AnnData — make sure cell order matches
umap = umap.reindex(ad_atlas.obs_names)  # align by barcode
ad_atlas.obsm["X_umap"] = umap.values.astype(np.float32)

# Resave with UMAP included
ad_atlas.write_h5ad(base + "atlas.h5ad")

# Save
ad_atlas.write_h5ad(base + "MZ_RNA.h5ad")
print("Saved!")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Check the column is there
print(ad_atlas.obs["celltype_wsnn_res.0.2"].value_counts())
print(ad_atlas.obs["celltype_wsnn_res.0.2"].nunique())


In [ ]:
import matplotlib.pyplot as plt

# Check how many clusters
n_clusters = ad_atlas.obs["celltype_wsnn_res.0.2"].nunique()
print(f"Number of clusters: {n_clusters}")

# Use tab20 for ≤20, or generate more colors if needed
if n_clusters <= 20:
    palette = "tab20"
else:
    palette = [plt.cm.hsv(i / n_clusters) for i in range(n_clusters)]

sc.pl.umap(
    ad_atlas,
    color="celltype_wsnn_res.0.2",
    legend_loc="right margin",
    legend_fontsize=8,
    frameon=False,
    title="Atlas cell types",
    palette=palette,
    size=1,
)

In [ ]:
print(ad_atlas.obs.columns.tolist())
print(ad_atlas.obs["celltype_wsnn_res.0.2"].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Check conditions
print(ad_atlas.obs["condition"].value_counts())

# Build count table
count_df = pd.crosstab(
    ad_atlas.obs["condition"],
    ad_atlas.obs["celltype_wsnn_res.0.2"]
)
print(count_df.shape)

# Generate colors for 60 clusters
import matplotlib.cm as cm
n_clusters = count_df.shape[1]
colors = [cm.tab20(i % 20) if i < 20 
          else cm.tab20b(i % 20) if i < 40 
          else cm.tab20c(i % 20) 
          for i in range(n_clusters)]

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

bottom = np.zeros(len(count_df))
for i, cluster in enumerate(count_df.columns):
    ax.bar(
        count_df.index,
        count_df[cluster].values,
        bottom=bottom,
        color=colors[i],
        label=cluster,
        edgecolor="white",
        linewidth=0.3
    )
    bottom += count_df[cluster].values

ax.set_xlabel("Condition", fontsize=12)
ax.set_ylabel("Number of cells", fontsize=12)
ax.set_title("Cell type composition per condition", fontsize=13)
ax.tick_params(axis="x", rotation=45)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Cell type", bbox_to_anchor=(1.01, 0.5),
          loc="center left", fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig("/path/to/output/",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
clusters = sorted(ad_atlas.obs["celltype_wsnn_res.0.2"].unique().tolist())
print(clusters)

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

# Check what your condition column is called
print(ad_atlas.obs.columns.tolist())

# Replace 'condition' with your actual column name
condition_col = 'condition'   # ← change this

# Count cells per celltype per condition
counts = (
    ad_atlas.obs
    .groupby([condition_col, 'celltype_wsnn_res.0.2'])
    .size()
    .reset_index(name='count')
)

# Normalize to percentage per condition
counts['total'] = counts.groupby(condition_col)['count'].transform('sum')
counts['pct'] = counts['count'] / counts['total'] * 100

print(counts.head(20))

import seaborn as sns

pivot = counts.pivot(index='celltype_wsnn_res.0.2', columns=condition_col, values='pct').fillna(0)

fig, ax = plt.subplots(figsize=(10, 12))
pivot.plot(kind='barh', ax=ax, colormap='tab10')
ax.set_xlabel('% of cells')
ax.set_title('Cell type proportions per condition')
ax.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('celltype_per_condition.pdf')
plt.show()

In [ ]:
import matplotlib.colors as mcolors
import numpy as np

def generate_shades(base_hex, n, min_lightness=0.3, max_lightness=0.95):
    """
    Generate n shades of a base color varying lightness in HSL space.
    min_lightness = darkest shade, max_lightness = lightest shade
    """
    import colorsys
    base_rgb = mcolors.to_rgb(base_hex)
    h, l, s  = colorsys.rgb_to_hls(*base_rgb)
    shades   = []
    for i in range(n):
        lightness = max_lightness - (max_lightness - min_lightness) * i / max(n - 1, 1)
        r, g, b   = colorsys.hls_to_rgb(h, lightness, s)
        shades.append((r, g, b))
    return shades  # light → dark

# ── Refined base colors ───────────────────────────────────────
# Astrocytes    : sage green
# Neurons_D     : deep violet
# Neurons_V     : mauve/pink-purple
# Oligodendroglia: navy/slate blue
# Perivascular  : warm red/coral
# Macrophages   : burnt orange
# Microglia     : golden amber
# Ependymal     : charcoal/slate grey

color_config = {
    "Astrocytes":      ("#3a7d44", astro_cl),         # sage green
    "Neurons_D":       ("#3b1f6e", [c for c in neuron_cl if "Neurons_D" in c]),  # deep violet
    "Neurons_V":       ("#9b59b6", [c for c in neuron_cl if "Neurons_V" in c]),  # mauve
    "Oligodendroglia": ("#1a3a6e", oligo_cl),          # navy blue
    "Perivascular":    ("#e91e8c", periv_cl),          # hot pink/crimson — moved away from orange
    "Macrophages":     ("#b94600", macro_cl),          # deep burnt orange
    "Microglia":       ("#e6a817", micro_cl),          # golden amber
    "Ependymal":       ("#4a4a4a", ependy_cl),         # charcoal
}

# ── Build palette ─────────────────────────────────────────────
custom_palette = {}
for group, (base_color, cluster_list) in color_config.items():
    n = len(cluster_list)
    if n == 0:
        continue
    shades = generate_shades(base_color, n, min_lightness=0.25, max_lightness=0.80)
    for cluster, shade in zip(cluster_list, shades):
        custom_palette[cluster] = shade

# ── Preview palette ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
col_order = (astro_cl +
             [c for c in neuron_cl if "Neurons_D" in c] +
             [c for c in neuron_cl if "Neurons_V" in c] +
             oligo_cl + periv_cl + macro_cl + micro_cl + ependy_cl)

for i, cluster in enumerate(col_order):
    ax.bar(i, 1, color=custom_palette[cluster], edgecolor="white", linewidth=0.5)
    ax.text(i, 1.02, cluster, rotation=90, ha="center", va="bottom", fontsize=5)

ax.set_xlim(-0.5, len(col_order) - 0.5)
ax.axis("off")
ax.set_title("Color palette preview", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── UMAP ─────────────────────────────────────────────────────
sc.pl.umap(
    ad_atlas,
    color="celltype_wsnn_res.0.2",
    legend_loc="right margin",
    legend_fontsize=7,
    frameon=False,
    title="Atlas cell types",
    palette=custom_palette,
    size=1,
)


In [ ]:
# ── UMAP ─────────────────────────────────────────────────────
sc.pl.umap(
    ad_atlas,
    color="celltype_wsnn_res.0.2",
    legend_loc="right margin",
    legend_fontsize=7,
    frameon=False,
    title="Atlas cell types",
    palette=custom_palette,
    size=1,
)

# ── Stacked bar ───────────────────────────────────────────────
count_df = pd.crosstab(
    ad_atlas.obs["condition"],
    ad_atlas.obs["celltype_wsnn_res.0.2"]
)
count_df = count_df[col_order]  # reorder columns by cell type group
count_pct = count_df.div(count_df.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(22, 6))

for ax, (df, ylabel, title) in zip(axes, [
    (count_df,    "Number of cells", "Raw counts"),
    (count_pct,   "% of cells",      "Proportions"),
]):
    bottom = np.zeros(len(df))
    for cluster in df.columns:
        ax.bar(df.index, df[cluster].values, bottom=bottom,
               color=custom_palette[cluster],
               label=cluster, edgecolor="white", linewidth=0.3)
        bottom += df[cluster].values

    ax.set_xlabel("Condition", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.tick_params(axis="x", rotation=45)
    ax.spines[["top", "right"]].set_visible(False)

# Shared legend grouped by cell type
handles = [plt.Rectangle((0, 0), 1, 1, color=custom_palette[c], label=c)
           for c in col_order]
fig.legend(handles=handles, title="Cell type", bbox_to_anchor=(1.01, 0.5),
           loc="center left", fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig("/path/to/output/",
            dpi=150, bbox_inches="tight")
plt.show()

## Run Tangram with Margherita dataset

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import tangram as tg
import os

# ── Paths ─────────────────────────────────────────────────────
save_dir = "/path/to/output/"
proj_dir = "/path/to/project/"
os.makedirs(proj_dir, exist_ok=True)

# ── Step 1 — Load objects ─────────────────────────────────────
print("Loading objects...")
ad_atlas     = sc.read_h5ad(f"{save_dir}MZ_RNA.h5ad")
adata_subset = sc.read_h5ad(f"{save_dir}adata_subset_tangram.h5ad")

print(f"Atlas  : {ad_atlas.shape}")
print(f"Spatial: {adata_subset.shape}")

# ── Step 2 — Fix normalization ────────────────────────────────
adata_subset.X = adata_subset.layers["normalized"]
print(f"Spatial X max: {adata_subset.X.max()}")
print(f"Atlas X max  : {ad_atlas.X.max()}")

# ── Step 3 — Stratified subsample for training ────────────────
print("\nBuilding stratified subsample...")
np.random.seed(42)
stratified_idx = []
target_n       = 100000

for batch in adata_subset.obs["batch"].unique():
    batch_mask = adata_subset.obs["batch"] == batch
    batch_idx  = np.where(batch_mask)[0]
    n_sample   = max(1, int(target_n * len(batch_idx) / adata_subset.shape[0]))
    sampled    = np.random.choice(batch_idx, 
                                   size=min(n_sample, len(batch_idx)), 
                                   replace=False)
    stratified_idx.extend(sampled)

stratified_idx = np.array(stratified_idx)
print(f"Stratified subsample: {len(stratified_idx)} spots")
print("Spots per batch:")
print(pd.Series(adata_subset.obs["batch"].values[stratified_idx]).value_counts())

# ── Step 4 — Train Tangram on stratified subsample ────────────
print("\nTraining Tangram on stratified subsample...")
adata_train   = adata_subset[stratified_idx].copy()
adata_train.X = adata_train.layers["normalized"]

tg.pp_adatas(ad_atlas, adata_train, genes=None)
print(f"Shared genes: {len(ad_atlas.uns['overlap_genes'])}")

ad_map_atlas = tg.map_cells_to_space(
    adata_sc=ad_atlas,
    adata_sp=adata_train,
    mode="clusters",
    cluster_label="celltype_wsnn_res.0.2",
    density_prior="rna_count_based",
    num_epochs=300,
    device="cpu",
    verbose=True
)

# Save mapping matrix
ad_map_atlas.write_h5ad(f"{proj_dir}ad_map_atlas_stratified.h5ad")
print("Mapping matrix saved!")


In [ ]:
# Find the correct case for overlap genes in spatial data
# Build a lowercase → original case mapping for spatial genes
spatial_gene_map = {g.lower(): g for g in adata_subset.var_names}
atlas_gene_map   = {g.lower(): g for g in ad_atlas.var_names}

# Get overlap genes in correct case for each object
overlap_genes_lower   = [g.lower() for g in overlap_genes]
overlap_spatial = [spatial_gene_map[g] for g in overlap_genes_lower if g in spatial_gene_map]
overlap_atlas   = [atlas_gene_map[g]   for g in overlap_genes_lower if g in atlas_gene_map]

# Keep only genes present in both
common_lower  = set(g.lower() for g in overlap_spatial) & set(g.lower() for g in overlap_atlas)
overlap_spatial = [spatial_gene_map[g] for g in common_lower]
overlap_atlas   = [atlas_gene_map[g]   for g in common_lower]

print(f"Overlap genes after case fix: {len(overlap_spatial)}")

# Recompute cluster centroids using correct atlas gene names
atlas_expr = pd.DataFrame(
    ad_atlas[:, overlap_atlas].X.toarray()
    if hasattr(ad_atlas[:, overlap_atlas].X, "toarray")
    else np.array(ad_atlas[:, overlap_atlas].X),
    index=ad_atlas.obs_names,
    columns=overlap_atlas
)

cluster_labels    = ad_atlas.obs["celltype_wsnn_res.0.2"]
cluster_centroids = atlas_expr.groupby(cluster_labels).mean()
print(f"Cluster centroids shape: {cluster_centroids.shape}")

# Now project in batches using spatial gene names
from sklearn.metrics.pairwise import cosine_similarity

batch_size = 100000
all_idx    = np.arange(adata_subset.shape[0])
batches    = np.array_split(all_idx, int(np.ceil(len(all_idx) / batch_size)))
all_preds  = {}

for i, batch_idx in enumerate(batches):
    print(f"\nBatch {i+1}/{len(batches)} ({len(batch_idx)} spots)...")

    adata_batch = adata_subset[batch_idx]

    # Use spatial gene names (correct case)
    spatial_expr = pd.DataFrame(
        adata_batch[:, overlap_spatial].X.toarray()
        if hasattr(adata_batch[:, overlap_spatial].X, "toarray")
        else np.array(adata_batch[:, overlap_spatial].X),
        index=adata_batch.obs_names,
        columns=overlap_spatial
    )

    # Rename spatial columns to match atlas for cosine similarity
    spatial_expr.columns = overlap_atlas

    # Cosine similarity
    similarity = cosine_similarity(spatial_expr.values,
                                   cluster_centroids.values)
    pred_batch = pd.DataFrame(
        similarity,
        index=adata_batch.obs_names,
        columns=cluster_centroids.index
    )

    # Normalize to probabilities
    pred_batch = pred_batch.div(pred_batch.sum(axis=1), axis=0)
    all_preds[i] = pred_batch
    print(f"  Done: {pred_batch.shape}")

# Combine
print("\nCombining predictions...")
full_pred_df = pd.concat(list(all_preds.values()))
full_pred_df = full_pred_df.reindex(adata_subset.obs_names)
print(f"Full predictions shape: {full_pred_df.shape}")

# Assign
adata_subset.obs["celltype_tangram"]       = full_pred_df.idxmax(axis=1).values
adata_subset.obs["celltype_tangram_score"] = full_pred_df.max(axis=1).values
adata_subset.obsm["tangram_atlas_pred"]    = full_pred_df

print("\nFull cell type distribution:")
print(adata_subset.obs["celltype_tangram"].value_counts())

# Save
print("\nSaving...")
adata_subset.write_h5ad(f"{proj_dir}adata_subset_atlas_projected.h5ad")
full_pred_df.to_csv(f"{proj_dir}tangram_atlas_predictions_full.csv")
print(f"All saved in {proj_dir}")

In [ ]:
# Build case mapping
spatial_gene_map = {g.lower(): g for g in adata_subset.var_names}
atlas_gene_map   = {g.lower(): g for g in ad_atlas.var_names}

# Get overlap genes in correct case for each object
overlap_genes_lower = [g.lower() for g in overlap_genes]
common_lower        = set(overlap_genes_lower) & set(spatial_gene_map.keys()) & set(atlas_gene_map.keys())

overlap_spatial = [spatial_gene_map[g] for g in common_lower]  # title case e.g. 'Col1a1'
overlap_atlas   = [atlas_gene_map[g]   for g in common_lower]  # lowercase e.g. 'col1a1'

print(f"Overlap genes after case fix: {len(overlap_spatial)}")

# Cluster centroids from atlas
atlas_expr = pd.DataFrame(
    ad_atlas[:, overlap_atlas].X.toarray()
    if hasattr(ad_atlas[:, overlap_atlas].X, "toarray")
    else np.array(ad_atlas[:, overlap_atlas].X),
    index=ad_atlas.obs_names,
    columns=overlap_atlas
)
cluster_labels    = ad_atlas.obs["celltype_wsnn_res.0.2"]
cluster_centroids = atlas_expr.groupby(cluster_labels).mean()
print(f"Cluster centroids shape: {cluster_centroids.shape}")

# Project in batches
from sklearn.metrics.pairwise import cosine_similarity

batch_size = 100000
all_idx    = np.arange(adata_subset.shape[0])
batches    = np.array_split(all_idx, int(np.ceil(len(all_idx) / batch_size)))
all_preds  = {}

for i, batch_idx in enumerate(batches):
    print(f"\nBatch {i+1}/{len(batches)} ({len(batch_idx)} spots)...")

    adata_batch  = adata_subset[batch_idx]
    spatial_expr = pd.DataFrame(
        adata_batch[:, overlap_spatial].X.toarray()
        if hasattr(adata_batch[:, overlap_spatial].X, "toarray")
        else np.array(adata_batch[:, overlap_spatial].X),
        index=adata_batch.obs_names,
        columns=overlap_atlas  # rename to match atlas
    )

    similarity = cosine_similarity(spatial_expr.values,
                                   cluster_centroids.values)
    pred_batch = pd.DataFrame(
        similarity,
        index=adata_batch.obs_names,
        columns=cluster_centroids.index
    )
    pred_batch   = pred_batch.div(pred_batch.sum(axis=1), axis=0)
    all_preds[i] = pred_batch
    print(f"  Done: {pred_batch.shape}")

# Combine
print("\nCombining predictions...")
full_pred_df = pd.concat(list(all_preds.values()))
full_pred_df = full_pred_df.reindex(adata_subset.obs_names)
print(f"Full predictions shape: {full_pred_df.shape}")
print(f"NaN rows: {full_pred_df.isna().all(axis=1).sum()}")

# Assign
adata_subset.obs["celltype_tangram"]       = full_pred_df.idxmax(axis=1).values
adata_subset.obs["celltype_tangram_score"] = full_pred_df.max(axis=1).values
adata_subset.obsm["tangram_atlas_pred"]    = full_pred_df

print("\nFull cell type distribution:")
print(adata_subset.obs["celltype_tangram"].value_counts())
print(f"Spots with prediction   : {adata_subset.obs['celltype_tangram'].notna().sum()}")
print(f"Spots without prediction: {adata_subset.obs['celltype_tangram'].isna().sum()}")

# Save
print("\nSaving...")
adata_subset.write_h5ad(f"{proj_dir}adata_subset_atlas_projected.h5ad")
full_pred_df.to_csv(f"{proj_dir}tangram_atlas_predictions_full.csv")
print(f"All saved in {proj_dir}")

## Validation of Tangram results

In [ ]:
# ── Quick validation ──────────────────────────────────────────

# 1. Distribution of predicted cell types
print("Predicted cell type distribution:")
print(adata_subset.obs["celltype_tangram"].value_counts())

# 2. Compare with existing annotation
print("\nExisting celltype distribution:")
print(adata_subset.obs["celltype"].value_counts())

# 3. Confidence scores
print("\nPrediction score stats:")
print(adata_subset.obs["celltype_tangram_score"].describe())

# 4. Cross-tabulation — does tangram agree with existing annotation?
crosstab = pd.crosstab(
    adata_subset.obs["celltype"],
    adata_subset.obs["celltype_tangram"].str.split("_").str[0]  # broad cell type
)
print("\nCross-tab (existing vs tangram broad cell type):")
print(crosstab)

In [ ]:
# Broad cell type from tangram (strip cluster number)
adata_subset.obs["celltype_tangram_broad"] = (
    adata_subset.obs["celltype_tangram"]
    .str.replace("_\d+$", "", regex=True)  # remove _0, _1 etc
    .str.replace("_D$|_V$", "", regex=True)  # remove _D, _V from neurons
    .str.replace("Neurons_D|Neurons_V", "Neurons", regex=True)
)

# Cross-tab
crosstab = pd.crosstab(
    adata_subset.obs["celltype"],
    adata_subset.obs["celltype_tangram_broad"],
    normalize="index"  # normalize by row = % per existing cell type
) * 100

print(crosstab.round(1))

# Also check score distribution per predicted cell type
print("\nMean score per predicted cell type:")
print(adata_subset.obs.groupby("celltype_tangram_broad")["celltype_tangram_score"].mean().sort_values(ascending=False).round(3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ── Build normalized crosstab ─────────────────────────────────
crosstab = pd.crosstab(
    adata_subset.obs["celltype"],
    adata_subset.obs["celltype_tangram_broad"],
    normalize="index"
) * 100

# Order columns nicely
col_order = sorted(crosstab.columns.tolist())
crosstab  = crosstab[col_order]

fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(
    crosstab,
    ax=ax,
    cmap="YlOrRd",
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={"label": "% of spots"},
    annot_kws={"size": 8}
)

ax.set_title("Existing annotation vs Tangram atlas projection\n(% per existing cell type)",
             fontsize=13)
ax.set_xlabel("Tangram predicted cell type (broad)", fontsize=11)
ax.set_ylabel("Existing cell type annotation", fontsize=11)
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(f"{proj_dir}crosstab_existing_vs_tangram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cluster_col = "celltype_tangram"

categories = adata_crush_unified.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 6                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = adata_crush_unified.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = adata_crush_unified.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = adata_crush_unified.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show() 

## Projection of Margherita dataset with the fibro subpopulations

In [ ]:
# Check both objects
print("=== adata_crush ===")
print(adata_crush.obs["fibro_subtype"].value_counts())
print(f"Shape: {adata_crush.shape}")

print("\n=== adata_subset ===")
print(f"Shape: {adata_subset.shape}")

# Check barcode overlap
overlap = adata_crush.obs_names.isin(adata_subset.obs_names).sum()
print(f"\nMatching barcodes: {overlap} / {adata_crush.shape[0]}")

In [ ]:
# Start with celltype_tangram_broad as string
combined_broad = adata_subset.obs["celltype_tangram_broad"].astype(str).copy()

# Assign fibroblast subtypes
fibro_mask  = adata_subset.obs["celltype"] == "Fibroblasts"
has_subtype = adata_subset.obs["fibro_subtype"].notna()

combined_broad[fibro_mask & has_subtype]  = adata_subset.obs.loc[
    fibro_mask & has_subtype, "fibro_subtype"].astype(str)
combined_broad[fibro_mask & ~has_subtype] = "Fibroblasts"

adata_subset.obs["celltype_tangram_broad"] = combined_broad

pd.set_option("display.max_rows", None)
print(adata_subset.obs["celltype_tangram_broad"].value_counts())
pd.reset_option("display.max_rows")

# Save
adata_subset.write_h5ad(f"{proj_dir}adata_subset_broad_unified.h5ad")
print("Done!")

In [ ]:
def plot_neighborhood_over_time_unified(adata, tree, cell_pops,
                                         timepoint_col="day",
                                         celltype_col="combined_tangram",
                                         normalized=False, n_neighbors=10,
                                         tangram_threshold=0.003,
                                         mode="heatmap",
                                         use_phases=False,
                                         phase_map=None,
                                         phase_order=None,
                                         phase_labels=None,
                                         exclude_celltypes=None,
                                         show_celltypes=None,
                                         include_fibro_interactions=True,
                                         custom_palette=None,
                                         save_dir=None, dpi=150):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ── Timepoint ordering ────────────────────────────────────
    day_order = ["uninjured", "d1", "d2", "d3", "d4", "d5", "d7", "d9", "d14"]

    def sort_timepoint(tp):
        return day_order.index(tp) if tp in day_order else 99

    def clean_label(tp):
        return tp

    timepoints = sorted(adata.obs[timepoint_col].unique().tolist(),
                        key=sort_timepoint)
    print(f"Timepoint order: {timepoints}")

    # ── Setup ─────────────────────────────────────────────────
    print("Reusing existing KDTree...")
    coords          = adata.obsm["spatial"]
    combined_array  = adata.obs[celltype_col].values
    subtype_array   = adata.obs["fibro_subtype"].values
    timepoint_array = adata.obs[timepoint_col].values

    if exclude_celltypes is None:
        exclude_celltypes = []

    # Target cell types for neighbor counting
    all_cats = adata.obs[celltype_col].unique().tolist()

    if include_fibro_interactions:
        # Include all cell types EXCEPT unsubtyped fibroblasts and excluded
        target_celltypes = [ct for ct in all_cats
                            if ct != "Fibroblasts"
                            and ct not in exclude_celltypes]
    else:
        # Exclude fibroblast subtypes from neighbors too
        target_celltypes = [ct for ct in all_cats
                            if ct not in cell_pops
                            and ct != "Fibroblasts"
                            and ct not in exclude_celltypes]

    target_celltypes = sorted(target_celltypes)
    print(f"Target cell types for counting ({len(target_celltypes)})")

    # show_celltypes = subset to display in plots (after counting)
    if show_celltypes is not None:
        display_celltypes = [ct for ct in show_celltypes if ct in target_celltypes]
        missing = [ct for ct in show_celltypes if ct not in target_celltypes]
        if missing:
            print(f"Warning: {missing} not found in target_celltypes — skipping")
    else:
        display_celltypes = target_celltypes
    print(f"Display cell types ({len(display_celltypes)}): {display_celltypes}")

    # ── Compute scores per timepoint ──────────────────────────
    print("Computing scores across timepoints...")
    all_results = {}

    for tp in timepoints:
        tp_mask       = timepoint_array == tp
        tp_fibro_mask = tp_mask & np.isin(combined_array, cell_pops)
        tp_fibro_idx  = np.where(tp_fibro_mask)[0]

        if len(tp_fibro_idx) == 0:
            print(f"  {tp}: no fibroblast spots — skipping")
            continue

        fibro_coords_tp = coords[tp_fibro_idx]
        _, indices_tp   = tree.query(fibro_coords_tp, k=n_neighbors + 1)
        indices_tp      = indices_tp[:, 1:]

        background_tp = pd.Series(combined_array[tp_mask]).value_counts(normalize=True) * 100
        background_tp = background_tp.reindex(target_celltypes, fill_value=0)

        for pop in cell_pops:
            pop_mask_tp = subtype_array[tp_fibro_idx] == pop
            pop_rows    = np.where(pop_mask_tp)[0]

            if pop not in all_results:
                all_results[pop] = {}

            if len(pop_rows) == 0:
                all_results[pop][tp] = {ct: np.nan for ct in target_celltypes}
                continue

            neighbor_idx   = indices_tp[pop_rows].flatten()
            neighbor_cts   = combined_array[neighbor_idx]
            unique, counts = np.unique(neighbor_cts, return_counts=True)
            count_dict     = dict(zip(unique, counts))

            total = sum(count_dict.get(ct, 0) for ct in target_celltypes)
            if total > 0:
                raw = {ct: count_dict.get(ct, 0) / total * 100 for ct in target_celltypes}
            else:
                raw = {ct: 0 for ct in target_celltypes}

            if normalized:
                all_results[pop][tp] = {
                    ct: raw[ct] / background_tp[ct] if background_tp[ct] > 0 else np.nan
                    for ct in target_celltypes
                }
            else:
                all_results[pop][tp] = raw

    # ── Aggregate into phases ─────────────────────────────────
    def aggregate_phases(results_pop, phase_map, phase_order):
        phase_results = {}
        for phase in phase_order:
            tps_in_phase = [tp for tp, ph in phase_map.items()
                            if ph == phase and tp in results_pop]
            if not tps_in_phase:
                continue
            phase_data = {}
            for ct in target_celltypes:
                vals = [results_pop[tp][ct] for tp in tps_in_phase
                        if not np.isnan(results_pop[tp].get(ct, np.nan))]
                phase_data[ct] = np.mean(vals) if vals else np.nan
            phase_results[phase] = phase_data
        return phase_results

    # ── Palette ───────────────────────────────────────────────
    if custom_palette is None:
        ct_palette = dict(zip(target_celltypes,
                              sns.color_palette("tab10", len(target_celltypes))))
    else:
        ct_palette = custom_palette

    norm_label   = "normalized" if normalized else "raw"
    thresh_label = f"thresh={tangram_threshold}" if tangram_threshold else "no_thresh"

    # ── Plot one figure per fibroblast subtype ────────────────
    for pop in cell_pops:
        if pop not in all_results:
            continue

        if use_phases and phase_map is not None:
            phase_results = aggregate_phases(all_results[pop], phase_map, phase_order)
            df_pop   = pd.DataFrame(phase_results).T
            df_pop   = df_pop.reindex(phase_order).dropna(how="all")
            df_pop   = df_pop[target_celltypes]
            x_labels = phase_labels[:len(df_pop)]
        else:
            df_pop   = pd.DataFrame(all_results[pop]).T
            df_pop   = df_pop.reindex(timepoints).dropna(how="all")
            df_pop   = df_pop[target_celltypes]
            x_labels = [clean_label(tp) for tp in df_pop.index]

        # Filter to display_celltypes for plotting only
        df_plot = df_pop[display_celltypes]

        # ── HEATMAP ───────────────────────────────────────────
        if mode == "heatmap":
            fig, ax = plt.subplots(figsize=(len(df_plot) * 1.5 + 2, 
                                            len(display_celltypes) * 0.4 + 2))
            cmap   = "RdBu_r" if normalized else "YlOrRd"
            center = 1.0 if normalized else None
            fmt    = ".2f" if normalized else ".1f"

            sns.heatmap(
                df_plot.T, ax=ax, cmap=cmap, center=center,
                annot=True, fmt=fmt, linewidths=0.5,
                cbar_kws={"label": "Enrichment" if normalized else "% neighbors"},
                annot_kws={"size": 7}
            )
            ax.set_xticklabels(x_labels, rotation=0, ha="center", fontsize=10)
            ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("Cell type", fontsize=11)

        # ── LINEPLOT ──────────────────────────────────────────
        elif mode == "lineplot":
            fig, ax = plt.subplots(figsize=(len(df_plot) * 1.5 + 2, 5))

            for ct in display_celltypes:
                ax.plot(x_labels, df_plot[ct].values,
                        marker="o", label=ct,
                        color=ct_palette.get(ct, "grey"), linewidth=2)

            if normalized:
                ax.axhline(y=1.0, color="grey", linestyle="--",
                           linewidth=1, label="expected (=1)")
            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("Enrichment" if normalized else "% of neighbors", fontsize=11)
            ax.tick_params(axis="x", rotation=0)
            ax.spines[["top", "right"]].set_visible(False)
            ax.legend(title="Cell type", bbox_to_anchor=(1.01, 0.5),
                      loc="center left", fontsize=9)

        # ── BUBBLE ────────────────────────────────────────────
        elif mode == "bubble":
            fig, ax = plt.subplots(figsize=(len(df_plot) * 1.5 + 2,
                                            len(display_celltypes) * 0.5 + 2))

            for i, ct in enumerate(display_celltypes):
                for j, tp in enumerate(df_plot.index):
                    val = df_plot.loc[tp, ct]
                    if np.isnan(val):
                        continue
                    size  = abs(val) * 50 if normalized else val * 10
                    color = ct_palette.get(ct, "grey")
                    ax.scatter(j, i, s=size, color=color,
                               alpha=0.8, edgecolors="white", linewidth=0.5)
                    ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                            fontsize=6, color="white" if size > 200 else "black")

            ax.set_xticks(range(len(x_labels)))
            ax.set_xticklabels(x_labels, rotation=0, ha="center", fontsize=10)
            ax.set_yticks(range(len(display_celltypes)))
            ax.set_yticklabels(display_celltypes, fontsize=8)
            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("Cell type", fontsize=11)
            ax.grid(True, alpha=0.2)

            for ref_val in ([1, 2, 3] if normalized else [10, 25, 50]):
                ref_size = ref_val * 50 if normalized else ref_val * 10
                ax.scatter([], [], s=ref_size, color="grey", alpha=0.6,
                           label=f"{ref_val}{'x' if normalized else '%'}")
            ax.legend(title="Score", bbox_to_anchor=(1.01, 0.5),
                      loc="center left", fontsize=9)

        # ── STACKED BAR ───────────────────────────────────────
        elif mode == "stacked_bar":
            fig, ax = plt.subplots(figsize=(len(df_plot) * 1.5 + 2, 5))
            bottom = np.zeros(len(df_plot))

            for ct in display_celltypes:
                vals = df_plot[ct].fillna(0).values
                ax.bar(x_labels, vals, bottom=bottom,
                       color=ct_palette.get(ct, "grey"),
                       label=ct, edgecolor="white", linewidth=0.5)
                bottom += vals

            ax.set_xlabel("Phase" if use_phases else "Timepoint", fontsize=11)
            ax.set_ylabel("% of neighbors", fontsize=11)
            ax.tick_params(axis="x", rotation=0)
            ax.spines[["top", "right"]].set_visible(False)
            ax.legend(title="Cell type", bbox_to_anchor=(1.01, 0.5),
                      loc="center left", fontsize=9)

        fibro_label = "with_fibro" if include_fibro_interactions else "no_fibro"
        time_label  = "phases" if use_phases else "timepoints"
        fig.suptitle(f"{pop} — {mode} ({norm_label}, k={n_neighbors}, {fibro_label})",
                     fontsize=13, y=1.02)
        plt.tight_layout()

        if save_dir:
            save_path = (f"{save_dir}neighborhood_unified_{pop}_"
                         f"{mode}_{norm_label}_{time_label}_{fibro_label}_k{n_neighbors}.pdf")
            plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
            print(f"Saved: {save_path}")

        plt.show()
        plt.close()


In [ ]:
# ---- Only change these ----
mode       = "heatmap"
normalized = False
use_phases = False

# Filter which cell types to show in plots (None = show all)
show_celltypes = ["Macrophages_0", "Macrophages_1", "Macrophages_2", 
                  "Microglia_0", "Microglia_2"]  # e.g. ["Astrocytes", "Microglia_0", "Oligodendroglia_1"]

# Include fibroblast-fibroblast interactions
include_fibro_interactions = True  # True = show fibro subtypes as neighbors too

phase_map = {
    "uninjured": "0_Uninjured",
    "d1": "1_Acute",    "d2": "1_Acute",    "d3": "1_Acute",
    "d4": "2_Subacute", "d5": "2_Subacute", "d7": "2_Subacute",
    "d9": "3_Late",     "d14": "3_Late",
}
phase_order  = ["0_Uninjured", "1_Acute", "2_Subacute", "3_Late"]
phase_labels = ["Uninjured", "Acute", "Subacute", "Late"]
# ---------------------------

# ── Run ───────────────────────────────────────────────────────
plot_neighborhood_over_time_unified(
    adata=adata_crush_unified,
    tree=tree_unified,
    cell_pops=cell_pops,
    timepoint_col="day",
    celltype_col="combined_tangram",
    normalized=normalized,
    n_neighbors=n_neighbors,
    tangram_threshold=tangram_threshold,
    mode=mode,
    use_phases=use_phases,
    phase_map=phase_map,
    phase_order=phase_order,
    phase_labels=phase_labels,
    exclude_celltypes=[],
    show_celltypes=show_celltypes,           # None = all, or list of specific types
    include_fibro_interactions=include_fibro_interactions,
    custom_palette=None,
    save_dir=save_dir
)

In [ ]:
def plot_neighborhood_summary_heatmap(adata, tree, cell_pops,
                                       timepoint_col="day",
                                       celltype_col="combined_tangram_broad",
                                       normalized=False, n_neighbors=10,
                                       tangram_threshold=0.003,
                                       use_phases=True,
                                       phase_map=None,
                                       phase_order=None,
                                       phase_labels=None,
                                       include_fibro_interactions=True,
                                       exclude_celltypes=None,
                                       show_celltypes=None,
                                       annot=False,           # show numbers or not
                                       save_dir=None, dpi=150):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    day_order = ["uninjured", "d1", "d2", "d3", "d4", "d5", "d7", "d9", "d14"]

    def sort_timepoint(tp):
        return day_order.index(tp) if tp in day_order else 99

    timepoints = sorted(adata.obs[timepoint_col].unique().tolist(),
                        key=sort_timepoint)

    coords          = adata.obsm["spatial"]
    combined_array  = adata.obs[celltype_col].values
    subtype_array   = adata.obs["fibro_subtype"].values
    timepoint_array = adata.obs[timepoint_col].values

    if exclude_celltypes is None:
        exclude_celltypes = []

    all_cats = adata.obs[celltype_col].unique().tolist()
    if include_fibro_interactions:
        target_celltypes = [ct for ct in all_cats
                            if ct != "Fibroblasts"
                            and ct not in exclude_celltypes]
    else:
        target_celltypes = [ct for ct in all_cats
                            if ct not in cell_pops
                            and ct != "Fibroblasts"
                            and ct not in exclude_celltypes]
    target_celltypes = sorted(target_celltypes)

    if show_celltypes is not None:
        display_celltypes = [ct for ct in show_celltypes if ct in target_celltypes]
    else:
        display_celltypes = target_celltypes

    # ── Compute scores ────────────────────────────────────────
    print("Computing neighborhood scores...")
    all_results = {}

    for tp in timepoints:
        tp_mask       = timepoint_array == tp
        tp_fibro_mask = tp_mask & np.isin(combined_array, cell_pops)
        tp_fibro_idx  = np.where(tp_fibro_mask)[0]

        if len(tp_fibro_idx) == 0:
            continue

        fibro_coords_tp = coords[tp_fibro_idx]
        _, indices_tp   = tree.query(fibro_coords_tp, k=n_neighbors + 1)
        indices_tp      = indices_tp[:, 1:]

        background_tp = pd.Series(combined_array[tp_mask]).value_counts(normalize=True) * 100
        background_tp = background_tp.reindex(target_celltypes, fill_value=0)

        for pop in cell_pops:
            pop_mask_tp = subtype_array[tp_fibro_idx] == pop
            pop_rows    = np.where(pop_mask_tp)[0]

            if pop not in all_results:
                all_results[pop] = {}

            if len(pop_rows) == 0:
                all_results[pop][tp] = {ct: np.nan for ct in target_celltypes}
                continue

            neighbor_idx   = indices_tp[pop_rows].flatten()
            neighbor_cts   = combined_array[neighbor_idx]
            unique, counts = np.unique(neighbor_cts, return_counts=True)
            count_dict     = dict(zip(unique, counts))

            total = sum(count_dict.get(ct, 0) for ct in target_celltypes)
            if total > 0:
                raw = {ct: count_dict.get(ct, 0) / total * 100 for ct in target_celltypes}
            else:
                raw = {ct: 0 for ct in target_celltypes}

            if normalized:
                all_results[pop][tp] = {
                    ct: raw[ct] / background_tp[ct] if background_tp[ct] > 0 else np.nan
                    for ct in target_celltypes
                }
            else:
                all_results[pop][tp] = raw

    # ── Aggregate phases ──────────────────────────────────────
    def aggregate_phases(results_pop):
        phase_results = {}
        for phase in phase_order:
            tps_in_phase = [tp for tp, ph in phase_map.items()
                            if ph == phase and tp in results_pop]
            if not tps_in_phase:
                continue
            phase_data = {}
            for ct in target_celltypes:
                vals = [results_pop[tp][ct] for tp in tps_in_phase
                        if not np.isnan(results_pop[tp].get(ct, np.nan))]
                phase_data[ct] = np.mean(vals) if vals else np.nan
            phase_results[phase] = phase_data
        return phase_results

    # ── Build combined matrix ─────────────────────────────────
    print("Building summary matrix...")
    dfs = []
    for pop in cell_pops:
        if pop not in all_results:
            continue

        if use_phases and phase_map is not None:
            phase_results = aggregate_phases(all_results[pop])
            df_pop = pd.DataFrame(phase_results).T
            df_pop = df_pop.reindex(phase_order).dropna(how="all")
            df_pop = df_pop[display_celltypes]
            col_labels = phase_labels[:len(df_pop)]
        else:
            df_pop = pd.DataFrame(all_results[pop]).T
            df_pop = df_pop.reindex(timepoints).dropna(how="all")
            df_pop = df_pop[display_celltypes]
            col_labels = [tp for tp in df_pop.index]

        df_pop.index = col_labels
        df_T = df_pop.T
        df_T.columns = [f"{pop}\n{c}" for c in df_T.columns]
        dfs.append(df_T)

    full_matrix = pd.concat(dfs, axis=1)
    print(f"Summary matrix shape: {full_matrix.shape}")

    # ── Plot ──────────────────────────────────────────────────
    cmap   = "RdBu_r" if normalized else "YlOrRd"
    center = 1.0 if normalized else None
    fmt    = ".2f" if normalized else ".1f"

    n_rows = full_matrix.shape[0]
    n_cols = full_matrix.shape[1]

    fig, ax = plt.subplots(figsize=(n_cols * 0.8 + 2, n_rows * 0.5 + 2))

    sns.heatmap(
        full_matrix,
        ax=ax,
        cmap=cmap,
        center=center,
        annot=annot,
        fmt=fmt if annot else None,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Enrichment" if normalized else "% neighbors"},
        annot_kws={"size": 7} if annot else None
    )

    # Vertical lines to separate fibro subtypes
    n_phases = len(phase_order) if use_phases else len(timepoints)
    for i in range(1, len(cell_pops)):
        ax.axvline(x=i * n_phases, color="black", linewidth=2)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    ax.set_xlabel("Fibroblast subtype × Phase", fontsize=11)
    ax.set_ylabel("Neighboring cell type", fontsize=11)

    norm_label = "normalized" if normalized else "raw"
    fig.suptitle(
        f"Fibroblast neighborhood interactions across time\n({norm_label}, k={n_neighbors})",
        fontsize=13, y=1.02
    )
    plt.tight_layout()

    if save_dir:
        save_path = f"{save_dir}neighborhood_summary_heatmap_{norm_label}.pdf"
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()


In [ ]:
# ---- Only change these ----
normalized                 = False
use_phases                 = False
include_fibro_interactions = False
show_celltypes             = ["Macrophages_0", "Macrophages_1", "Macrophages_2", "Microglia_0","Microglia_2"] #if all "None"
annot                      = False

# Custom order for fibro subtypes on x axis
cell_pops_ordered = ["PvFib", "Activated_Fib", "Mc5r+", "MyoFib_1", 
                     "MyoFib_2", "Ly6a+", "Sox9+", "Cxcl14+"]

phase_map_no_uninjured    = {
    "d1": "1_Acute",    "d2": "1_Acute",    "d3": "1_Acute",
    "d4": "2_Subacute", "d5": "2_Subacute", "d7": "2_Subacute",
    "d9": "3_Late",     "d14": "3_Late",
}
phase_order_no_uninjured  = ["1_Acute", "2_Subacute", "3_Late"]
phase_labels_no_uninjured = ["Acute", "Subacute", "Late"]
# ---------------------------

plot_neighborhood_summary_heatmap(
    adata=adata_crush_unified,
    tree=tree_unified,
    cell_pops=cell_pops_ordered,            # ordered list
    timepoint_col="day",
    celltype_col="combined_tangram",
    normalized=normalized,
    n_neighbors=n_neighbors,
    tangram_threshold=tangram_threshold,
    use_phases=use_phases,
    phase_map=phase_map_no_uninjured,
    phase_order=phase_order_no_uninjured,
    phase_labels=phase_labels_no_uninjured,
    include_fibro_interactions=include_fibro_interactions,
    show_celltypes=show_celltypes,
    annot=annot,
    save_dir=save_dir
)

In [ ]:
# ── Parameters ────────────────────────────────────────────────
batch_id = "crush_d5_8"

populations = [  
    ("MyoFib_1",      "red"),
    ("Macrophages_3","#5a81de"),
    ("Macrophages_2","#567fe0"), # (population name, color)
]

dot_size = 50
# ─────────────────────────────────────────────────────────────

# Subset to section
mask    = adata_subset.obs["batch"] == batch_id
adata_s = adata_subset[mask]

x = adata_s.obs["x"].values
y = adata_s.obs["y"].values
ct = adata_s.obs["combined_tangram"].astype(str).values

# Build combined mask for background
pop_names   = [p[0] for p in populations]
mask_any    = np.isin(ct, pop_names)
mask_other  = ~mask_any

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(50, 25))

# Background
ax.scatter(x[mask_other], y[mask_other], c="lightgrey",
           s=dot_size * 0.5, zorder=1, rasterized=True, alpha=0.5)

# Each population
for i, (pop, color) in enumerate(populations):
    mask_pop = ct == pop
    print(f"{pop}: {mask_pop.sum()} spots")
    ax.scatter(x[mask_pop], y[mask_pop], c=color, s=dot_size,
               label=pop, zorder=i + 2, alpha=1, rasterized=True)

ax.legend(title="Population", fontsize=10, markerscale=3,
          bbox_to_anchor=(1.01, 0.5), loc="center left")
ax.set_title(f"{batch_id}", fontsize=13)
ax.axis("off")

plt.tight_layout()

pop_label = "_".join(pop_names)
plt.savefig(f"{save_dir}{batch_id}_{pop_label}.png",
            dpi=150, bbox_inches="tight")
plt.show()

## Margehrita dataset annotation

In [ ]:
ad_atlas

In [ ]:
import scipy.sparse as sp
import scanpy as sc
import gc

ad_atlas = sc.read_h5ad("/path/to/your/file.h5ad")


In [ ]:
import scanpy as sc

# Your genes of interest
genes = ['Pecam1', 'Vwf', 'Col1a1', 'Rgs5', "Atp13a5", "Vcam1", "Ly6c1", "Tagln", "Acta2"]  # ← replace with your genes

sc.pl.dotplot(
    ad_atlas,
    var_names=genes,
    groupby='celltype_wsnn_res.0.2',
    standard_scale='var',   # normalize color per gene
    figsize=(8, 15),
    show=True,
    save='perivascular_dotplot.pdf'
)

In [ ]:
# Your genes of interest
genes = ['Ccr2', 'Ptprc', 'Mrc1', 'Ifng', "Ly6c2", "Plac8", "Ms4a7", "Gpr84", "Cd68", "Cd80", "Cd86"]  # ← replace with your genes

sc.pl.dotplot(
    ad_atlas,
    var_names=genes,
    groupby='celltype_wsnn_res.0.2',
    standard_scale='var',   # normalize color per gene
    figsize=(8, 15),
    show=True,
    save='macrophages_dotplot.pdf'
)

## Spatial dataset annotation with more cell types

In [ ]:
adata_reclustered = sc.read_h5ad("/path/to/your/file.h5ad")

# ── Transfer leiden_2.0 to adata_subset ──────────────────────
print("Transferring leiden_2.0...")

overlap = adata_reclustered.obs_names.isin(adata_subset.obs_names).sum()
print(f"Matching barcodes: {overlap} / {adata_reclustered.shape[0]}")

adata_subset.obs["leiden_2"] = np.nan
adata_subset.obs.loc[adata_reclustered.obs_names, "leiden_2"] = \
    adata_reclustered.obs["leiden_2.0"].astype(str).values

print("\nleiden_2 distribution:")
print(adata_subset.obs["leiden_2"].value_counts())
print(f"NaN (non-crush spots): {adata_subset.obs['leiden_2'].isna().sum()}")

# ── Clear uns before saving ───────────────────────────────────
adata_subset.uns = {}

# ── Save ──────────────────────────────────────────────────────
adata_subset.write_h5ad(f"{proj_dir}adata_subset_unified.h5ad")
print("Saved!")

In [ ]:
# ── Rebuild adata_crush_unified with leiden_2 ─────────────────
crush_conditions = [c for c in adata_subset.obs["inj_type_per_day"].unique()
                    if c.startswith("crush") or c == "uninjured_uninjured"]

adata_crush_unified = adata_subset[
    adata_subset.obs["inj_type_per_day"].isin(crush_conditions)
].copy()

print(f"adata_crush_unified: {adata_crush_unified.shape[0]} spots")
print(adata_crush_unified.obs["leiden_2"].value_counts())

# ── Rebuild KDTree ─────────────────────────────────────────────
from sklearn.neighbors import KDTree
print("Building KDTree...")
tree_unified = KDTree(adata_crush_unified.obsm["spatial"])
print("Done!")

# ── Rerun neighbourhood using leiden_2 ────────────────────────
plot_neighborhood_summary_heatmap(
    adata=adata_crush_unified,
    tree=tree_unified,
    cell_pops=cell_pops_ordered,
    timepoint_col="day",
    celltype_col="leiden_2",            # use leiden_2 as cell type
    normalized=normalized,
    n_neighbors=n_neighbors,
    tangram_threshold=tangram_threshold,
    use_phases=use_phases,
    phase_map=phase_map_no_uninjured,
    phase_order=phase_order_no_uninjured,
    phase_labels=phase_labels_no_uninjured,
    include_fibro_interactions=include_fibro_interactions,
    show_celltypes=None,
    annot=annot,
    save_dir=save_dir
)

# Immune cell annotation

In [ ]:
save_dir = "/path/to/output/"
proj_dir = "/path/to/project/"
os.makedirs(proj_dir, exist_ok=True)

# ── Step 1 — Load objects ─────────────────────────────────────
print("Loading objects...")
# ad_atlas     = sc.read_h5ad(f"{save_dir}MZ_RNA.h5ad")
adata_subset = sc.read_h5ad(f"{save_dir}adata_subset_tangram.h5ad")
MZ_immune_cells = sc.read_h5ad("/path/to/your/file.h5ad")

# print(f"Atlas  : {ad_atlas.shape}")
print(f"Spatial: {adata_subset.shape}")


## Subset only immune cells from spatial

In [ ]:
adata_immune = adata_subset[adata_subset.obs['celltype'] == 'Immune_cells'].copy()


In [ ]:
adata_immune.X = adata_immune.layers['normalized']


In [ ]:
sc.pl.umap(adata_immune, color='celltype', frameon=False, size = 10)


## Subset only immune cells from Margherita

In [ ]:
save_dir = "/path/to/output/"
proj_dir = "/path/to/project/"
os.makedirs(proj_dir, exist_ok=True)

ad_atlas = sc.read_h5ad(f"{save_dir}MZ_RNA.h5ad")

In [ ]:
ad_atlas

In [ ]:
sc.pl.umap(ad_atlas, color='clusters_forsubcl', frameon=False, legend_loc='on data')


In [ ]:
MZ_immune_cells = ad_atlas[ad_atlas.obs['clusters_forsubcl'].isin(['Macrophages', 'Microglia'])].copy()


In [ ]:
color_map = {
    'Dendritic_cells':    "#87C254",
    'MG_High_mito':       "#00B4FF",
    'MG_Homeostatic':     "#C2DEFF",
    'MG_Ifn':             "#2E31C0",
    'MG_Inflammatory':    "#1F77FF",
    'MG_Lipid_metabo':    "#93D9FF",
    'MG_Prolif':          "#00325D",
    'MG_Repair':          "#00C7E4",
    'MG_migrating':       "#00A7AE",
    'Macro_Inflammatory': "#FBB03B",
    'Macro_chemotaxis':   "#F15A24",
    'Monocytes':          "#D62728",
    'T_cells':            "#93278F",
}

sc.pl.umap(MZ_immune_cells, color='celltype_annotated', frameon=False, legend_loc='right margin', size = 20, palette = color_map)


In [ ]:
# Your genes of interest
genes = ['Ptprc', 'Itgam', 'Hexb','P2ry12','Trem2', 'Gpnmb', 'Ms4a7','Cd63', 'Spp1', 'Mki67',"Isg15", "Irf7","Cxcr4",'Msr1', 'Igf1', 'Ccr2', 'Cebpb', 'Arg1', "Hmox1", "Ccl7",'H2-Aa', 'Cd3e']  # ← replace with your genes

cluster_order = [ 'MG_Homeostatic', 'MG_Inflammatory','MG_High_mito','MG_Lipid_metabo','MG_migrating','MG_Repair','MG_Prolif','MG_Ifn',
                  'Monocytes','Macro_Inflammatory','Macro_chemotaxis', 'Dendritic_cells','T_cells'
]

sc.pl.dotplot(
    MZ_immune_cells,
    var_names=genes,
    groupby='celltype_annotated',
    standard_scale='var',
    figsize=(15, 6),
    swap_axes=True,
    categories_order=cluster_order,
    show=True
)


In [ ]:
print(MZ_immune_cells.X.min(), MZ_immune_cells.X.max())  # should be 0 to ~10

## Run Tangram version 1 

In [ ]:
print("MZ_immune_cells X range:", MZ_immune_cells.X.min(), MZ_immune_cells.X.max())
print("adata_immune X range:", adata_immune.X.min(), adata_immune.X.max())

### To load

In [ ]:
proj_dir = "/path/to/project/"
save_dir = "/path/to/output/"
os.makedirs(proj_dir, exist_ok=True)

adata_immune = sc.read_h5ad(f"{proj_dir}adata_immune_tangram.h5ad")
ad_map_immune = sc.read_h5ad(f"{proj_dir}ad_map_immune.h5ad")
ad_ge = sc.read_h5ad(f"{proj_dir}ad_ge.h5ad")
MZ_immune_cells = sc.read_h5ad(f"{proj_dir}MZ_immune_cells_tangram.h5ad")

# Restore variables
cell_types = adata_immune.obsm["tangram_ct_pred"].columns.tolist()


In [ ]:
adata_subset = sc.read_h5ad(f"{save_dir}adata_subset_tangram.h5ad")


In [ ]:
# Downcast both dense matrices

ad_ge.X = ad_ge.X.astype(np.float32)
adata_immune.X = adata_immune.X.astype(np.float32)

### Manual selection of informative genes for the projection

In [ ]:
sc.pl.umap(MZ_immune_cells, 
           color=['condition', 'celltype_wsnn_res.0.3', 'celltype_annotated'], 
           frameon=False, legend_loc='right margin',
           ncols=3)

In [ ]:
sc.pl.umap(adata_immune, 
           color=['celltype'], 
           frameon=False, legend_loc='right margin',
           ncols=3)

In [ ]:
# Step 1: get the intersection (mandatory)
shared_genes = [g for g in MZ_immune_cells.var_names 
                if g in adata_immune.var_names]

# Step 2: subset scRNA-seq to shared genes and rank
MZ_immune_cells_500genes = MZ_immune_cells[:, shared_genes].copy()


In [ ]:
# 1. Genes present in both datasets
shared_genes = [g for g in adata_immune.var_names 
                if g in MZ_immune_cells_500genes.var_names]

# 2. Overall mean expression correlation across the 500 genes
sp_mean = np.array(adata_immune[:, shared_genes].X.mean(axis=0)).flatten()
sc_mean = np.array(MZ_immune_cells_500genes[:, shared_genes].X.mean(axis=0)).flatten()

gene_corr_df = pd.DataFrame({
    "gene": shared_genes,
    "mean_sc": sc_mean,
    "mean_sp": sp_mean,
    "log_sc": np.log1p(sc_mean),
    "log_sp": np.log1p(sp_mean),
})


In [ ]:
# 3. Filter: expressed in spatial + concordant overall
from scipy.stats import spearmanr

gene_corr_df["ratio"] = gene_corr_df["log_sc"] / (gene_corr_df["log_sp"] + 1e-6)

good_genes = gene_corr_df[
    (gene_corr_df["mean_sp"] > 0.1) &     # expressed in spatial
    (gene_corr_df["mean_sc"] > 0.1) &     # expressed in scRNAseq
    (gene_corr_df["ratio"] < 10)           # not wildly discordant between datasets
]["gene"].tolist()

# Print the list
print(f"Number of selected genes: {len(good_genes)}")
print(sorted(good_genes))

# Visualize on the correlation plot
fig, ax = plt.subplots(figsize=(7, 7))

# Plot all shared genes in grey
ax.scatter(
    gene_corr_df["log_sc"], 
    gene_corr_df["log_sp"], 
    alpha=1, 
    color="lightgrey", 
    label="excluded genes"
)

# Highlight selected genes in blue
selected_df = gene_corr_df[gene_corr_df["gene"].isin(good_genes)]
ax.scatter(
    selected_df["log_sc"], 
    selected_df["log_sp"], 
    alpha=1, 
    color="black", 
    label="selected genes"
)

#for g in good_genes:  # your immune markers from option 2
    #row = gene_corr_df[gene_corr_df["gene"] == g]
    #if len(row):
        #ax.annotate(
    #g,
    #xy=(row["log_sc"].values[0], row["log_sp"].values[0]),       # point location
    #xytext=(3, 1),                                                # offset in pixels
    #textcoords="offset points",
    #fontsize=7,
    #arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

corr, pval = spearmanr(gene_corr_df["log_sc"], gene_corr_df["log_sp"])
print(f"Spearman r={corr:.2f}, p={pval:.2e}")

ax.set_xlabel("log mean expression (Zamboni scRNA-seq)")
ax.set_ylabel("log mean expression (MERSCOPE spatial)")
ax.set_title(f"Gene expression concordance\nSpearman r={corr:.2f} | {len(good_genes)} genes selected")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(f"{save_dir}gene_concordance.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tg.pp_adatas(MZ_immune_cells, adata_immune, genes=good_genes)
print(f"Genes used for mapping: {len(MZ_immune_cells.uns['training_genes'])}")
print(MZ_immune_cells.uns['training_genes'])

In [ ]:
# Run mapping (ad_map is the mapping matrix)
ad_map_immune = tg.map_cells_to_space(
    adata_sc=MZ_immune_cells,
    adata_sp=adata_immune,
    mode="clusters",
    cluster_label="celltype_annotated",
    density_prior="rna_count_based",  # or pass your custom density array
    num_epochs=1000,                  # increase from 300, cluster mode needs more
    learning_rate=0.01,                # default, reduce to 0.01 if loss is unstable
    lambda_g1=1,                      # main gene expression loss — keep at 1
    lambda_g2=0,                    # spatial smoothing — helps with immune cells
    lambda_r=0,                     # sparsity of mapping — tune between 0.01-1
    lambda_d=0,                       # density prior weight — increase if density matters
    random_state=42,                  # reproducibility
    device="cpu",
    verbose=True
)

ad_map_immune.write_h5ad(f"{save_dir}ad_map_immune.h5ad")
print("ad_map_immune saved!")

In [ ]:
tg.plot_training_scores(ad_map_immune, bins=30, alpha=0.5)


### Cell projection

In [ ]:
tg.project_cell_annotations(
    ad_map_immune,
    adata_immune,
    annotation="celltype_annotated"
)

In [ ]:
# Transfer to obs

# adata_immune.obsm["tangram_ct_pred"]  
# DataFrame of shape (229018 spots × 13 cell types)
# each value = predicted proportion of that cell type at that spot
# this IS your cell type prediction

for ct in adata_immune.obsm["tangram_ct_pred"].columns:
    adata_immune.obs[ct] = adata_immune.obsm["tangram_ct_pred"][ct].values

# adata_immune.obs["predicted_celltype"]  
# the dominant cell type per spot (from idxmax)
# single categorical label per spot

adata_immune.obs["predicted_celltype"] = (
    adata_immune.obsm["tangram_ct_pred"].idxmax(axis=1)
)

adata_immune.write_h5ad(f"{proj_dir}adata_immune_tangram.h5ad")
ad_map_immune.write_h5ad(f"{proj_dir}ad_map_immune.h5ad")
ad_ge.write_h5ad(f"{proj_dir}ad_ge.h5ad")


In [ ]:
# Plot predicted proportions per spatial spot

# ---- Parameters to change ----
batch_to_plot = ["crush_d7_1"]   
populations = "all"                  # "all" or a single string or list
panel_size = (5, 4)                              # size per panel
# ------------------------------

# Select populations
if populations == "all":
    selected_pops = cell_types
elif isinstance(populations, str):
    selected_pops = [populations]
else:
    selected_pops = populations

# One row per batch, one column per population
fig, axes = plt.subplots(
    len(batch_to_plot), len(selected_pops),
    figsize=(panel_size[0] * len(selected_pops), panel_size[1] * len(batch_to_plot))
)

# Handle case of single row or column
if len(batch_to_plot) == 1:
    axes = [axes]
if len(selected_pops) == 1:
    axes = [[ax] for ax in axes]

for i, batch in enumerate(batch_to_plot):
    adata_batch = adata_immune[adata_immune.obs["batch"] == batch].copy()
    
    for j, pop in enumerate(selected_pops):
        sc.pl.embedding(
            adata_batch,
            basis="spatial",
            color=pop,
            frameon=False,
            vmax="p99",
            cmap="Reds",
            title=f"{batch} - {pop}",
            ax=axes[i][j],
            show=False
        )

plt.tight_layout()
plt.show()

In [ ]:
# 'Dendritic_cells', 'MG_High_mito', 'MG_Homeostatic', 'MG_Ifn', 'MG_Inflammatory', 
# 'MG_Lipid_metabo', 'MG_Prolif', 'MG_Repair', 'MG_migrating', 'Macro_Inflammatory', 'Macro_chemotaxis', 'Monocytes', 'T_cells'

# Sections to use: "uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2", 
#                  "myelin_d1_1", "myelin_d3_4", "myelin_d4_3", "myelin_d5_1", "myelin_d7_11", "myelin_d9_9", "myelin_d14_11", 
#                  "liver_d1_2", "liver_d3_4", "liver_d4_1", "liver_d5_6", "liver_d7_14", "liver_d9_11", "liver_d14_11"


color_map = {
    'Dendritic_cells':    "#87C254",
    'MG_High_mito':       "#00B4FF",
    'MG_Homeostatic':     "#C2DEFF",
    'MG_Ifn':             "#2E31C0",
    'MG_Inflammatory':    "#1F77FF",
    'MG_Lipid_metabo':    "#93D9FF",
    'MG_Prolif':          "#00325D",
    'MG_Repair':          "#00C7E4",
    'MG_migrating':       "#00A7AE",
    'Macro_Inflammatory': "#FBB03B",
    'Macro_chemotaxis':   "#FBB03B",
    'Monocytes':          "#D62728",
    'T_cells':            "#93278F",
}

# ---- Parameters to change ----
batch_to_plot = ["crush_d9_1"]
populations = "all"   # "all" or e.g. ["MG_Inflammatory", "CD8_T"]
panel_size = (30, 15)
# ------------------------------

# Mask spots not in selected populations
adata_plot = adata_immune[adata_immune.obs["batch"].isin(batch_to_plot)].copy()

if populations != "all":
    selected_pops = [populations] if isinstance(populations, str) else populations
    adata_plot = adata_plot[adata_plot.obs["predicted_celltype"].isin(selected_pops)].copy()

fig, axes = plt.subplots(
    1, len(batch_to_plot),
    figsize=(panel_size[0] * len(batch_to_plot), panel_size[1])
)

if len(batch_to_plot) == 1:
    axes = [axes]

for ax, batch in zip(axes, batch_to_plot):
    adata_batch = adata_plot[adata_plot.obs["batch"] == batch].copy()
    
    sc.pl.embedding(
        adata_batch,
        basis="spatial",
        color="predicted_celltype",
        frameon=False,
        title=batch,
        ax=ax,
        show=False,
        size=150,
        palette = color_map
    )

plt.tight_layout()
plt.show()

### Gene projection

In [ ]:
# Check the mapping between cluster numbers and cell type names
cluster_mapping = MZ_immune_cells.obs["celltype_annotated"].astype("category")
print(cluster_mapping.cat.categories)  # index position = cluster number

In [ ]:
import anndata as ad
import pandas as pd

# Get cluster categories in order
categories = MZ_immune_cells.obs["celltype_annotated"].astype("category").cat.categories

# Average expression per cluster, indexed by number (0, 1, 2...)
cluster_means = pd.DataFrame(
    index=[str(i) for i in range(len(categories))],  # "0", "1", "2"... to match ad_map_immune
    columns=MZ_immune_cells.var_names,
    data=[
        MZ_immune_cells[MZ_immune_cells.obs["celltype_annotated"] == ct].X.mean(axis=0).A1
        for ct in categories
    ]
)

# Build AnnData
adata_sc_cluster = ad.AnnData(
    X=cluster_means.values,
    obs=pd.DataFrame(index=cluster_means.index),
    var=MZ_immune_cells.var
)

print(adata_sc_cluster.obs.index)   # should be ['0','1',...,'12']
print(ad_map_immune.obs.index)      # should match

# Now project
ad_ge = tg.project_genes(
    adata_map=ad_map_immune,
    adata_sc=adata_sc_cluster
)

ad_ge.write_h5ad(f"{save_dir}ad_ge_immune.h5ad")
print("ad_ge_immune saved!")

In [ ]:
# Transfer spatial coordinates and batch info from adata_immune to ad_ge
ad_ge.obsm["spatial"] = adata_immune.obsm["spatial"]
ad_ge.obs["batch"] = adata_immune.obs["batch"].values


In [ ]:
# ---- Parameters to change ----
batch_to_plot = ["liver_d9_11", "myelin_d9_9", "crush_d9_1"]
genes_to_plot = ["skap1"]  # note: mouse genes are lowercase in Zamboni
panel_size = (5, 4)
# ------------------------------

fig, axes = plt.subplots(
    len(batch_to_plot), len(genes_to_plot),
    figsize=(panel_size[0] * len(genes_to_plot), panel_size[1] * len(batch_to_plot))
)

if len(batch_to_plot) == 1:
    axes = [axes]
if len(genes_to_plot) == 1:
    axes = [[ax] for ax in axes]

for i, batch in enumerate(batch_to_plot):
    ad_ge_batch = ad_ge[ad_ge.obs["batch"] == batch].copy()
    
    for j, gene in enumerate(genes_to_plot):
        sc.pl.embedding(
            ad_ge_batch,
            basis="spatial",
            color=gene,
            frameon=False,
            cmap="Reds",
            vmax="p99",
            sort_order=True,
            title=f"{batch} - {gene}",
            ax=axes[i][j],
            show=False, 
            size = 200
        )

plt.tight_layout()
plt.show()

In [ ]:
# ---- Parameters to change ----
batch_to_plot = ["crush_d5_8", "crush_d7_1", "crush_d9_1"]
genes_to_plot = ["stat4", "cd3e", "mrc1", "hexb"]  # note: mouse genes are lowercase in Zamboni
panel_size = (5, 4)
# ------------------------------

fig, axes = plt.subplots(
    len(batch_to_plot), len(genes_to_plot),
    figsize=(panel_size[0] * len(genes_to_plot), panel_size[1] * len(batch_to_plot))
)

if len(batch_to_plot) == 1:
    axes = [axes]
if len(genes_to_plot) == 1:
    axes = [[ax] for ax in axes]

for i, batch in enumerate(batch_to_plot):
    ad_ge_batch = ad_ge[ad_ge.obs["batch"] == batch].copy()
    
    for j, gene in enumerate(genes_to_plot):
        sc.pl.embedding(
            ad_ge_batch,
            basis="spatial",
            color=gene,
            frameon=False,
            cmap="Reds",
            vmax="p99",
            sort_order=True,
            title=f"{batch} - {gene}",
            ax=axes[i][j],
            show=False
        )

plt.tight_layout()
plt.show()

In [ ]:
# Get the cluster names in the correct order
cluster_names = MZ_immune_cells.obs['celltype_annotated'].astype('category').cat.categories.tolist()
print(len(cluster_names))  # should be 17 to match ad_map_immune

# Rename the obs index
ad_map_immune.obs.index = cluster_names
print(ad_map_immune.obs.index.tolist())  # should now show named clusters

### Correlation to evaluate the projection

In [ ]:
from scipy.stats import spearmanr

ad_ge.var_names = ad_ge.var_names.str.capitalize()

# Find genes present in ALL three datasets
genes_adge = set(ad_ge.var_names)
genes_sp = set(adata_immune.var_names)
genes_sc = set(MZ_immune_cells.var_names)

# Intersection
common_genes = list(genes_adge & genes_sp & genes_sc)
print(f"Common genes across all three: {len(common_genes)}")

# Now use these directly — no case conversion needed
pred_mean = np.array(ad_ge[:, common_genes].X.mean(axis=0)).flatten()
sp_mean = np.array(adata_immune[:, common_genes].X.mean(axis=0)).flatten()
sc_mean = np.array(MZ_immune_cells[:, common_genes].X.mean(axis=0)).flatten()

# Correlations
corr_sp, pval_sp = spearmanr(pred_mean, sp_mean)
corr_sc, pval_sc = spearmanr(pred_mean, sc_mean)
corr_sp_sc, pval_sp_sc = spearmanr(sp_mean, sc_mean)

print(f"Predicted vs Spatial: r={corr_sp:.3f}")
print(f"Predicted vs Zamboni: r={corr_sc:.3f}")
print(f"Spatial vs Zamboni: r={corr_sp_sc:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Predicted vs Spatial
axes[0].scatter(np.log1p(pred_mean), np.log1p(sp_mean), alpha=1, color="black")
axes[0].set_xlabel("log mean predicted expression")
axes[0].set_ylabel("log mean spatial expression (MERSCOPE)")
axes[0].set_title(f"Predicted vs Spatial\nSpearman r={corr_sp:.3f}")

# Plot 2: Predicted vs Zamboni
axes[1].scatter(np.log1p(pred_mean), np.log1p(sc_mean), alpha=1, color="black")
axes[1].set_xlabel("log mean predicted expression")
axes[1].set_ylabel("log mean expression (Zamboni)")
axes[1].set_title(f"Predicted vs Zamboni\nSpearman r={corr_sc:.3f}")

# Plot 3: Merscope vs Zamboni
axes[2].scatter(np.log1p(sp_mean), np.log1p(sc_mean), alpha=1, color="black")
axes[2].set_xlabel("log mean spatial expression (MERSCOPE)")
axes[2].set_ylabel("log mean expression (Zamboni)")
axes[2].set_title(f"Spatial vs Zamboni\nSpearman r={corr_sp_sc:.3f}")

# Get training genes in the correct case
#training_genes_matched = [g for g in common_genes 
#                          if g.lower() in [t.lower() for t in MZ_immune_cells.uns["training_genes"]]]

#for ax, y_vals in zip(axes, [sp_mean, sc_mean]):
#    texts = []
#    for g in training_genes_matched:
#        idx = common_genes.index(g)
#        texts.append(ax.text(np.log1p(pred_mean[idx]), np.log1p(y_vals[idx]), g, fontsize=6))

plt.tight_layout()
plt.savefig(f"{save_dir}gene_concordance_all.pdf", dpi=300, bbox_inches="tight")
plt.show()

print(f"Predicted vs Spatial: r={corr_sp:.3f}")
print(f"Predicted vs Zamboni: r={corr_sc:.3f}")
print(f"Spatial vs Zamboni: r={corr_sp_sc:.3f}")

In [ ]:
# Get hexb expression
pred_hexb = pd.Series(
    np.array(ad_ge[:, "zeb1"].X).flatten(),
    index=ad_ge.obs_names
)
sp_hexb = pd.Series(
    np.array(adata_immune[:, "Zeb1"].X).flatten(),
    index=adata_immune.obs_names
)

# Normalize to 0-1
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

# Assign only to matching cells
common_cells = adata_immune.obs_names.intersection(ad_ge.obs_names)
print(f"Common cells: {len(common_cells)}")

adata_immune.obs["csf1r_predicted_norm"] = minmax(pred_hexb).reindex(adata_immune.obs_names)
adata_immune.obs["csf1r_spatial_norm"] = minmax(sp_hexb)

# Plot
batch = "crush_d7_1"
adata_batch = adata_immune[adata_immune.obs["batch"] == batch].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.embedding(adata_batch, basis="spatial", color="csf1r_predicted_norm",
    frameon=False, cmap="Reds", vmin=0, vmax=1,
    title="hexb - predicted (normalized)", ax=axes[0], show=False)

sc.pl.embedding(adata_batch, basis="spatial", color="csf1r_spatial_norm",
    frameon=False, cmap="Reds", vmin=0, vmax=1,
    title="hexb - measured (normalized)", ax=axes[1], show=False)

plt.tight_layout()
plt.show()

### Projection of cell_type predicted of immune cell on the spatial object with all the cells

In [ ]:
# Add predicted_celltype to adata_subset, non-immune cells will be NaN
adata_subset.obs["immune_celltype"] = np.nan  # default = NaN for non-immune cells

# Transfer only for immune cells using index alignment
adata_subset.obs.loc[
    adata_immune.obs_names, "immune_celltype"
] = adata_immune.obs["predicted_celltype"].values

# Check
print(adata_subset.obs["immune_celltype"].value_counts(dropna=False))

In [ ]:
# Sections to use: "uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2", 
#                  "myelin_d1_1", "myelin_d3_4", "myelin_d4_3", "myelin_d5_1", "myelin_d7_11", "myelin_d9_9", "myelin_d14_11", 
#                  "liver_d1_2", "liver_d3_4", "liver_d4_1", "liver_d5_6", "liver_d7_14", "liver_d9_11", "liver_d14_11"

# ---- Parameters to change ----
batch_to_plot = ["uninjured_1"]
populations = "all"
panel_size = (8, 5)
grey_alpha = 0.2
cell_types = adata_immune.obsm["tangram_ct_pred"].columns.tolist()
save_path = f"{proj_dir}immune_celltype_{'_'.join(batch_to_plot)}.pdf"

# ---- Color palette ----
color_map = {
    'Dendritic_cells':    "#87C254",
    'MG_High_mito':       "#00B4FF",
    'MG_Homeostatic':     "#C2DEFF",
    'MG_Ifn':             "#2E31C0",
    'MG_Inflammatory':    "#1F77FF",
    'MG_Lipid_metabo':    "#93D9FF",
    'MG_Prolif':          "#00325D",
    'MG_Repair':          "#00C7E4",
    'MG_migrating':       "#00A7AE",
    'Macro_Inflammatory': "#FBB03B",
    'Macro_chemotaxis':   "#FBB03B",
    'Monocytes':          "#D62728",
    'T_cells':            "#93278F",
}
# ------------------------------

# Select populations
if populations == "all":
    selected_pops = cell_types
elif isinstance(populations, str):
    selected_pops = [populations]
else:
    selected_pops = populations

# Filter immune cell types
adata_subset.obs["immune_celltype_filtered"] = adata_subset.obs["immune_celltype"].copy()
adata_subset.obs.loc[
    ~adata_subset.obs["immune_celltype"].isin(selected_pops),
    "immune_celltype_filtered"
] = np.nan

# Plot
fig, axes = plt.subplots(1, len(batch_to_plot),
    figsize=(panel_size[0] * len(batch_to_plot), panel_size[1]))

if len(batch_to_plot) == 1:
    axes = [axes]

for ax, batch in zip(axes, batch_to_plot):
    adata_batch = adata_subset[adata_subset.obs["batch"] == batch].copy()

    # Plot grey background cells first
    is_nan = adata_batch.obs["immune_celltype_filtered"].isna()
    coords = adata_batch.obsm["spatial"]
    ax.scatter(
        coords[is_nan, 0], coords[is_nan, 1],
        c="lightgrey", s=30, alpha=grey_alpha, rasterized=True
    )

    # Plot immune cells on top
    adata_immune_batch = adata_batch[~is_nan].copy()

    # Convert to categorical and build matching palette
    adata_immune_batch.obs["immune_celltype_filtered"] = (
        adata_immune_batch.obs["immune_celltype_filtered"].astype("category")
    )
    categories = adata_immune_batch.obs["immune_celltype_filtered"].cat.categories.tolist()
    palette = [color_map[ct] for ct in categories]

    sc.pl.embedding(
        adata_immune_batch,
        basis="spatial",
        color="immune_celltype_filtered",
        palette=palette,
        frameon=False,
        title=batch,
        ax=ax,
        show=False,
        size=200
    )

plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight", format="pdf")
plt.show()
print(f"Saved to {save_path}")

In [ ]:
color_map = {
    'Dendritic_cells':    "#87C254",
    'MG_High_mito':       "#00B4FF",
    'MG_Homeostatic':     "#C2DEFF",
    'MG_Ifn':             "#2E31C0",
    'MG_Inflammatory':    "#1F77FF",
    'MG_Lipid_metabo':    "#93D9FF",
    'MG_Prolif':          "#00325D",
    'MG_Repair':          "#00C7E4",
    'MG_migrating':       "#00A7AE",
    'Macro_Inflammatory': "#FBB03B",
    'Macro_chemotaxis':   "#FBB03B",
    'Monocytes':          "#D62728",
    'T_cells':            "#93278F",
}

# ---- Parameters to change ----
inj_types_to_plot = ["myelin", "liver"]
cell_type_to_plot = "all"              # "all" or e.g. "MG_Inflammatory"
time_mode = "phase"                    # "individual" or "phase"
group_celltypes = True                 # True = grouped, False = individual cell types
# ------------------------------

# ---- Cell type grouping ----
celltype_groups = {
    "Microglia":    ["MG_High_mito", "MG_Homeostatic", "MG_Ifn", "MG_Inflammatory",
                     "MG_Lipid_metabo", "MG_Prolif", "MG_Repair", "MG_migrating"],
    "Macrophages":  ["Macro_Inflammatory", "Macro_chemotaxis"],
    "Dendritic_cells": ["Dendritic_cells"],
    "Monocytes":    ["Monocytes"],
    "T_cells":      ["T_cells"],
}
group_colors = {
    "Microglia":       "steelblue",
    "Macrophages":     "salmon",
    "Dendritic_cells": "mediumseagreen",
    "Monocytes":       "orange",
    "T_cells":         "mediumpurple",
}

# Individual cell type colors
individual_colors = dict(zip(cell_types, plt.cm.tab20.colors[:len(cell_types)]))

# Phase definitions
phase_map = {
    "0":  "uninjured",
    "1":  "acute", "2": "acute", "3": "acute",
    "4":  "subacute", "5": "subacute", "7": "subacute",
    "9":  "late", "14": "late"
}
phase_order = ["uninjured", "acute", "subacute", "late"]

# Filter
mask = (adata_subset.obs["inj_type"].isin(inj_types_to_plot)) | \
       (adata_subset.obs["inj_type"] == "uninjured")
adata_inj = adata_subset[mask].copy()

# Fix day column
adata_inj.obs["day_plot"] = adata_inj.obs["day"].astype(str).str.replace("d", "", regex=False)
adata_inj.obs.loc[adata_inj.obs["inj_type"] == "uninjured", "day_plot"] = "0"
adata_inj.obs["phase"] = adata_inj.obs["day_plot"].map(phase_map)

days = sorted(
    [d for d in adata_inj.obs["day_plot"].unique() if d != "0"],
    key=lambda x: int(x)
)

# ---- Decide which labels to use ----
if group_celltypes:
    plot_labels = list(celltype_groups.keys())
    colors = group_colors
else:
    plot_labels = cell_types
    colors = individual_colors

# ---- Build dataframe ----
rows = []

def compute_rows(adata_sub, day, inj, phase):
    total_cells = len(adata_sub)
    immune_mask = adata_sub.obs["immune_celltype"].notna()
    total_immune = immune_mask.sum()

    if group_celltypes:
        for group, members in celltype_groups.items():
            count = adata_sub.obs["immune_celltype"].isin(members).sum()
            rows.append({
                "day": day, "inj_type": inj, "phase": phase, "cell_type": group,
                "raw": count,
                "norm_immune": count / total_immune if total_immune > 0 else 0,
                "norm_total": count / total_cells if total_cells > 0 else 0,
            })
    else:
        for ct in cell_types:
            count = (adata_sub.obs["immune_celltype"] == ct).sum()
            rows.append({
                "day": day, "inj_type": inj, "phase": phase, "cell_type": ct,
                "raw": count,
                "norm_immune": count / total_immune if total_immune > 0 else 0,
                "norm_total": count / total_cells if total_cells > 0 else 0,
            })

if time_mode == "individual":
    compute_rows(adata_inj[adata_inj.obs["inj_type"] == "uninjured"],
                 "0", "uninjured", "uninjured")
    for inj in inj_types_to_plot:
        for day in days:
            sub = adata_inj[(adata_inj.obs["day_plot"] == day) &
                            (adata_inj.obs["inj_type"] == inj)]
            if len(sub) > 0:
                compute_rows(sub, day, inj, phase_map.get(day, day))

elif time_mode == "phase":
    compute_rows(adata_inj[adata_inj.obs["inj_type"] == "uninjured"],
                 "0", "uninjured", "uninjured")
    for inj in inj_types_to_plot:
        for phase in phase_order[1:]:
            phase_days = [d for d, p in phase_map.items() if p == phase]
            sub = adata_inj[(adata_inj.obs["day_plot"].isin(phase_days)) &
                            (adata_inj.obs["inj_type"] == inj)]
            if len(sub) > 0:
                compute_rows(sub, phase, inj, phase)

df = pd.DataFrame(rows)

# ---- Build x-axis order ----
if time_mode == "individual":
    groups = [("0", "uninjured")]
    for day in days:
        for inj in inj_types_to_plot:
            if len(df[(df["day"] == day) & (df["inj_type"] == inj)]) > 0:
                groups.append((day, inj))
    x_labels = ["uninj" if g[1] == "uninjured" else f"d{g[0]}\n{g[1]}"
                for g in groups]

elif time_mode == "phase":
    groups = [("0", "uninjured")]
    for phase in phase_order[1:]:
        for inj in inj_types_to_plot:
            if len(df[(df["day"] == phase) & (df["inj_type"] == inj)]) > 0:
                groups.append((phase, inj))
    x_labels = ["uninj" if g[1] == "uninjured" else f"{g[0]}\n{g[1]}"
                for g in groups]

# ---- Plot ----
metrics = ["raw", "norm_immune", "norm_total"]
titles = ["Raw counts", "Normalized over immune cells", "Normalized over total cells"]
ylabels = ["Number of cells", "Proportion of immune cells", "Proportion of total cells"]

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, metric, title, ylabel in zip(axes, metrics, titles, ylabels):
    for x_pos, (day, inj) in enumerate(groups):
        sub = df[(df["day"] == day) & (df["inj_type"] == inj)]
        bottom = 0
        for label in plot_labels:
            val = sub[sub["cell_type"] == label][metric].values
            val = val[0] if len(val) > 0 else 0
            ax.bar(x_pos, val, bottom=bottom, color=colors[label],
                   edgecolor="white", linewidth=0.3, width=0.8)
            bottom += val

    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(x_labels, fontsize=7, rotation=45, ha="right")
    ax.set_title(f"{' vs '.join(inj_types_to_plot)} ({time_mode})\n{title}")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Day post injury" if time_mode == "individual" else "Phase")

# Legend
from matplotlib.patches import Patch
handles = [Patch(color=colors[label], label=label) for label in plot_labels]
fig.legend(handles=handles, loc="center right",
           bbox_to_anchor=(1.13, 0.5), title="Cell type", fontsize=8)

plt.tight_layout()
plt.savefig(
    f"{proj_dir}immune_per_day_{'_vs_'.join(inj_types_to_plot)}_{cell_type_to_plot}_{time_mode}_{'grouped' if group_celltypes else 'individual'}.pdf",
    bbox_inches="tight", format="pdf"
)
plt.show()

In [ ]:
adata_immune = sc.read_h5ad("/path/to/your/file.h5ad")


### EdgeR on predicted expression for microglia (Redo the replicate according to the cell type)

In [ ]:
import os
import subprocess
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETERS — change only this block
# ══════════════════════════════════════════════════════════════════════════════
celltype_value       = "Microglia"          # must match celltype_groups keys
inj_types            = ["myelin", "liver"]  # [reference, test] — positive logFC = higher in test
phase_to_test        = "acute"           # "acute", "subacute", or "late"
SCALE_FACTOR         = 10000                # scales predicted expression to integer-like counts
use_day_covariate    = False                # include day as covariate
use_injury_covariate = False                # include injury type as additional covariate
MIN_CELLS_PER_REPLICATE = 10

# ── Volcano / significance parameters ────────────────────────────────────────
use_padj      = False   # True = filter on padj, False = filter on pvalue
pval_thresh   = 0.05    # threshold for pvalue or padj
lfc_thresh    = 0.58    # log2 fold change threshold (0.58 ≈ 1.5x, 1.0 = 2x)
top_n_labels  = 10      # number of top genes to label on volcano

OUT_DIR  = f"{proj_dir}DEG_tangram/"
RSCRIPT  = "/usr/local/bin/Rscript"

COVARIATE_R_THRESHOLD            = 0.75
MIN_REPS_PER_GROUP_FOR_COVARIATE = 3
# ══════════════════════════════════════════════════════════════════════════════

analysis_label = (f"tangram_{celltype_value}_"
                  f"{inj_types[0]}_vs_{inj_types[1]}_{phase_to_test}")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Column names ──────────────────────────────────────────────────────────────
injury_col = "inj_type"
time_col   = "day"
sample_col = "batch"
rep_col    = "replicate"

# ── Cell type groups ──────────────────────────────────────────────────────────
celltype_groups = {
    "Microglia":       ["MG_High_mito", "MG_Homeostatic", "MG_Ifn", "MG_Inflammatory",
                        "MG_Lipid_metabo", "MG_Prolif", "MG_Repair", "MG_migrating"],
    "Macrophages":     ["Macro_Inflammatory", "Macro_chemotaxis"],
    "Dendritic_cells": ["Dendritic_cells"],
    "Monocytes":       ["Monocytes"],
    "T_cells":         ["T_cells"],
}

# ── Phase definitions ─────────────────────────────────────────────────────────
phase_map = {
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

# ── Replicate map ─────────────────────────────────────────────────────────────
replicate_map = {
    # ── ACUTE ─────────────────────────────────────────────────────────────────
    # liver
    "liver_d1_1":  "rep_1",  "liver_d1_7":  "rep_2",  "liver_d1_6":  "rep_3",
    "liver_d1_5":  "rep_4",  "liver_d1_9":  "rep_4",  "liver_d1_8":  "rep_6",
    "liver_d1_4":  "rep_7",  "liver_d1_3":  "rep_8",  "liver_d1_2":  "rep_9",
    "liver_d2_4":  "rep_10", "liver_d2_5":  "rep_11", "liver_d2_2":  "rep_12",
    "liver_d2_3":  "rep_13", "liver_d2_1":  "rep_11", "liver_d2_6":  "rep_11",
    "liver_d2_7":  "rep_11", "liver_d3_1":  "rep_17", "liver_d3_6":  "rep_18",
    "liver_d3_7":  "rep_19", "liver_d3_4":  "rep_20", "liver_d3_8":  "rep_21",
    "liver_d3_5":  "rep_22", "liver_d3_2":  "rep_23", "liver_d3_3":  "rep_24",
    # myelin
    "myelin_d1_7": "rep_25", "myelin_d1_6": "rep_26", "myelin_d1_1": "rep_27",
    "myelin_d1_3": "rep_28", "myelin_d1_2": "rep_29", "myelin_d1_9": "rep_30",
    "myelin_d1_8": "rep_25", "myelin_d1_4": "rep_25", "myelin_d2_2": "rep_33",
    "myelin_d2_3": "rep_34", "myelin_d2_4": "rep_35", "myelin_d2_5": "rep_36",
    "myelin_d2_6": "rep_37", "myelin_d2_7": "rep_38", "myelin_d2_1": "rep_39",
    "myelin_d3_1": "rep_40", "myelin_d3_2": "rep_41", "myelin_d3_3": "rep_39",
    "myelin_d3_4": "rep_43",
    # ── SUBACUTE ──────────────────────────────────────────────────────────────
    # liver
    "liver_d4_2":  "rep_45", "liver_d4_3":  "rep_46", "liver_d4_8":  "rep_47",
    "liver_d4_9":  "rep_48", "liver_d4_5":  "rep_49", "liver_d4_6":  "rep_50",
    "liver_d4_7":  "rep_51", "liver_d4_10": "rep_51", "liver_d4_1":  "rep_53",
    "liver_d4_11": "rep_51", "liver_d5_6":  "rep_55", "liver_d5_7":  "rep_56",
    "liver_d5_2":  "rep_57", "liver_d5_3":  "rep_58", "liver_d5_8":  "rep_59",
    "liver_d5_4":  "rep_60", "liver_d5_5":  "rep_61", "liver_d5_10": "rep_62",
    "liver_d7_18": "rep_63", "liver_d7_14": "rep_64", "liver_d7_15": "rep_65",
    "liver_d7_19": "rep_66", "liver_d7_12": "rep_67", "liver_d7_13": "rep_68",
    "liver_d7_10": "rep_69", "liver_d7_11": "rep_70", "liver_d7_9":  "rep_71",
    "liver_d7_16": "rep_72", "liver_d7_17": "rep_73",
    # myelin
    "myelin_d4_4":  "rep_74", "myelin_d4_14": "rep_75", "myelin_d4_8":  "rep_76",
    "myelin_d4_9":  "rep_77", "myelin_d4_5":  "rep_78", "myelin_d4_2":  "rep_79",
    "myelin_d4_3":  "rep_80", "myelin_d4_1":  "rep_81", "myelin_d4_6":  "rep_82",
    "myelin_d4_7":  "rep_83", "myelin_d5_1":  "rep_84", "myelin_d5_6":  "rep_85",
    "myelin_d5_7":  "rep_86", "myelin_d5_5":  "rep_87", "myelin_d5_2":  "rep_88",
    "myelin_d5_3":  "rep_89", "myelin_d7_12": "rep_90", "myelin_d7_13": "rep_91",
    "myelin_d7_14": "rep_92", "myelin_d7_9":  "rep_93", "myelin_d7_8":  "rep_94",
    "myelin_d7_10": "rep_95", "myelin_d7_11": "rep_96",
    # ── LATE ──────────────────────────────────────────────────────────────────
    # liver
    "liver_d9_10":  "rep_98",  "liver_d9_11":  "rep_99",  "liver_d9_8":   "rep_100",
    "liver_d9_9":   "rep_101", "liver_d14_1":  "rep_102", "liver_d14_13": "rep_103",
    "liver_d14_12": "rep_104", "liver_d14_11": "rep_104", "liver_d14_9":  "rep_106",
    "liver_d14_10": "rep_106", "liver_d14_2":  "rep_108", "liver_d14_3":  "rep_102",
    # myelin
    "myelin_d9_1":   "rep_110", "myelin_d9_6":   "rep_111", "myelin_d9_7":   "rep_112",
    "myelin_d9_8":   "rep_113", "myelin_d9_4":   "rep_114", "myelin_d9_5":   "rep_115",
    "myelin_d9_9":   "rep_116", "myelin_d9_2":   "rep_117", "myelin_d14_10": "rep_118",
    "myelin_d14_7":  "rep_119", "myelin_d14_11": "rep_120", "myelin_d14_1":  "rep_121",
    "myelin_d14_2":  "rep_118", "myelin_d14_3":  "rep_118", "myelin_d14_12": "rep_124",
    "myelin_d14_8":  "rep_125", "myelin_d14_9":  "rep_126", "myelin_d14_13": "rep_127",
}

# ══════════════════════════════════════════════════════════════════════════════
# 1. Transfer metadata to ad_ge
# ══════════════════════════════════════════════════════════════════════════════
for col in [sample_col, "predicted_celltype"]:
    if col not in ad_ge.obs.columns:
        ad_ge.obs[col] = adata_immune.obs[col].reindex(ad_ge.obs_names).values

ad_ge.obs[injury_col] = ad_ge.obs[sample_col].str.split("_d").str[0]
ad_ge.obs[time_col]   = "d" + ad_ge.obs[sample_col].str.split("_d").str[1].str.split("_").str[0]
ad_ge.obs["phase"]    = ad_ge.obs[time_col].map(phase_map)

# ══════════════════════════════════════════════════════════════════════════════
# 2. Subset
# ══════════════════════════════════════════════════════════════════════════════
members       = celltype_groups[celltype_value]
celltype_mask = ad_ge.obs["predicted_celltype"].isin(members)
injury_mask   = ad_ge.obs[injury_col].isin(inj_types)
phase_mask    = ad_ge.obs["phase"] == phase_to_test

adata_deg = ad_ge[celltype_mask & injury_mask & phase_mask].copy()
print(f"Cells after subsetting: {adata_deg.n_obs}")

if adata_deg.n_obs == 0:
    raise ValueError(f"No cells found for {celltype_value}, {inj_types}, {phase_to_test}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Check expression values
# ══════════════════════════════════════════════════════════════════════════════
X_check = adata_deg.X
if hasattr(X_check, "toarray"):
    X_check = X_check.toarray()
else:
    X_check = np.asarray(X_check)

print(f"\nExpression value check:")
print(f"  min:     {X_check.min():.6f}")
print(f"  max:     {X_check.max():.6f}")
print(f"  mean:    {X_check.mean():.6f}")
print(f"  % zeros: {(X_check == 0).mean() * 100:.1f}%")

if X_check.max() > 100:
    print("⚠ WARNING: values look like raw counts — make sure you are using ad_ge")
else:
    print(f"✅ Values look like predicted expression — will scale by {SCALE_FACTOR}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Apply replicate map
# ══════════════════════════════════════════════════════════════════════════════
adata_deg.obs[rep_col] = adata_deg.obs[sample_col].map(replicate_map)

unmapped = adata_deg.obs.loc[adata_deg.obs[rep_col].isna(), sample_col].unique()
if len(unmapped) > 0:
    print(f"\n⚠ Unmapped batches (will be dropped): {sorted(unmapped)}")
    adata_deg = adata_deg[adata_deg.obs[rep_col].notna()].copy()
else:
    print(f"\n✅ All batches mapped successfully")

print(f"Unique replicates: {adata_deg.obs[rep_col].nunique()}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Scale predicted expression
# ══════════════════════════════════════════════════════════════════════════════
X = adata_deg.X
if hasattr(X, "toarray"):
    X = X.toarray()
else:
    X = np.asarray(X)

X_scaled = np.round(X * SCALE_FACTOR).astype(int)
adata_deg.obs["total_counts"] = X_scaled.sum(axis=1)

print(f"\nAfter scaling (x{SCALE_FACTOR}):")
print(f"  min={X_scaled.min()}, max={X_scaled.max()}, mean={X_scaled.mean():.2f}")

# ══════════════════════════════════════════════════════════════════════════════
# 6. Pseudobulk
# ══════════════════════════════════════════════════════════════════════════════
def make_pseudobulk(adata, X_scaled, rep_col):
    replicates  = adata.obs[rep_col].astype(str).values
    unique_reps = pd.Index(pd.unique(replicates))

    pb_rows = []
    for r in unique_reps:
        idx    = np.where(replicates == r)[0]
        summed = X_scaled[idx].sum(axis=0)
        pb_rows.append(summed)

    pb_counts = pd.DataFrame(
        pb_rows,
        index=unique_reps,
        columns=adata.var_names,
        dtype=int,
    )

    pb_meta = (
        adata.obs
        .groupby(rep_col, observed=True)
        [[injury_col, "phase", time_col]]
        .agg("first")
        .copy()
    )
    pb_meta["n_cells"]     = adata.obs.groupby(rep_col).size()
    pb_meta["mean_counts"] = adata.obs.groupby(rep_col)["total_counts"].mean().round(4)

    return pb_counts, pb_meta

pb_counts, pb_meta = make_pseudobulk(adata_deg, X_scaled, rep_col)

print(f"\npb_counts shape: {pb_counts.shape}")
print(f"\nCell counts per replicate:")
print(pb_meta[[injury_col, time_col, "n_cells"]]
      .sort_values([injury_col, time_col, "n_cells"])
      .to_string())

print(f"\nReplicates below {MIN_CELLS_PER_REPLICATE} cells:")
low = pb_meta[pb_meta["n_cells"] < MIN_CELLS_PER_REPLICATE]
print(low[[injury_col, time_col, "n_cells"]].to_string() if len(low) > 0 else "None ✅")

print(f"\nDay balance per condition:")
print(pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0))

print(f"\nSummary per condition:")
print(pb_meta.groupby(injury_col)["n_cells"]
      .agg(["count", "sum", "min", "max", "mean"])
      .rename(columns={"count": "n_reps", "sum": "total_cells", "mean": "mean_cells"})
      .round(1))

# ══════════════════════════════════════════════════════════════════════════════
# 7. QC plot
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=adata_deg.obs, x=injury_col, y="total_counts",
            palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
            order=inj_types, ax=axes[0], showfliers=False)
sns.stripplot(data=adata_deg.obs, x=injury_col, y="total_counts",
              palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
              order=inj_types, ax=axes[0], size=2, alpha=0.3, jitter=True)
axes[0].set_title("Scaled predicted expression per cell")
axes[0].set_xlabel("")

day_counts = pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0)
day_counts.T.plot(kind="bar", ax=axes[1],
                  color={inj_types[0]: "steelblue", inj_types[1]: "orange"})
axes[1].set_title("Replicates per day per condition")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("N replicates")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle(f"{celltype_value} — {phase_to_test} | {inj_types[0]} vs {inj_types[1]}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}QC_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 8. Covariate decision (mean_counts)
# ══════════════════════════════════════════════════════════════════════════════
def decide_covariate_use(meta_ph, inj_types):
    counts_by_group = meta_ph.groupby(injury_col).size().to_dict()
    n_0 = int(counts_by_group.get(inj_types[0], 0))
    n_1 = int(counts_by_group.get(inj_types[1], 0))

    injury_binary = (meta_ph[injury_col] == inj_types[0]).astype(int)

    if meta_ph["mean_counts"].nunique() <= 1:
        r, p = np.nan, np.nan
    else:
        r, p = stats.pointbiserialr(injury_binary, meta_ph["mean_counts"])

    enough_reps      = (n_0 >= MIN_REPS_PER_GROUP_FOR_COVARIATE) and \
                       (n_1 >= MIN_REPS_PER_GROUP_FOR_COVARIATE)
    low_collinearity = pd.notna(r) and (abs(r) <= COVARIATE_R_THRESHOLD)
    use_covariate    = bool(enough_reps and low_collinearity)

    print(f"\nCorrelation injury ~ mean_counts: r={r:.3f}, p={p:.3f}")
    print(f"n_{inj_types[0]}={n_0}, n_{inj_types[1]}={n_1}")
    if not enough_reps:
        print(f"  → DO NOT use mean_counts covariate: too few replicates")
    elif pd.isna(r):
        print(f"  → DO NOT use mean_counts covariate: no variation")
    elif abs(r) > COVARIATE_R_THRESHOLD:
        print(f"  → DO NOT use mean_counts covariate: collinearity too high (|r|={abs(r):.3f})")
    else:
        print(f"  → USE mean_counts covariate (|r|={abs(r):.3f})")
    print(f"  → Day covariate:    {'YES' if use_day_covariate else 'NO'}")
    print(f"  → Injury covariate: {'YES' if use_injury_covariate else 'NO'}")

    return use_covariate

use_mc_covariate = decide_covariate_use(pb_meta, inj_types)

# ── Build design string ───────────────────────────────────────────────────────
terms = []
if use_day_covariate:    terms.append("day")
if use_mc_covariate:     terms.append("mc_scaled")
if use_injury_covariate: terms.append("injury_type")
terms.append("injury")
design_str = "~ " + " + ".join(terms)
print(f"\nFinal design: {design_str}")

# ══════════════════════════════════════════════════════════════════════════════
# 9. edgeR R script
# ══════════════════════════════════════════════════════════════════════════════
R_SCRIPT_TEMPLATE = r"""
suppressPackageStartupMessages({{ library(edgeR) }})

counts_path          <- "{counts_path}"
meta_path            <- "{meta_path}"
out_path             <- "{out_path}"
use_mc_covariate     <- {use_mc_covariate}
use_day_covariate    <- {use_day_covariate}
use_injury_covariate <- {use_injury_covariate}
ref_level            <- "{ref_level}"
test_level           <- "{test_level}"

counts_df <- read.csv(counts_path, row.names = 1, check.names = FALSE)
counts    <- t(as.matrix(counts_df))
meta      <- read.csv(meta_path, row.names = 1, check.names = FALSE)
meta      <- meta[colnames(counts), , drop = FALSE]

injury      <- factor(meta$inj_type, levels = c(ref_level, test_level))
day         <- factor(meta$day)
injury_type <- factor(meta$inj_type)

terms <- c()
if (use_day_covariate)    terms <- c(terms, "day")
if (use_mc_covariate) {{
    mc_scaled <- meta$mean_counts_scaled
    terms <- c(terms, "mc_scaled")
}}
if (use_injury_covariate) terms <- c(terms, "injury_type")
terms <- c(terms, "injury")

formula_str <- paste("~", paste(terms, collapse = " + "))
cat("Formula:", formula_str, "\n")

design <- model.matrix(as.formula(formula_str))
rownames(design) <- colnames(counts)
cat("Design:", paste(colnames(design), collapse=", "), "\n")
cat("Design dimensions:", nrow(design), "x", ncol(design), "\n")

dge  <- DGEList(counts = counts, group = injury)
keep <- rowSums(dge$counts) >= 5
cat("Genes passing rowSums filter:", sum(keep), "\n")
dge  <- dge[keep, , keep.lib.sizes = FALSE]
dge  <- calcNormFactors(dge, method = "TMM")
dge  <- estimateDisp(dge, design, robust = TRUE)
fit  <- glmQLFit(dge, design, robust = TRUE)
qlf  <- glmQLFTest(fit, coef = ncol(design))

result <- topTags(qlf, n = Inf, sort.by = "PValue")$table
colnames(result)[colnames(result) == "logFC"]  <- "log2FoldChange"
colnames(result)[colnames(result) == "PValue"] <- "pvalue"
colnames(result)[colnames(result) == "FDR"]    <- "padj"
result$gene <- rownames(result)
result <- result[, c("gene", "log2FoldChange", "logCPM", "F", "pvalue", "padj")]
write.csv(result, out_path, row.names = FALSE)
cat("edgeR done:", nrow(result), "genes tested\n")
"""

# ══════════════════════════════════════════════════════════════════════════════
# 10. Run edgeR
# ══════════════════════════════════════════════════════════════════════════════
def run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
              use_day_covariate, use_injury_covariate,
              min_cells=MIN_CELLS_PER_REPLICATE):

    meta_ph   = pb_meta.copy()
    counts_ph = pb_counts.loc[meta_ph.index].copy()

    valid   = meta_ph["n_cells"] >= min_cells
    dropped = meta_ph[~valid].index.tolist()
    if dropped:
        print(f"⚠ Dropping {len(dropped)} replicate(s) with <{min_cells} cells: {dropped}")
    meta_ph   = meta_ph[valid]
    counts_ph = counts_ph.loc[meta_ph.index]

    conditions = meta_ph[injury_col].unique()
    if not set(inj_types).issubset(set(conditions)):
        raise ValueError(f"Lost a condition after filtering. Found: {conditions}\n"
                         f"Try reducing MIN_CELLS_PER_REPLICATE (currently {min_cells})")

    keep      = (counts_ph > 0).sum(axis=0) >= 2
    counts_ph = counts_ph.loc[:, keep]
    print(f"Replicates: {len(meta_ph)} | Genes after pre-filter: {keep.sum()}")
    print(f"Day balance:\n"
          f"{meta_ph.groupby([injury_col, time_col]).size().unstack(fill_value=0)}")

    meta_ph = meta_ph.copy()
    meta_ph["mean_counts_scaled"] = meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()

    with tempfile.TemporaryDirectory() as tmpdir:
        counts_path = os.path.join(tmpdir, "counts.csv")
        meta_path   = os.path.join(tmpdir, "meta.csv")
        out_path    = os.path.join(tmpdir, "result.csv")
        r_path      = os.path.join(tmpdir, "run_edgeR.R")

        counts_ph.to_csv(counts_path)
        meta_ph[[injury_col, time_col, "mean_counts", "mean_counts_scaled"]].to_csv(meta_path)

        r_script = R_SCRIPT_TEMPLATE.format(
            counts_path=counts_path,
            meta_path=meta_path,
            out_path=out_path,
            use_mc_covariate="TRUE" if use_mc_covariate else "FALSE",
            use_day_covariate="TRUE" if use_day_covariate else "FALSE",
            use_injury_covariate="TRUE" if use_injury_covariate else "FALSE",
            ref_level=inj_types[0],
            test_level=inj_types[1],
        )
        with open(r_path, "w") as f:
            f.write(r_script)

        proc = subprocess.run([RSCRIPT, "--vanilla", r_path],
                              capture_output=True, text=True)

        if proc.returncode != 0:
            print(f"R stdout:\n{proc.stdout}")
            print(f"R stderr:\n{proc.stderr}")
            raise RuntimeError("edgeR failed")

        print(proc.stdout.strip())
        res = pd.read_csv(out_path, index_col="gene")

    return res.sort_values("pvalue")

res = run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
                use_day_covariate, use_injury_covariate)
out_csv = f"{OUT_DIR}DEG_{analysis_label}.csv"
res.to_csv(out_csv)

# ══════════════════════════════════════════════════════════════════════════════
# 11. Diagnostic — p-value distribution
# ══════════════════════════════════════════════════════════════════════════════
pval_col = "padj" if use_padj else "pvalue"
n_sig = ((res[pval_col] < pval_thresh) & (res["log2FoldChange"].abs() > lfc_thresh)).sum()

print(f"\nResults saved → {out_csv}")
print(f"Significant genes ({pval_col}<{pval_thresh}, |lfc|>{lfc_thresh}): {n_sig}")

print(f"\nP-value distribution:")
print(res["pvalue"].describe().round(4))
print(f"\nPadj distribution:")
print(res["padj"].describe().round(4))
print(f"\nGenes with pvalue < 0.05: {(res['pvalue'] < 0.05).sum()}")
print(f"Genes with pvalue < 0.01: {(res['pvalue'] < 0.01).sum()}")
print(f"Genes with padj  < 0.05: {(res['padj']  < 0.05).sum()}")
print(f"Genes with padj  < 0.10: {(res['padj']  < 0.10).sum()}")
print(f"Genes with padj  < 0.20: {(res['padj']  < 0.20).sum()}")

print(f"\nTop 20 genes by pvalue:")
print(res.head(20)[["log2FoldChange", "pvalue", "padj"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(res["pvalue"].dropna(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("pvalue")
axes[0].set_ylabel("Count")
axes[0].set_title("P-value distribution\n(enriched near 0 = real signal)")
axes[1].hist(res["padj"].dropna(), bins=50, color="salmon", edgecolor="white")
axes[1].set_xlabel("padj")
axes[1].set_title("Padj distribution")
plt.suptitle(f"{celltype_value} — {phase_to_test} | {inj_types[0]} vs {inj_types[1]}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}pval_distribution_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 12. Volcano plot
# ══════════════════════════════════════════════════════════════════════════════
def volcano(df, analysis_label, inj_types, celltype_value, phase_to_test,
            design_str, use_padj=True, pval_thresh=0.05, lfc_thresh=0.58, top_n=10):

    pval_col   = "padj" if use_padj else "pvalue"
    pval_label = "padj" if use_padj else "p-value"

    df = df.dropna(subset=["log2FoldChange", pval_col]).copy()
    df["neglog10_pvalue"] = -np.log10(df["pvalue"].clip(lower=1e-300))

    sig_test = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] >  lfc_thresh)
    sig_ref  = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    other    = ~(sig_test | sig_ref)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.scatter(df.loc[other,    "log2FoldChange"], df.loc[other,    "neglog10_pvalue"],
               s=10, alpha=0.35, color="grey",      label="Not significant")
    ax.scatter(df.loc[sig_ref,  "log2FoldChange"], df.loc[sig_ref,  "neglog10_pvalue"],
               s=20, alpha=0.85, color="steelblue",
               label=f"Higher in {inj_types[0]} (n={sig_ref.sum()})")
    ax.scatter(df.loc[sig_test, "log2FoldChange"], df.loc[sig_test, "neglog10_pvalue"],
               s=20, alpha=0.85, color="orange",
               label=f"Higher in {inj_types[1]} (n={sig_test.sum()})")

    ax.axvline(-lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)
    ax.axvline( lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)

    if use_padj:
        sig_boundary = df[df[pval_col] < pval_thresh]["pvalue"]
        if len(sig_boundary) > 0:
            ax.axhline(-np.log10(sig_boundary.max()), linestyle="--",
                       linewidth=0.9, color="black", alpha=0.5)
    else:
        ax.axhline(-np.log10(pval_thresh), linestyle="--",
                   linewidth=0.9, color="black", alpha=0.5)

    for gene, row in df[sig_test].sort_values(pval_col).head(top_n).iterrows():
        ax.text(row["log2FoldChange"], row["neglog10_pvalue"], gene,
                fontsize=7.5, ha="left", va="bottom")
    for gene, row in df[sig_ref].sort_values(pval_col).head(top_n).iterrows():
        ax.text(row["log2FoldChange"], row["neglog10_pvalue"], gene,
                fontsize=7.5, ha="right", va="bottom")

    ax.set_xlabel(f"log₂ Fold Change ({inj_types[0]} vs {inj_types[1]})", fontsize=12)
    ax.set_ylabel("−log₁₀ p-value", fontsize=12)
    ax.set_title(
        f"{celltype_value} — {phase_to_test}\n"
        f"{inj_types[0]} vs {inj_types[1]} | "
        f"{pval_label}<{pval_thresh}, |lfc|>{lfc_thresh} | design: {design_str}",
        fontsize=10
    )
    ax.legend(frameon=False)
    plt.tight_layout()
    save = f"{OUT_DIR}volcano_{analysis_label}.pdf"
    plt.savefig(save, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Volcano saved → {save}")

volcano(res, analysis_label, inj_types, celltype_value, phase_to_test,
        design_str, use_padj=use_padj, pval_thresh=pval_thresh,
        lfc_thresh=lfc_thresh, top_n=top_n_labels)

### EdgeR on raw data from the spatial dataset - Microglia

In [ ]:
adata = sc.read_h5ad("/path/to/your/file.h5ad")

In [ ]:
adata_immune = sc.read_h5ad(
    "/path/to/output/")

In [ ]:
adata_immune.obs_names = adata_immune.obs_names.str.rsplit("-", n=1).str[0]

adata_immune.layers["counts"] = adata[adata_immune.obs_names, adata_immune.var_names].X.copy()

adata_immune.layers["counts"] = adata[
    adata_immune.obs_names, adata_immune.var_names
].X.copy()

adata_immune.layers["counts"].shape
adata_immune.shape

adata_immune.layers["counts"] = adata_immune.X.copy()

from scipy import sparse
import numpy as np

A = adata[adata_immune.obs_names, adata_immune.var_names].X
adata_immune.layers["counts"] = A.copy()
B = adata_immune.layers["counts"]

A_dense = A.toarray() if sparse.issparse(A) else np.asarray(A)
B_dense = B.toarray() if sparse.issparse(B) else np.asarray(B)

print("identical:", np.array_equal(A_dense, B_dense))

adata_immune.layers["counts"].sum()

In [ ]:
mean_rna = adata_immune.obs["n_transcripts"].mean()
sd_rna = adata_immune.obs["n_transcripts"].std()

print(mean_rna)
print(sd_rna)

In [ ]:
mean_genes = adata_immune.obs["n_genes"].mean()
sd_genes = adata_immune.obs["n_genes"].std()

print(mean_genes)
print(sd_genes)

In [ ]:
# ── Check what's in adata_immune ─────────────────────────────────────────────
print(f"adata_immune shape: {adata_immune.shape}")
print(f"\nLayers available: {list(adata_immune.layers.keys())}")
print(f"\nX matrix stats:")

X = adata_immune.X
if hasattr(X, "toarray"):
    X = X.toarray()
else:
    X = np.asarray(X)

print(f"  dtype:   {X.dtype}")
print(f"  min:     {X.min():.4f}")
print(f"  max:     {X.max():.4f}")
print(f"  mean:    {X.mean():.4f}")
print(f"  % zeros: {(X == 0).mean() * 100:.1f}%")
print(f"  % integers: {(X == np.round(X)).mean() * 100:.1f}%")

# Interpret
if X.max() > 10 and (X == np.round(X)).mean() > 0.99:
    print("\n✅ X looks like raw integer counts — good to use directly")
elif X.max() <= 10 and X.min() >= 0:
    print("\n⚠ X looks like normalized data (log1p or similar)")
    print("  → Check layers for raw counts")
else:
    print("\n⚠ Unclear — check layers")

# Check each layer
if adata_immune.layers:
    for layer_name in adata_immune.layers.keys():
        L = adata_immune.layers[layer_name]
        if hasattr(L, "toarray"):
            L = L.toarray()
        else:
            L = np.asarray(L)
        pct_int = (L == np.round(L)).mean() * 100
        print(f"\nLayer '{layer_name}':")
        print(f"  min={L.min():.4f}, max={L.max():.4f}, "
              f"mean={L.mean():.4f}, % integers={pct_int:.1f}%")

In [ ]:
import os
import subprocess
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETERS — change only this block
# ══════════════════════════════════════════════════════════════════════════════
celltype_value       = "Monocytes"
inj_types            = ["myelin", "liver"]
phase_to_test        = "late"
use_day_covariate    = True
use_injury_covariate = False
MIN_CELLS_PER_REPLICATE = 10

use_padj      = False
pval_thresh   = 0.05
lfc_thresh    = 0.58
top_n_labels  = 10

proj_dir = "/path/to/project/"

OUT_DIR  = f"{proj_dir}DEG_spatial_raw/"
RSCRIPT  = "/usr/local/bin/Rscript"

COVARIATE_R_THRESHOLD            = 0.75
MIN_REPS_PER_GROUP_FOR_COVARIATE = 3
# ══════════════════════════════════════════════════════════════════════════════

analysis_label = (f"spatial_raw_{celltype_value}_"
                  f"{inj_types[0]}_vs_{inj_types[1]}_{phase_to_test}")
os.makedirs(OUT_DIR, exist_ok=True)

injury_col = "inj_type"
time_col   = "day"
sample_col = "manual_anno"
rep_col    = "replicate"

celltype_groups = {
    "Micriglia":       ['MG_Homeostatic','MG_Activated','MG_Reactive','MG_Serpina','MG_Chronic'],
    "Macrophages":     ['Macro_early','Macro_middle','Macro_late','Macro_chronic'],
    "Dendritic_cells": ["Dendritic_cells"],
    "Monocytes":       ["Monocytes"],
}

phase_map = {
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

replicate_map = {
    # ── ACUTE ─────────────────────────────────────────────────────────────────
    "liver_d1_1":  "rep_1",  "liver_d1_7":  "rep_2",  "liver_d1_6":  "rep_3",
    "liver_d1_5":  "rep_4",  "liver_d1_9":  "rep_4",  "liver_d1_8":  "rep_6",
    "liver_d1_4":  "rep_7",  "liver_d1_3":  "rep_8",  "liver_d1_2":  "rep_9",
    "liver_d2_4":  "rep_10", "liver_d2_5":  "rep_11", "liver_d2_2":  "rep_12",
    "liver_d2_3":  "rep_13", "liver_d2_1":  "rep_11", "liver_d2_6":  "rep_11",
    "liver_d2_7":  "rep_11", "liver_d3_1":  "rep_17", "liver_d3_6":  "rep_18",
    "liver_d3_7":  "rep_19", "liver_d3_4":  "rep_20", "liver_d3_8":  "rep_21",
    "liver_d3_5":  "rep_22", "liver_d3_2":  "rep_23", "liver_d3_3":  "rep_24",
    "myelin_d1_7": "rep_25", "myelin_d1_6": "rep_26", "myelin_d1_1": "rep_27",
    "myelin_d1_3": "rep_28", "myelin_d1_2": "rep_29", "myelin_d1_9": "rep_30",
    "myelin_d1_8": "rep_25", "myelin_d1_4": "rep_25", "myelin_d2_2": "rep_33",
    "myelin_d2_3": "rep_34", "myelin_d2_4": "rep_35", "myelin_d2_5": "rep_36",
    "myelin_d2_6": "rep_37", "myelin_d2_7": "rep_38", "myelin_d2_1": "rep_39",
    "myelin_d3_1": "rep_40", "myelin_d3_2": "rep_41", "myelin_d3_3": "rep_39",
    "myelin_d3_4": "rep_43",
    # ── SUBACUTE ──────────────────────────────────────────────────────────────
    "liver_d4_2":  "rep_45", "liver_d4_3":  "rep_46", "liver_d4_8":  "rep_47",
    "liver_d4_9":  "rep_48", "liver_d4_5":  "rep_49", "liver_d4_6":  "rep_50",
    "liver_d4_7":  "rep_51", "liver_d4_10": "rep_51", "liver_d4_1":  "rep_53",
    "liver_d4_11": "rep_51", "liver_d5_6":  "rep_55", "liver_d5_7":  "rep_56",
    "liver_d5_2":  "rep_57", "liver_d5_3":  "rep_58", "liver_d5_8":  "rep_59",
    "liver_d5_4":  "rep_60", "liver_d5_5":  "rep_61", "liver_d5_10": "rep_62",
    "liver_d7_18": "rep_63", "liver_d7_14": "rep_64", "liver_d7_15": "rep_65",
    "liver_d7_19": "rep_66", "liver_d7_12": "rep_67", "liver_d7_13": "rep_68",
    "liver_d7_10": "rep_69", "liver_d7_11": "rep_70", "liver_d7_9":  "rep_71",
    "liver_d7_16": "rep_72", "liver_d7_17": "rep_73",
    "myelin_d4_4":  "rep_74", "myelin_d4_14": "rep_75", "myelin_d4_8":  "rep_76",
    "myelin_d4_9":  "rep_77", "myelin_d4_5":  "rep_78", "myelin_d4_2":  "rep_79",
    "myelin_d4_3":  "rep_80", "myelin_d4_1":  "rep_81", "myelin_d4_6":  "rep_82",
    "myelin_d4_7":  "rep_83", "myelin_d5_1":  "rep_84", "myelin_d5_6":  "rep_85",
    "myelin_d5_7":  "rep_86", "myelin_d5_5":  "rep_87", "myelin_d5_2":  "rep_88",
    "myelin_d5_3":  "rep_89", "myelin_d7_12": "rep_90", "myelin_d7_13": "rep_91",
    "myelin_d7_14": "rep_92", "myelin_d7_9":  "rep_93", "myelin_d7_8":  "rep_94",
    "myelin_d7_10": "rep_95", "myelin_d7_11": "rep_96",
    # ── LATE ──────────────────────────────────────────────────────────────────
    "liver_d9_10":  "rep_98",  "liver_d9_11":  "rep_99",  "liver_d9_8":   "rep_100",
    "liver_d9_9":   "rep_101", "liver_d14_1":  "rep_102", "liver_d14_13": "rep_103",
    "liver_d14_12": "rep_104", "liver_d14_11": "rep_104", "liver_d14_9":  "rep_106",
    "liver_d14_10": "rep_106", "liver_d14_2":  "rep_108", "liver_d14_3":  "rep_102",
    "myelin_d9_1":   "rep_110", "myelin_d9_6":   "rep_111", "myelin_d9_7":   "rep_112",
    "myelin_d9_8":   "rep_113", "myelin_d9_4":   "rep_114", "myelin_d9_5":   "rep_115",
    "myelin_d9_9":   "rep_116", "myelin_d9_2":   "rep_117", "myelin_d14_10": "rep_118",
    "myelin_d14_7":  "rep_119", "myelin_d14_11": "rep_120", "myelin_d14_1":  "rep_121",
    "myelin_d14_2":  "rep_118", "myelin_d14_3":  "rep_118", "myelin_d14_12": "rep_124",
    "myelin_d14_8":  "rep_125", "myelin_d14_9":  "rep_126", "myelin_d14_13": "rep_127",
}

# ══════════════════════════════════════════════════════════════════════════════
# 1. Prepare metadata
# ══════════════════════════════════════════════════════════════════════════════
adata_immune.obs[injury_col] = adata_immune.obs[sample_col].str.split("_d").str[0]
adata_immune.obs[time_col]   = "d" + adata_immune.obs[sample_col].str.split("_d").str[1].str.split("_").str[0]
adata_immune.obs["phase"]    = adata_immune.obs[time_col].map(phase_map)

# ══════════════════════════════════════════════════════════════════════════════
# 2. Subset
# ══════════════════════════════════════════════════════════════════════════════
members       = celltype_groups[celltype_value]
celltype_mask = adata_immune.obs["immune_subset"].isin(members)
injury_mask   = adata_immune.obs[injury_col].isin(inj_types)
phase_mask    = adata_immune.obs["phase"] == phase_to_test

adata_deg = adata_immune[celltype_mask & injury_mask & phase_mask].copy()
print(f"Cells after subsetting: {adata_deg.n_obs}")

if adata_deg.n_obs == 0:
    raise ValueError(f"No cells found for {celltype_value}, {inj_types}, {phase_to_test}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Extract raw counts from layers["counts"] ← KEY DIFFERENCE
# ══════════════════════════════════════════════════════════════════════════════
X = adata_deg.layers["counts"]  # ← raw integer counts
if hasattr(X, "toarray"):
    X = X.toarray()
else:
    X = np.asarray(X)

X_scaled = np.round(X).astype(int)  # already integers, just ensure dtype
adata_deg.obs["total_counts"] = X_scaled.sum(axis=1)

print(f"\nRaw counts check (layers['counts']):")
print(f"  min:        {X_scaled.min()}")
print(f"  max:        {X_scaled.max()}")
print(f"  mean:       {X_scaled.mean():.4f}")
print(f"  % zeros:    {(X_scaled == 0).mean() * 100:.1f}%")
print(f"  % integers: {(X_scaled == np.round(X_scaled)).mean() * 100:.1f}%")
print("✅ Using raw counts from layers['counts']")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Apply replicate map
# ══════════════════════════════════════════════════════════════════════════════
adata_deg.obs[rep_col] = adata_deg.obs[sample_col].map(replicate_map)

unmapped = adata_deg.obs.loc[adata_deg.obs[rep_col].isna(), sample_col].unique()
if len(unmapped) > 0:
    print(f"\n⚠ Unmapped batches (will be dropped): {sorted(unmapped)}")
    keep_mask = adata_deg.obs[rep_col].notna().values
    adata_deg = adata_deg[keep_mask].copy()
    X_scaled  = X_scaled[keep_mask]
else:
    print(f"\n✅ All batches mapped successfully")

print(f"Unique replicates: {adata_deg.obs[rep_col].nunique()}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Pseudobulk
# ══════════════════════════════════════════════════════════════════════════════
def make_pseudobulk(adata, X_scaled, rep_col):
    replicates  = adata.obs[rep_col].astype(str).values
    unique_reps = pd.Index(pd.unique(replicates))

    pb_rows = []
    for r in unique_reps:
        idx    = np.where(replicates == r)[0]
        summed = X_scaled[idx].sum(axis=0)
        pb_rows.append(summed)

    pb_counts = pd.DataFrame(
        pb_rows,
        index=unique_reps,
        columns=adata.var_names,
        dtype=int,
    )

    pb_meta = (
        adata.obs
        .groupby(rep_col, observed=True)
        [[injury_col, "phase", time_col]]
        .agg("first")
        .copy()
    )
    pb_meta["n_cells"]     = adata.obs.groupby(rep_col).size()
    pb_meta["mean_counts"] = adata.obs.groupby(rep_col)["total_counts"].mean().round(4)

    return pb_counts, pb_meta

pb_counts, pb_meta = make_pseudobulk(adata_deg, X_scaled, rep_col)

print(f"\npb_counts shape: {pb_counts.shape}")
print(f"\nCell counts per replicate:")
print(pb_meta[[injury_col, time_col, "n_cells"]]
      .sort_values([injury_col, time_col, "n_cells"])
      .to_string())

print(f"\nReplicates below {MIN_CELLS_PER_REPLICATE} cells:")
low = pb_meta[pb_meta["n_cells"] < MIN_CELLS_PER_REPLICATE]
print(low[[injury_col, time_col, "n_cells"]].to_string() if len(low) > 0 else "None ✅")

print(f"\nDay balance per condition:")
print(pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0))

print(f"\nSummary per condition:")
print(pb_meta.groupby(injury_col)["n_cells"]
      .agg(["count", "sum", "min", "max", "mean"])
      .rename(columns={"count": "n_reps", "sum": "total_cells", "mean": "mean_cells"})
      .round(1))

# ══════════════════════════════════════════════════════════════════════════════
# 6. QC plot
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=adata_deg.obs, x=injury_col, y="total_counts",
            palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
            order=inj_types, ax=axes[0], showfliers=False)
sns.stripplot(data=adata_deg.obs, x=injury_col, y="total_counts",
              palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
              order=inj_types, ax=axes[0], size=2, alpha=0.3, jitter=True)
axes[0].set_title("Raw counts per cell (layers['counts'])")
axes[0].set_xlabel("")

day_counts = pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0)
day_counts.T.plot(kind="bar", ax=axes[1],
                  color={inj_types[0]: "steelblue", inj_types[1]: "orange"})
axes[1].set_title("Replicates per day per condition")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("N replicates")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}QC_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 7. Covariate decision
# ══════════════════════════════════════════════════════════════════════════════
def decide_covariate_use(meta_ph, inj_types):
    counts_by_group = meta_ph.groupby(injury_col).size().to_dict()
    n_0 = int(counts_by_group.get(inj_types[0], 0))
    n_1 = int(counts_by_group.get(inj_types[1], 0))

    injury_binary = (meta_ph[injury_col] == inj_types[0]).astype(int)

    if meta_ph["mean_counts"].nunique() <= 1:
        r, p = np.nan, np.nan
    else:
        r, p = stats.pointbiserialr(injury_binary, meta_ph["mean_counts"])

    enough_reps      = (n_0 >= MIN_REPS_PER_GROUP_FOR_COVARIATE) and \
                       (n_1 >= MIN_REPS_PER_GROUP_FOR_COVARIATE)
    low_collinearity = pd.notna(r) and (abs(r) <= COVARIATE_R_THRESHOLD)
    use_covariate    = bool(enough_reps and low_collinearity)

    print(f"\nCorrelation injury ~ mean_counts: r={r:.3f}, p={p:.3f}")
    print(f"n_{inj_types[0]}={n_0}, n_{inj_types[1]}={n_1}")
    if not enough_reps:
        print(f"  → DO NOT use mean_counts covariate: too few replicates")
    elif pd.isna(r):
        print(f"  → DO NOT use mean_counts covariate: no variation")
    elif abs(r) > COVARIATE_R_THRESHOLD:
        print(f"  → DO NOT use mean_counts covariate: collinearity too high (|r|={abs(r):.3f})")
    else:
        print(f"  → USE mean_counts covariate (|r|={abs(r):.3f})")
    print(f"  → Day covariate:    {'YES' if use_day_covariate else 'NO'}")
    print(f"  → Injury covariate: {'YES' if use_injury_covariate else 'NO'}")

    return use_covariate

use_mc_covariate = decide_covariate_use(pb_meta, inj_types)

terms = []
if use_day_covariate:    terms.append("day")
if use_mc_covariate:     terms.append("mc_scaled")
if use_injury_covariate: terms.append("injury_type")
terms.append("injury")
design_str = "~ " + " + ".join(terms)
print(f"\nFinal design: {design_str}")

# ══════════════════════════════════════════════════════════════════════════════
# 8. edgeR R script
# ══════════════════════════════════════════════════════════════════════════════
R_SCRIPT_TEMPLATE = r"""
suppressPackageStartupMessages({{ library(edgeR) }})

counts_path          <- "{counts_path}"
meta_path            <- "{meta_path}"
out_path             <- "{out_path}"
use_mc_covariate     <- {use_mc_covariate}
use_day_covariate    <- {use_day_covariate}
use_injury_covariate <- {use_injury_covariate}
ref_level            <- "{ref_level}"
test_level           <- "{test_level}"

counts_df <- read.csv(counts_path, row.names = 1, check.names = FALSE)
counts    <- t(as.matrix(counts_df))
meta      <- read.csv(meta_path, row.names = 1, check.names = FALSE)
meta      <- meta[colnames(counts), , drop = FALSE]

injury      <- factor(meta$inj_type, levels = c(ref_level, test_level))
day         <- factor(meta$day)
injury_type <- factor(meta$inj_type)

terms <- c()
if (use_day_covariate)    terms <- c(terms, "day")
if (use_mc_covariate) {{
    mc_scaled <- meta$mean_counts_scaled
    terms <- c(terms, "mc_scaled")
}}
if (use_injury_covariate) terms <- c(terms, "injury_type")
terms <- c(terms, "injury")

formula_str <- paste("~", paste(terms, collapse = " + "))
cat("Formula:", formula_str, "\n")

design <- model.matrix(as.formula(formula_str))
rownames(design) <- colnames(counts)
cat("Design:", paste(colnames(design), collapse=", "), "\n")
cat("Design dimensions:", nrow(design), "x", ncol(design), "\n")

dge  <- DGEList(counts = counts, group = injury)
keep <- rowSums(dge$counts) >= 5
cat("Genes passing rowSums filter:", sum(keep), "\n")
dge  <- dge[keep, , keep.lib.sizes = FALSE]
dge  <- calcNormFactors(dge, method = "TMM")
dge  <- estimateDisp(dge, design, robust = TRUE)
fit  <- glmQLFit(dge, design, robust = TRUE)
qlf  <- glmQLFTest(fit, coef = ncol(design))

result <- topTags(qlf, n = Inf, sort.by = "PValue")$table
colnames(result)[colnames(result) == "logFC"]  <- "log2FoldChange"
colnames(result)[colnames(result) == "PValue"] <- "pvalue"
colnames(result)[colnames(result) == "FDR"]    <- "padj"
result$gene <- rownames(result)
result <- result[, c("gene", "log2FoldChange", "logCPM", "F", "pvalue", "padj")]
write.csv(result, out_path, row.names = FALSE)
cat("edgeR done:", nrow(result), "genes tested\n")
"""

# ══════════════════════════════════════════════════════════════════════════════
# 9. Run edgeR
# ══════════════════════════════════════════════════════════════════════════════
def run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
              use_day_covariate, use_injury_covariate,
              min_cells=MIN_CELLS_PER_REPLICATE):

    meta_ph   = pb_meta.copy()
    counts_ph = pb_counts.loc[meta_ph.index].copy()

    valid   = meta_ph["n_cells"] >= min_cells
    dropped = meta_ph[~valid].index.tolist()
    if dropped:
        print(f"⚠ Dropping {len(dropped)} replicate(s) with <{min_cells} cells: {dropped}")
    meta_ph   = meta_ph[valid]
    counts_ph = counts_ph.loc[meta_ph.index]

    conditions = meta_ph[injury_col].unique()
    if not set(inj_types).issubset(set(conditions)):
        raise ValueError(f"Lost a condition after filtering. Found: {conditions}\n"
                         f"Try reducing MIN_CELLS_PER_REPLICATE (currently {min_cells})")

    keep      = (counts_ph > 0).sum(axis=0) >= 2
    counts_ph = counts_ph.loc[:, keep]
    print(f"Replicates: {len(meta_ph)} | Genes after pre-filter: {keep.sum()}")
    print(f"Day balance:\n"
          f"{meta_ph.groupby([injury_col, time_col]).size().unstack(fill_value=0)}")

    meta_ph = meta_ph.copy()
    meta_ph["mean_counts_scaled"] = meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()

    with tempfile.TemporaryDirectory() as tmpdir:
        counts_path = os.path.join(tmpdir, "counts.csv")
        meta_path   = os.path.join(tmpdir, "meta.csv")
        out_path    = os.path.join(tmpdir, "result.csv")
        r_path      = os.path.join(tmpdir, "run_edgeR.R")

        counts_ph.to_csv(counts_path)
        meta_ph[[injury_col, time_col, "mean_counts", "mean_counts_scaled"]].to_csv(meta_path)

        r_script = R_SCRIPT_TEMPLATE.format(
            counts_path=counts_path,
            meta_path=meta_path,
            out_path=out_path,
            use_mc_covariate="TRUE" if use_mc_covariate else "FALSE",
            use_day_covariate="TRUE" if use_day_covariate else "FALSE",
            use_injury_covariate="TRUE" if use_injury_covariate else "FALSE",
            ref_level=inj_types[0],
            test_level=inj_types[1],
        )
        with open(r_path, "w") as f:
            f.write(r_script)

        proc = subprocess.run([RSCRIPT, "--vanilla", r_path],
                              capture_output=True, text=True)

        if proc.returncode != 0:
            print(f"R stdout:\n{proc.stdout}")
            print(f"R stderr:\n{proc.stderr}")
            raise RuntimeError("edgeR failed")

        print(proc.stdout.strip())
        res = pd.read_csv(out_path, index_col="gene")

    return res.sort_values("pvalue")

res = run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
                use_day_covariate, use_injury_covariate)
out_csv = f"{OUT_DIR}DEG_{analysis_label}.csv"
res.to_csv(out_csv)

# ══════════════════════════════════════════════════════════════════════════════
# 10. Diagnostic
# ══════════════════════════════════════════════════════════════════════════════
pval_col = "padj" if use_padj else "pvalue"
n_sig = ((res[pval_col] < pval_thresh) & (res["log2FoldChange"].abs() > lfc_thresh)).sum()

print(f"\nResults saved → {out_csv}")
print(f"Significant genes ({pval_col}<{pval_thresh}, |lfc|>{lfc_thresh}): {n_sig}")
print(f"\nGenes with pvalue < 0.05: {(res['pvalue'] < 0.05).sum()}")
print(f"Genes with pvalue < 0.01: {(res['pvalue'] < 0.01).sum()}")
print(f"Genes with padj  < 0.05: {(res['padj']  < 0.05).sum()}")
print(f"Genes with padj  < 0.10: {(res['padj']  < 0.10).sum()}")
print(f"\nTop 20 genes by pvalue:")
print(res.head(20)[["log2FoldChange", "pvalue", "padj"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(res["pvalue"].dropna(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("pvalue")
axes[0].set_ylabel("Count")
axes[0].set_title("P-value distribution\n(enriched near 0 = real signal)")
axes[1].hist(res["padj"].dropna(), bins=50, color="salmon", edgecolor="white")
axes[1].set_xlabel("padj")
axes[1].set_title("Padj distribution")
plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}pval_distribution_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 11. Volcano
# ══════════════════════════════════════════════════════════════════════════════
def volcano(df, analysis_label, inj_types, celltype_value, phase_to_test,
            design_str, use_padj=True, pval_thresh=0.05, lfc_thresh=0.58, top_n=10):

    pval_col   = "padj" if use_padj else "pvalue"
    pval_label = "padj" if use_padj else "p-value"

    df = df.dropna(subset=["log2FoldChange", pval_col]).copy()
    df["neglog10_pvalue"] = -np.log10(df["pvalue"].clip(lower=1e-300))

    sig_test = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] >  lfc_thresh)
    sig_ref  = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    other    = ~(sig_test | sig_ref)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.scatter(df.loc[other,    "log2FoldChange"], df.loc[other,    "neglog10_pvalue"],
               s=10, alpha=0.35, color="grey",      label="Not significant")
    ax.scatter(df.loc[sig_ref,  "log2FoldChange"], df.loc[sig_ref,  "neglog10_pvalue"],
               s=20, alpha=0.85, color="steelblue",
               label=f"Higher in {inj_types[0]} (n={sig_ref.sum()})")
    ax.scatter(df.loc[sig_test, "log2FoldChange"], df.loc[sig_test, "neglog10_pvalue"],
               s=20, alpha=0.85, color="orange",
               label=f"Higher in {inj_types[1]} (n={sig_test.sum()})")

    ax.axvline(-lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)
    ax.axvline( lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)

    if use_padj:
        sig_boundary = df[df[pval_col] < pval_thresh]["pvalue"]
        if len(sig_boundary) > 0:
            ax.axhline(-np.log10(sig_boundary.max()), linestyle="--",
                       linewidth=0.9, color="black", alpha=0.5)
    else:
        ax.axhline(-np.log10(pval_thresh), linestyle="--",
                   linewidth=0.9, color="black", alpha=0.5)

    texts = []
    for gene, row in df[sig_test].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))
    for gene, row in df[sig_ref].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))

    ax.set_xlabel(f"log₂ Fold Change ({inj_types[0]} vs {inj_types[1]})", fontsize=12)
    ax.set_ylabel("−log₁₀ p-value", fontsize=12)
    ax.set_title(
        f"{celltype_value} — {phase_to_test} (raw spatial counts)\n"
        f"{inj_types[0]} vs {inj_types[1]} | "
        f"{pval_label}<{pval_thresh}, |lfc|>{lfc_thresh} | design: {design_str}",
        fontsize=10
    )
    ax.legend(frameon=False)
    plt.tight_layout()
    save = f"{OUT_DIR}volcano_{analysis_label}.pdf"
    plt.savefig(save, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Volcano saved → {save}")

volcano(res, analysis_label, inj_types, celltype_value, phase_to_test,
        design_str, use_padj=use_padj, pval_thresh=pval_thresh,
        lfc_thresh=lfc_thresh, top_n=top_n_labels)

### EdgeR on raw data from the spatial dataset - Macrophage/Monocytes (not possible on late, no cell enough)

In [ ]:
import os
import subprocess
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETERS — change only this block
# ══════════════════════════════════════════════════════════════════════════════
celltype_value       = "Macrophages"
inj_types            = ["myelin", "liver"]
phase_to_test        = "subacute"
use_day_covariate    = True
use_injury_covariate = False
MIN_CELLS_PER_REPLICATE = 10

use_padj      = False
pval_thresh   = 0.05
lfc_thresh    = 0.58
top_n_labels  = 10

OUT_DIR  = f"{proj_dir}DEG_spatial_raw_Macor_Mono/"
RSCRIPT  = "/usr/local/bin/Rscript"

COVARIATE_R_THRESHOLD            = 0.75
MIN_REPS_PER_GROUP_FOR_COVARIATE = 3
# ══════════════════════════════════════════════════════════════════════════════

analysis_label = (f"spatial_raw_{celltype_value}_"
                  f"{inj_types[0]}_vs_{inj_types[1]}_{phase_to_test}")
os.makedirs(OUT_DIR, exist_ok=True)

injury_col = "inj_type"
time_col   = "day"
sample_col = "batch"
rep_col    = "replicate"

celltype_groups = {
    "Microglia":       ["MG_High_mito", "MG_Homeostatic", "MG_Ifn", "MG_Inflammatory",
                        "MG_Lipid_metabo", "MG_Prolif", "MG_Repair", "MG_migrating"],
    "Macrophages":     ["Macro_Inflammatory", "Macro_chemotaxis", "Monocytes"],
    "Dendritic_cells": ["Dendritic_cells"],
    "T_cells":         ["T_cells"],
}

phase_map = {
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

replicate_map_Macro = {
    # ── ACUTE ─────────────────────────────────────────────────────────────────
    # liver
    "liver_d1_1":  "rep_1",   "liver_d1_7":  "rep_2",   "liver_d1_6":  "rep_3",
    "liver_d1_5":  "rep_4",   "liver_d1_9":  "rep_4",   "liver_d1_8":  "rep_6",
    "liver_d1_4":  "rep_7",   "liver_d1_3":  "rep_8",   "liver_d1_2":  "rep_9",
    "liver_d2_4":  "rep_10",  "liver_d2_5":  "rep_11",  "liver_d2_2":  "rep_12",
    "liver_d2_3":  "rep_13",  "liver_d2_1":  "rep_11",  "liver_d2_6":  "rep_11",
    "liver_d2_7":  "rep_11",  "liver_d3_1":  "rep_17",  "liver_d3_6":  "rep_18",
    "liver_d3_7":  "rep_19",  "liver_d3_4":  "rep_19",  "liver_d3_8":  "rep_21",
    "liver_d3_5":  "rep_21",  "liver_d3_2":  "rep_23",  "liver_d3_3":  "rep_24",
    # myelin
    "myelin_d1_7": "rep_25",  "myelin_d1_6": "rep_25",  "myelin_d1_1": "rep_27",
    "myelin_d1_3": "rep_28",  "myelin_d1_2": "rep_25",  "myelin_d1_9": "rep_30",
    "myelin_d1_8": "rep_30",  "myelin_d1_4": "rep_30",  "myelin_d2_2": "rep_30",
    "myelin_d2_3": "rep_30",  "myelin_d2_4": "rep_35",  "myelin_d2_5": "rep_36",
    "myelin_d2_6": "rep_37",  "myelin_d2_7": "rep_38",  "myelin_d2_1": "rep_38",
    "myelin_d3_1": "rep_47",  "myelin_d3_2": "rep_41",  "myelin_d3_3": "rep_43",
    "myelin_d3_4": "rep_43",
    # ── SUBACUTE ──────────────────────────────────────────────────────────────
    # liver
    "liver_d4_2":  "rep_45",  "liver_d4_3":  "rep_45",  "liver_d4_8":  "rep_45",
    "liver_d4_9":  "rep_45",  "liver_d4_5":  "rep_45",  "liver_d4_6":  "rep_45",
    "liver_d4_7":  "rep_45",  "liver_d4_10": "rep_45",  "liver_d4_1":  "rep_45",
    "liver_d4_11": "rep_45",  "liver_d5_6":  "rep_55",  "liver_d5_7":  "rep_45",
    "liver_d5_2":  "rep_45",  "liver_d5_3":  "rep_58",  "liver_d5_8":  "rep_59",
    "liver_d5_4":  "rep_60",  "liver_d5_5":  "rep_61",  "liver_d5_10": "rep_62",
    "liver_d7_18": "rep_63",  "liver_d7_14": "rep_62",  "liver_d7_15": "rep_62",
    "liver_d7_19": "rep_66",  "liver_d7_12": "rep_62",  "liver_d7_13": "rep_62",
    "liver_d7_10": "rep_69",  "liver_d7_11": "rep_62",  "liver_d7_9":  "rep_71",
    "liver_d7_16": "rep_72",  "liver_d7_17": "rep_73",
    # myelin
    "myelin_d4_4":  "rep_74", "myelin_d4_14": "rep_75", "myelin_d4_8":  "rep_76",
    "myelin_d4_9":  "rep_77", "myelin_d4_5":  "rep_78", "myelin_d4_2":  "rep_79",
    "myelin_d4_3":  "rep_80", "myelin_d4_1":  "rep_81", "myelin_d4_6":  "rep_82",
    "myelin_d4_7":  "rep_83", "myelin_d5_1":  "rep_84", "myelin_d5_6":  "rep_85",
    "myelin_d5_7":  "rep_86", "myelin_d5_5":  "rep_87", "myelin_d5_2":  "rep_88",
    "myelin_d5_3":  "rep_88", "myelin_d7_12": "rep_90", "myelin_d7_13": "rep_91",
    "myelin_d7_14": "rep_92", "myelin_d7_9":  "rep_93", "myelin_d7_8":  "rep_93",
    "myelin_d7_10": "rep_93", "myelin_d7_11": "rep_96",
    # ── LATE ──────────────────────────────────────────────────────────────────
    # liver
    "liver_d9_10":  "rep_98",  "liver_d9_11":  "rep_99",  "liver_d9_8":   "rep_100",
    "liver_d9_9":   "rep_101", "liver_d14_1":  "rep_102", "liver_d14_13": "rep_103",
    "liver_d14_12": "rep_104", "liver_d14_11": "rep_105", "liver_d14_9":  "rep_106",
    "liver_d14_10": "rep_107", "liver_d14_2":  "rep_108", "liver_d14_3":  "rep_109",
    # myelin
    "myelin_d9_1":   "rep_110", "myelin_d9_6":   "rep_111", "myelin_d9_7":   "rep_112",
    "myelin_d9_8":   "rep_113", "myelin_d9_4":   "rep_114", "myelin_d9_5":   "rep_115",
    "myelin_d9_9":   "rep_116", "myelin_d9_2":   "rep_117", "myelin_d14_10": "rep_118",
    "myelin_d14_7":  "rep_119", "myelin_d14_11": "rep_120", "myelin_d14_1":  "rep_121",
    "myelin_d14_2":  "rep_122", "myelin_d14_3":  "rep_123", "myelin_d14_12": "rep_124",
    "myelin_d14_8":  "rep_125", "myelin_d14_9":  "rep_126", "myelin_d14_13": "rep_127",
}

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"Total batches in map: {len(replicate_map_Macro)}")
print(f"Unique replicates:    {len(set(replicate_map_Macro.values()))}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. Prepare metadata
# ══════════════════════════════════════════════════════════════════════════════
adata_immune.obs[injury_col] = adata_immune.obs[sample_col].str.split("_d").str[0]
adata_immune.obs[time_col]   = "d" + adata_immune.obs[sample_col].str.split("_d").str[1].str.split("_").str[0]
adata_immune.obs["phase"]    = adata_immune.obs[time_col].map(phase_map)

# ══════════════════════════════════════════════════════════════════════════════
# 2. Subset
# ══════════════════════════════════════════════════════════════════════════════
members       = celltype_groups[celltype_value]
celltype_mask = adata_immune.obs["predicted_celltype"].isin(members)
injury_mask   = adata_immune.obs[injury_col].isin(inj_types)
phase_mask    = adata_immune.obs["phase"] == phase_to_test

adata_deg = adata_immune[celltype_mask & injury_mask & phase_mask].copy()
print(f"Cells after subsetting: {adata_deg.n_obs}")

if adata_deg.n_obs == 0:
    raise ValueError(f"No cells found for {celltype_value}, {inj_types}, {phase_to_test}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Extract raw counts from layers["counts"] ← KEY DIFFERENCE
# ══════════════════════════════════════════════════════════════════════════════
X = adata_deg.layers["counts"]  # ← raw integer counts
if hasattr(X, "toarray"):
    X = X.toarray()
else:
    X = np.asarray(X)

X_scaled = np.round(X).astype(int)  # already integers, just ensure dtype
adata_deg.obs["total_counts"] = X_scaled.sum(axis=1)

print(f"\nRaw counts check (layers['counts']):")
print(f"  min:        {X_scaled.min()}")
print(f"  max:        {X_scaled.max()}")
print(f"  mean:       {X_scaled.mean():.4f}")
print(f"  % zeros:    {(X_scaled == 0).mean() * 100:.1f}%")
print(f"  % integers: {(X_scaled == np.round(X_scaled)).mean() * 100:.1f}%")
print("✅ Using raw counts from layers['counts']")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Apply replicate map
# ══════════════════════════════════════════════════════════════════════════════
adata_deg.obs[rep_col] = adata_deg.obs[sample_col].map(replicate_map_Macro)

unmapped = adata_deg.obs.loc[adata_deg.obs[rep_col].isna(), sample_col].unique()
if len(unmapped) > 0:
    print(f"\n⚠ Unmapped batches (will be dropped): {sorted(unmapped)}")
    keep_mask = adata_deg.obs[rep_col].notna().values
    adata_deg = adata_deg[keep_mask].copy()
    X_scaled  = X_scaled[keep_mask]
else:
    print(f"\n✅ All batches mapped successfully")

print(f"Unique replicates: {adata_deg.obs[rep_col].nunique()}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Pseudobulk
# ══════════════════════════════════════════════════════════════════════════════
def make_pseudobulk(adata, X_scaled, rep_col):
    replicates  = adata.obs[rep_col].astype(str).values
    unique_reps = pd.Index(pd.unique(replicates))

    pb_rows = []
    for r in unique_reps:
        idx    = np.where(replicates == r)[0]
        summed = X_scaled[idx].sum(axis=0)
        pb_rows.append(summed)

    pb_counts = pd.DataFrame(
        pb_rows,
        index=unique_reps,
        columns=adata.var_names,
        dtype=int,
    )

    pb_meta = (
        adata.obs
        .groupby(rep_col, observed=True)
        [[injury_col, "phase", time_col]]
        .agg("first")
        .copy()
    )
    pb_meta["n_cells"]     = adata.obs.groupby(rep_col).size()
    pb_meta["mean_counts"] = adata.obs.groupby(rep_col)["total_counts"].mean().round(4)

    return pb_counts, pb_meta

pb_counts, pb_meta = make_pseudobulk(adata_deg, X_scaled, rep_col)

print(f"\npb_counts shape: {pb_counts.shape}")
print(f"\nCell counts per replicate:")
print(pb_meta[[injury_col, time_col, "n_cells"]]
      .sort_values([injury_col, time_col, "n_cells"])
      .to_string())

print(f"\nReplicates below {MIN_CELLS_PER_REPLICATE} cells:")
low = pb_meta[pb_meta["n_cells"] < MIN_CELLS_PER_REPLICATE]
print(low[[injury_col, time_col, "n_cells"]].to_string() if len(low) > 0 else "None ✅")

print(f"\nDay balance per condition:")
print(pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0))

print(f"\nSummary per condition:")
print(pb_meta.groupby(injury_col)["n_cells"]
      .agg(["count", "sum", "min", "max", "mean"])
      .rename(columns={"count": "n_reps", "sum": "total_cells", "mean": "mean_cells"})
      .round(1))

# ══════════════════════════════════════════════════════════════════════════════
# 6. QC plot
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=adata_deg.obs, x=injury_col, y="total_counts",
            palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
            order=inj_types, ax=axes[0], showfliers=False)
sns.stripplot(data=adata_deg.obs, x=injury_col, y="total_counts",
              palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
              order=inj_types, ax=axes[0], size=2, alpha=0.3, jitter=True)
axes[0].set_title("Raw counts per cell (layers['counts'])")
axes[0].set_xlabel("")

day_counts = pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0)
day_counts.T.plot(kind="bar", ax=axes[1],
                  color={inj_types[0]: "steelblue", inj_types[1]: "orange"})
axes[1].set_title("Replicates per day per condition")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("N replicates")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}QC_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 7. Covariate decision
# ══════════════════════════════════════════════════════════════════════════════
def decide_covariate_use(meta_ph, inj_types):
    counts_by_group = meta_ph.groupby(injury_col).size().to_dict()
    n_0 = int(counts_by_group.get(inj_types[0], 0))
    n_1 = int(counts_by_group.get(inj_types[1], 0))

    injury_binary = (meta_ph[injury_col] == inj_types[0]).astype(int)

    if meta_ph["mean_counts"].nunique() <= 1:
        r, p = np.nan, np.nan
    else:
        r, p = stats.pointbiserialr(injury_binary, meta_ph["mean_counts"])

    enough_reps      = (n_0 >= MIN_REPS_PER_GROUP_FOR_COVARIATE) and \
                       (n_1 >= MIN_REPS_PER_GROUP_FOR_COVARIATE)
    low_collinearity = pd.notna(r) and (abs(r) <= COVARIATE_R_THRESHOLD)
    use_covariate    = bool(enough_reps and low_collinearity)

    print(f"\nCorrelation injury ~ mean_counts: r={r:.3f}, p={p:.3f}")
    print(f"n_{inj_types[0]}={n_0}, n_{inj_types[1]}={n_1}")
    if not enough_reps:
        print(f"  → DO NOT use mean_counts covariate: too few replicates")
    elif pd.isna(r):
        print(f"  → DO NOT use mean_counts covariate: no variation")
    elif abs(r) > COVARIATE_R_THRESHOLD:
        print(f"  → DO NOT use mean_counts covariate: collinearity too high (|r|={abs(r):.3f})")
    else:
        print(f"  → USE mean_counts covariate (|r|={abs(r):.3f})")
    print(f"  → Day covariate:    {'YES' if use_day_covariate else 'NO'}")
    print(f"  → Injury covariate: {'YES' if use_injury_covariate else 'NO'}")

    return use_covariate

use_mc_covariate = decide_covariate_use(pb_meta, inj_types)

terms = []
if use_day_covariate:    terms.append("day")
if use_mc_covariate:     terms.append("mc_scaled")
if use_injury_covariate: terms.append("injury_type")
terms.append("injury")
design_str = "~ " + " + ".join(terms)
print(f"\nFinal design: {design_str}")

# ══════════════════════════════════════════════════════════════════════════════
# 8. edgeR R script
# ══════════════════════════════════════════════════════════════════════════════
R_SCRIPT_TEMPLATE = r"""
suppressPackageStartupMessages({{ library(edgeR) }})

counts_path          <- "{counts_path}"
meta_path            <- "{meta_path}"
out_path             <- "{out_path}"
use_mc_covariate     <- {use_mc_covariate}
use_day_covariate    <- {use_day_covariate}
use_injury_covariate <- {use_injury_covariate}
ref_level            <- "{ref_level}"
test_level           <- "{test_level}"

counts_df <- read.csv(counts_path, row.names = 1, check.names = FALSE)
counts    <- t(as.matrix(counts_df))
meta      <- read.csv(meta_path, row.names = 1, check.names = FALSE)
meta      <- meta[colnames(counts), , drop = FALSE]

injury      <- factor(meta$inj_type, levels = c(ref_level, test_level))
day         <- factor(meta$day)
injury_type <- factor(meta$inj_type)

terms <- c()
if (use_day_covariate)    terms <- c(terms, "day")
if (use_mc_covariate) {{
    mc_scaled <- meta$mean_counts_scaled
    terms <- c(terms, "mc_scaled")
}}
if (use_injury_covariate) terms <- c(terms, "injury_type")
terms <- c(terms, "injury")

formula_str <- paste("~", paste(terms, collapse = " + "))
cat("Formula:", formula_str, "\n")

design <- model.matrix(as.formula(formula_str))
rownames(design) <- colnames(counts)
cat("Design:", paste(colnames(design), collapse=", "), "\n")
cat("Design dimensions:", nrow(design), "x", ncol(design), "\n")

dge  <- DGEList(counts = counts, group = injury)
keep <- rowSums(dge$counts) >= 5
cat("Genes passing rowSums filter:", sum(keep), "\n")
dge  <- dge[keep, , keep.lib.sizes = FALSE]
dge  <- calcNormFactors(dge, method = "TMM")
dge  <- estimateDisp(dge, design, robust = TRUE)
fit  <- glmQLFit(dge, design, robust = TRUE)
qlf  <- glmQLFTest(fit, coef = ncol(design))

result <- topTags(qlf, n = Inf, sort.by = "PValue")$table
colnames(result)[colnames(result) == "logFC"]  <- "log2FoldChange"
colnames(result)[colnames(result) == "PValue"] <- "pvalue"
colnames(result)[colnames(result) == "FDR"]    <- "padj"
result$gene <- rownames(result)
result <- result[, c("gene", "log2FoldChange", "logCPM", "F", "pvalue", "padj")]
write.csv(result, out_path, row.names = FALSE)
cat("edgeR done:", nrow(result), "genes tested\n")
"""

# ══════════════════════════════════════════════════════════════════════════════
# 9. Run edgeR
# ══════════════════════════════════════════════════════════════════════════════
def run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
              use_day_covariate, use_injury_covariate,
              min_cells=MIN_CELLS_PER_REPLICATE):

    meta_ph   = pb_meta.copy()
    counts_ph = pb_counts.loc[meta_ph.index].copy()

    valid   = meta_ph["n_cells"] >= min_cells
    dropped = meta_ph[~valid].index.tolist()
    if dropped:
        print(f"⚠ Dropping {len(dropped)} replicate(s) with <{min_cells} cells: {dropped}")
    meta_ph   = meta_ph[valid]
    counts_ph = counts_ph.loc[meta_ph.index]

    conditions = meta_ph[injury_col].unique()
    if not set(inj_types).issubset(set(conditions)):
        raise ValueError(f"Lost a condition after filtering. Found: {conditions}\n"
                         f"Try reducing MIN_CELLS_PER_REPLICATE (currently {min_cells})")

    keep      = (counts_ph > 0).sum(axis=0) >= 2
    counts_ph = counts_ph.loc[:, keep]
    print(f"Replicates: {len(meta_ph)} | Genes after pre-filter: {keep.sum()}")
    print(f"Day balance:\n"
          f"{meta_ph.groupby([injury_col, time_col]).size().unstack(fill_value=0)}")

    meta_ph = meta_ph.copy()
    meta_ph["mean_counts_scaled"] = meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()

    with tempfile.TemporaryDirectory() as tmpdir:
        counts_path = os.path.join(tmpdir, "counts.csv")
        meta_path   = os.path.join(tmpdir, "meta.csv")
        out_path    = os.path.join(tmpdir, "result.csv")
        r_path      = os.path.join(tmpdir, "run_edgeR.R")

        counts_ph.to_csv(counts_path)
        meta_ph[[injury_col, time_col, "mean_counts", "mean_counts_scaled"]].to_csv(meta_path)

        r_script = R_SCRIPT_TEMPLATE.format(
            counts_path=counts_path,
            meta_path=meta_path,
            out_path=out_path,
            use_mc_covariate="TRUE" if use_mc_covariate else "FALSE",
            use_day_covariate="TRUE" if use_day_covariate else "FALSE",
            use_injury_covariate="TRUE" if use_injury_covariate else "FALSE",
            ref_level=inj_types[0],
            test_level=inj_types[1],
        )
        with open(r_path, "w") as f:
            f.write(r_script)

        proc = subprocess.run([RSCRIPT, "--vanilla", r_path],
                              capture_output=True, text=True)

        if proc.returncode != 0:
            print(f"R stdout:\n{proc.stdout}")
            print(f"R stderr:\n{proc.stderr}")
            raise RuntimeError("edgeR failed")

        print(proc.stdout.strip())
        res = pd.read_csv(out_path, index_col="gene")

    return res.sort_values("pvalue")

res = run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
                use_day_covariate, use_injury_covariate)
out_csv = f"{OUT_DIR}DEG_{analysis_label}.csv"
res.to_csv(out_csv)

# ══════════════════════════════════════════════════════════════════════════════
# 10. Diagnostic
# ══════════════════════════════════════════════════════════════════════════════
pval_col = "padj" if use_padj else "pvalue"
n_sig = ((res[pval_col] < pval_thresh) & (res["log2FoldChange"].abs() > lfc_thresh)).sum()

print(f"\nResults saved → {out_csv}")
print(f"Significant genes ({pval_col}<{pval_thresh}, |lfc|>{lfc_thresh}): {n_sig}")
print(f"\nGenes with pvalue < 0.05: {(res['pvalue'] < 0.05).sum()}")
print(f"Genes with pvalue < 0.01: {(res['pvalue'] < 0.01).sum()}")
print(f"Genes with padj  < 0.05: {(res['padj']  < 0.05).sum()}")
print(f"Genes with padj  < 0.10: {(res['padj']  < 0.10).sum()}")
print(f"\nTop 20 genes by pvalue:")
print(res.head(20)[["log2FoldChange", "pvalue", "padj"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(res["pvalue"].dropna(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("pvalue")
axes[0].set_ylabel("Count")
axes[0].set_title("P-value distribution\n(enriched near 0 = real signal)")
axes[1].hist(res["padj"].dropna(), bins=50, color="salmon", edgecolor="white")
axes[1].set_xlabel("padj")
axes[1].set_title("Padj distribution")
plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}pval_distribution_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 11. Volcano
# ══════════════════════════════════════════════════════════════════════════════
def volcano(df, analysis_label, inj_types, celltype_value, phase_to_test,
            design_str, use_padj=True, pval_thresh=0.05, lfc_thresh=0.58, top_n=10):

    pval_col   = "padj" if use_padj else "pvalue"
    pval_label = "padj" if use_padj else "p-value"

    df = df.dropna(subset=["log2FoldChange", pval_col]).copy()
    df["neglog10_pvalue"] = -np.log10(df["pvalue"].clip(lower=1e-300))

    sig_test = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] >  lfc_thresh)
    sig_ref  = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    other    = ~(sig_test | sig_ref)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.scatter(df.loc[other,    "log2FoldChange"], df.loc[other,    "neglog10_pvalue"],
               s=10, alpha=0.35, color="grey",      label="Not significant")
    ax.scatter(df.loc[sig_ref,  "log2FoldChange"], df.loc[sig_ref,  "neglog10_pvalue"],
               s=20, alpha=0.85, color="steelblue",
               label=f"Higher in {inj_types[0]} (n={sig_ref.sum()})")
    ax.scatter(df.loc[sig_test, "log2FoldChange"], df.loc[sig_test, "neglog10_pvalue"],
               s=20, alpha=0.85, color="orange",
               label=f"Higher in {inj_types[1]} (n={sig_test.sum()})")

    ax.axvline(-lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)
    ax.axvline( lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)

    if use_padj:
        sig_boundary = df[df[pval_col] < pval_thresh]["pvalue"]
        if len(sig_boundary) > 0:
            ax.axhline(-np.log10(sig_boundary.max()), linestyle="--",
                       linewidth=0.9, color="black", alpha=0.5)
    else:
        ax.axhline(-np.log10(pval_thresh), linestyle="--",
                   linewidth=0.9, color="black", alpha=0.5)

    texts = []
    for gene, row in df[sig_test].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))
    for gene, row in df[sig_ref].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))

    ax.set_xlabel(f"log₂ Fold Change ({inj_types[0]} vs {inj_types[1]})", fontsize=12)
    ax.set_ylabel("−log₁₀ p-value", fontsize=12)
    ax.set_title(
        f"{celltype_value} — {phase_to_test} (raw spatial counts)\n"
        f"{inj_types[0]} vs {inj_types[1]} | "
        f"{pval_label}<{pval_thresh}, |lfc|>{lfc_thresh} | design: {design_str}",
        fontsize=10
    )
    ax.legend(frameon=False)
    plt.tight_layout()
    save = f"{OUT_DIR}volcano_{analysis_label}.pdf"
    plt.savefig(save, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Volcano saved → {save}")

volcano(res, analysis_label, inj_types, celltype_value, phase_to_test,
        design_str, use_padj=use_padj, pval_thresh=pval_thresh,
        lfc_thresh=lfc_thresh, top_n=top_n_labels)

## Reclustering of immune cells from spatial

In [ ]:
immune_only_sp = sc.read_h5ad("/path/to/your/file.h5ad")

In [ ]:
immune_only_sp.obs['predicted_celltype'] = (adata_immune.obs['predicted_celltype']
                                             .reindex(immune_only_sp.obs_names)
                                             .values)

print(immune_only_sp.obs['predicted_celltype'].value_counts(dropna=False))

In [ ]:
immune_only_sp

In [ ]:
color_map = {
    'Dendritic_cells':    "#87C254",
    'MG_High_mito':       "#00B4FF",
    'MG_Homeostatic':     "#C2DEFF",
    'MG_Ifn':             "#2E31C0",
    'MG_Inflammatory':    "#1F77FF",
    'MG_Lipid_metabo':    "#93D9FF",
    'MG_Prolif':          "#00325D",
    'MG_Repair':          "#00C7E4",
    'MG_migrating':       "#00A7AE",
    'Macro_Inflammatory': "#FBB03B",
    'Macro_chemotaxis':   "#FBB03B",
    'Monocytes':          "#D62728",
    'T_cells':            "#93278F",
}
# Subset to liver, myelin and uninjured
mask = (
    immune_only_sp.obs['inj_type'].isin(['liver', 'myelin']) |
    immune_only_sp.obs['inj_type'].str.startswith('uninjured')
)
subset_sp = immune_only_sp[mask].copy()

sc.pl.umap(immune_only_sp,
           color=['predicted_celltype'],
           palette=color_map,
           frameon=False,
           legend_loc='right margin',
           ncols=3,
           size=20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cluster_col = "predicted_celltype"

categories = immune_only_sp.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 6                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = immune_only_sp.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = immune_only_sp.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = immune_only_sp.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Your genes of interest
genes = ['Ptprc', 'Itgam', 'Hexb','P2ry12','Trem2','Cd68',  'Mki67',"Cxcr4",'Msr1', 'Igf1', 'Ccr2', 'Arg1',  "Ccl7",'Cd3e']  # ← replace with your genes

cluster_order = [ 'MG_Homeostatic', 'MG_Inflammatory','MG_High_mito','MG_Lipid_metabo','MG_migrating','MG_Repair','MG_Prolif','MG_Ifn',
                  'Monocytes','Macro_Inflammatory','Macro_chemotaxis', 'Dendritic_cells','T_cells'
]

sc.settings.figdir = "/path/to/output/"

sc.pl.dotplot(
    immune_only_sp,
    var_names=genes,
    groupby='predicted_celltype',
    standard_scale='var',
    dot_min=0,  
    figsize=(15, 6),
    swap_axes=False,
    categories_order=cluster_order,
    show=True, 
    save='macrophages_dotplot.pdf'
)


In [ ]:
genes_of_interest = ['Col1a2', 'Mki67']

sc.pl.umap(immune_only_sp, color=genes_of_interest,
           frameon=False, legend_loc='right margin',
           ncols=3, cmap='Reds')

In [ ]:
sc.pl.umap(immune_only_sp,
           color=['leiden_1.2'],
           frameon=False,
           legend_loc='right margin',
           ncols=3,
           size=20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cluster_col = "leiden_IC_0.5"

categories = immune_only_sp.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 6                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = immune_only_sp.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = immune_only_sp.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = immune_only_sp.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## Tangram v2 

### Run tangram only on good cells and discriminating_genes

#### To load 

In [ ]:
proj_dir = "/path/to/project/"
save_dir = "/path/to/output/"
banksy_dir = "/path/to/output/"

immune_only_sp = sc.read_h5ad(f"{proj_dir}immune_only_tangram.h5ad")
MZ_immune_cells = sc.read_h5ad(f"{proj_dir}MZ_immune_cells_tangram.h5ad")


In [ ]:
sc.pl.umap(
    banksy_all,
    color="leiden_1.5",
    legend_loc="right margin",
    legend_fontsize=8,
    frameon=False,
    title="Banksy_all",
    size=1
)

#### Subset only good cells

In [ ]:
X = MZ_immune_cells.X
if sp.issparse(X):
    X = X.toarray()
print(f"MZ_immune_cells.X range: {X.min():.3f} to {X.max():.3f}")

X = immune_only_sp.X
if sp.issparse(X):
    X = X.toarray()
print(f"immune_only_sp.X range: {X.min():.3f} to {X.max():.3f}")

In [ ]:
# Set X to transformed (log1p normalized) for all analyses
immune_only_sp.X = immune_only_sp.layers['transformed']

# Verify
X = immune_only_sp.X
if sp.issparse(X):
    X = X.toarray()
print(f"X range: {X.min():.3f} to {X.max():.3f}")  # 0 to 3.66

In [ ]:
coarse_map = {
    'MG_Homeostatic'    : 'Microglia',
    'MG_Repair'         : 'Microglia',
    'MG_Inflammatory'   : 'Microglia',
    'MG_Ifn'            : 'Microglia',
    'MG_Lipid_metabo'   : 'Microglia',
    'MG_Prolif'         : 'Microglia',
    'MG_migrating'      : 'Microglia',
    'MG_High_mito'      : 'Microglia',
    'Macro_Inflammatory': 'Macrophage',
    'Macro_chemotaxis'  : 'Macrophage',
    'Monocytes'         : 'Monocyte',
    'T_cells'           : 'T_cell',
    'Dendritic_cells'   : 'DC'
}

MZ_immune_cells.obs['celltype_coarse'] = (
    MZ_immune_cells.obs['celltype_annotated']
    .map(coarse_map)
    .astype('category')
)
print("Coarse annotation counts:")
print(MZ_immune_cells.obs['celltype_coarse'].value_counts())


In [ ]:
# ─────────────────────────────────────────
# STEP 3 — Downsample scRNA-seq
# Keep all cells for rare populations
# Cap at 500 for abundant ones
# ─────────────────────────────────────────
subsample_map = {
    'Microglia' : 500,
    'Macrophage': 500,
    'Monocyte'  : 500,
    'DC'        : 370,  # take all
    'T_cell'    : 140   # take all
}

idx = (MZ_immune_cells.obs
       .groupby('celltype_coarse', observed=True)
       .apply(lambda x: x.sample(
           min(len(x), subsample_map[x.name]),
           random_state=42))
       .index.get_level_values(1))

MZ_sub = MZ_immune_cells[idx].copy()
print(f"\nDownsampled sc object: {MZ_sub.shape}")
print("Composition:")
print(MZ_sub.obs['celltype_coarse'].value_counts())



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

save_dir = "/path/to/output/"

# ── Choose condition ──
condition = 'myelin'  # ← change this to any condition

adata_crush = immune_only_sp[immune_only_sp.obs['inj_type'] == condition].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(adata_crush.obs['n_genes'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(adata_crush.obs['n_genes'].median(), color='red', linestyle='--', 
                label=f"median={adata_crush.obs['n_genes'].median():.0f}")
axes[0].axvline(50, color='orange', linestyle='--', label='threshold=50')
axes[0].set_xlabel('Genes detected per cell')
axes[0].set_ylabel('N cells')
axes[0].set_title(f'{condition} — all cells (n={adata_crush.n_obs})')  # ← dynamic
axes[0].legend()

for batch in adata_crush.obs['batch'].unique():
    sub = adata_crush[adata_crush.obs['batch'] == batch]
    axes[1].hist(sub.obs['n_genes'], bins=40, alpha=0.5, label=batch)
axes[1].axvline(50, color='black', linestyle='--', label='threshold=50')
axes[1].set_xlabel('Genes detected per cell')
axes[1].set_ylabel('N cells')
axes[1].set_title(f'{condition} — per batch')  # ← dynamic
axes[1].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig(f"{save_dir}{condition}_gene_distribution.pdf", bbox_inches='tight')  # ← dynamic filename
plt.show()

print(f"{condition} cells total: {adata_crush.n_obs}")
print(f"Median genes: {adata_crush.obs['n_genes'].median():.0f}")
for threshold in [20, 30, 50, 75, 100]:
    n = (adata_crush.obs['n_genes'] >= threshold).sum()
    print(f"Cells >= {threshold} genes: {n} ({n/adata_crush.n_obs*100:.1f}%)")

In [ ]:
# ─────────────────────────────────────────
# STEP 4 — Subset spatial to good quality
# cells (n_genes >= 75) across all conditions
# ─────────────────────────────────────────
adata_good = immune_only_sp[immune_only_sp.obs['n_genes'] >= 75].copy()
print(f"\nGood quality spatial cells (n_genes >= 75): {adata_good.n_obs}")
print(adata_good.obs['inj_type'].value_counts())


#### Gene selection

In [ ]:
# Step 1: get the intersection (mandatory)
shared_genes = [g for g in MZ_immune_cells.var_names 
                if g in immune_only_sp.var_names]

# Step 2: subset scRNA-seq to shared genes and rank
MZ_immune_cells_500genes = MZ_immune_cells[:, shared_genes].copy()


# 1. Genes present in both datasets
shared_genes = [g for g in immune_only_sp.var_names 
                if g in MZ_immune_cells_500genes.var_names]

# 2. Overall mean expression correlation across the 500 genes
sp_mean = np.array(immune_only_sp[:, shared_genes].X.mean(axis=0)).flatten()
sc_mean = np.array(MZ_immune_cells_500genes[:, shared_genes].X.mean(axis=0)).flatten()

gene_corr_df = pd.DataFrame({
    "gene": shared_genes,
    "mean_sc": sc_mean,
    "mean_sp": sp_mean,
    "log_sc": np.log1p(sc_mean),
    "log_sp": np.log1p(sp_mean),
})


# 3. Filter: expressed in spatial + concordant overall
from scipy.stats import spearmanr

gene_corr_df["ratio"] = gene_corr_df["log_sc"] / (gene_corr_df["log_sp"] + 1e-6)

good_genes = gene_corr_df[
    (gene_corr_df["mean_sp"] > 0.075) &     # expressed in spatial
    (gene_corr_df["mean_sc"] > 0.075) &     # expressed in scRNAseq
    (gene_corr_df["ratio"] < 10)           # not wildly discordant between datasets
]["gene"].tolist()

# Print the list
print(f"Number of selected genes: {len(good_genes)}")
print(sorted(good_genes))

# Visualize on the correlation plot
fig, ax = plt.subplots(figsize=(7, 7))

# Plot all shared genes in grey
ax.scatter(
    gene_corr_df["log_sc"], 
    gene_corr_df["log_sp"], 
    alpha=1, 
    color="lightgrey", 
    label="excluded genes"
)

# Highlight selected genes in blue
selected_df = gene_corr_df[gene_corr_df["gene"].isin(good_genes)]
ax.scatter(
    selected_df["log_sc"], 
    selected_df["log_sp"], 
    alpha=1, 
    color="black", 
    label="selected genes"
)

#for g in good_genes:  # your immune markers from option 2
    #row = gene_corr_df[gene_corr_df["gene"] == g]
    #if len(row):
        #ax.annotate(
    #g,
    #xy=(row["log_sc"].values[0], row["log_sp"].values[0]),       # point location
    #xytext=(3, 1),                                                # offset in pixels
    #textcoords="offset points",
    #fontsize=7,
    #arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

corr, pval = spearmanr(gene_corr_df["log_sc"], gene_corr_df["log_sp"])
print(f"Spearman r={corr:.2f}, p={pval:.2e}")

ax.set_xlabel("log mean expression (Zamboni scRNA-seq)")
ax.set_ylabel("log mean expression (MERSCOPE spatial)")
ax.set_title(f"Gene expression concordance\nSpearman r={corr:.2f} | {len(good_genes)} genes selected")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(f"{save_dir}gene_concordance.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Check why specific genes were excluded
genes_to_check = ['Hexb', 'Csf1r', 'Itgam', 'Cx3cr1', 'Mki67', 'Ccr2', 'Ly6c1', 'Cd3e', 'Cd8a', 'Cd4', 'P2ry12', 'Tmem119', 'Ccl2', 'Ccl7', 'Pcna']

print(f"{'Gene':<12} {'mean_sc':>8} {'mean_sp':>8} {'ratio':>8} {'in_spatial':>10} {'in_sc':>6} {'verdict':>12}")
print("-" * 70)

for g in genes_to_check:
    in_sp = g in immune_only_sp.var_names
    in_sc = g in MZ_immune_cells.var_names
    
    if g in gene_corr_df['gene'].values:
        row = gene_corr_df[gene_corr_df['gene'] == g].iloc[0]
        mean_sc = row['mean_sc']
        mean_sp = row['mean_sp']
        ratio   = row['ratio']
        
        # Diagnose why it was excluded
        if mean_sp <= 0.1:
            verdict = 'low in spatial'
        elif mean_sc <= 0.1:
            verdict = 'low in sc'
        elif ratio >= 10:
            verdict = 'discordant'
        else:
            verdict = 'passed ✓'
    else:
        mean_sc = mean_sp = ratio = float('nan')
        verdict = 'not shared'
    
    print(f"{g:<12} {mean_sc:>8.3f} {mean_sp:>8.3f} {ratio:>8.2f} {str(in_sp):>10} {str(in_sc):>6} {verdict:>12}")

all_extra = ['Ccr2', 'Ly6c2', 'Cd14', 'S100a8', 'S100a9',
             'Cd3e', 'Cd3d', 'Cd3g', 'Cd8a', 'Cd4', 'Trac',
             'P2ry12', 'Tmem119', 'Sall1', 'Fcrls', 'Olfml3',
             'Ccl2', 'Ccl7', 'Spp1', 'Gpnmb', 'Ms4a7']

in_spatial = [g for g in all_extra if g in immune_only_sp.var_names]
in_sc      = [g for g in all_extra if g in MZ_immune_cells.var_names]
in_both    = [g for g in all_extra if g in immune_only_sp.var_names 
                                   and g in MZ_immune_cells.var_names]

print(f"In spatial panel: {in_spatial}")
print(f"In scRNA-seq: {in_sc}")
print(f"In both (can use): {in_both}")

#### Check how the selected genes can separate the cells types

In [ ]:
# Add genes manually — only if present in both datasets
extra_genes = ['Hexb', 'Csf1r', 'Itgam', 'Cx3cr1', 'Mki67', 'P2ry12', 'Tmem119', 'Ccl2', 'Pcna']
extra_genes_valid = [g for g in extra_genes 
                     if g in immune_only_sp.var_names 
                     and g in MZ_immune_cells.var_names]

# Merge without duplicates
good_genes = list(set(good_genes + extra_genes_valid))

print(f"Genes after manual addition: {len(good_genes)}")
print(f"Added: {extra_genes_valid}")
print(f"Definitive list: {good_genes}")

In [ ]:
# Lowercase good_genes to match MZ_sub
#good_genes_lower = [g.lower() for g in good_genes]

genes_in_sub = [g for g in good_genes if g in MZ_sub.var_names]
print(f"Genes found: {len(genes_in_sub)}")

X = MZ_sub[:, genes_in_sub].X
if sp.issparse(X):
    X = X.toarray()

print(f"X shape: {X.shape}")
print(f"X range: {X.min():.3f} to {X.max():.3f}")

df = pd.DataFrame(X,
                  index=MZ_sub.obs_names,
                  columns=genes_in_sub)
df['celltype_coarse'] = MZ_sub.obs['celltype_coarse'].values

cluster_means = df.groupby('celltype_coarse').mean()
corr = cluster_means.T.corr()
print("\nCorrelation between cell types:")
print(corr.round(2))

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5)
ax.set_title('Cell type correlation\nacross 76 genes (scRNA-seq)')
plt.tight_layout()
plt.savefig(f"{save_dir}celltype_correlation.pdf", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Find genes that are most similar between Microglia and Macrophage
# These are the ones hurting discrimination
mg_mean   = cluster_means.loc['Microglia']
mac_mean  = cluster_means.loc['Macrophage']
mono_mean = cluster_means.loc['Monocyte']

gene_df = pd.DataFrame({
    'Microglia' : mg_mean,
    'Macrophage': mac_mean,
    'Monocyte'  : mono_mean,
    'MG_MAC_diff'  : abs(mg_mean - mac_mean),     # low = similar = bad
    'MG_MAC_ratio' : mg_mean / (mac_mean + 1e-6),  # >1 = MG specific
})

gene_df = gene_df.sort_values('MG_MAC_diff', ascending=True)

print("Genes most similar between Microglia and Macrophage (hurting discrimination):")
print(gene_df.head(20)[['Microglia', 'Macrophage', 'MG_MAC_diff', 'MG_MAC_ratio']].round(3))

print("\nGenes most different between Microglia and Macrophage (helping discrimination):")
print(gene_df.tail(20)[['Microglia', 'Macrophage', 'MG_MAC_diff', 'MG_MAC_ratio']].round(3))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# ── Plot 1: Correlation heatmap ──
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            linewidths=0.5)
axes[0].set_title('Cell type correlation\nacross 79 genes')

# ── Plot 2: Gene discrimination scatter ──
axes[1].scatter(gene_df['Microglia'], gene_df['Macrophage'],
                alpha=0.7, color='steelblue', s=50)

# Label top discriminating genes
top_disc = gene_df.tail(10)
for gene, row in top_disc.iterrows():
    axes[1].annotate(gene, (row['Microglia'], row['Macrophage']),
                     fontsize=7, xytext=(3, 3),
                     textcoords='offset points')

# Label most similar genes
top_sim = gene_df.head(5)
for gene, row in top_sim.iterrows():
    axes[1].annotate(gene, (row['Microglia'], row['Macrophage']),
                     fontsize=7, color='red', xytext=(3, 3),
                     textcoords='offset points')

axes[1].plot([0, gene_df[['Microglia','Macrophage']].max().max()],
             [0, gene_df[['Microglia','Macrophage']].max().max()],
             'r--', alpha=0.5, label='equal expression')
axes[1].set_xlabel('Microglia mean expression')
axes[1].set_ylabel('Macrophage mean expression')
axes[1].set_title('Gene expression:\nMicroglia vs Macrophage')
axes[1].legend()

# ── Plot 3: Top discriminating genes barplot ──
top20 = gene_df.tail(20).sort_values('MG_MAC_diff', ascending=True)
colors = ['#4C72B0' if r > 1 else '#DD8452' 
          for r in top20['MG_MAC_ratio']]

axes[2].barh(top20.index, top20['MG_MAC_diff'], color=colors)
axes[2].set_xlabel('|Microglia - Macrophage| expression difference')
axes[2].set_title('Top discriminating genes\n(blue=MG high, orange=MAC high)')
axes[2].axvline(0, color='black', linewidth=0.5)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='Higher in Microglia'),
                   Patch(facecolor='#DD8452', label='Higher in Macrophage')]
axes[2].legend(handles=legend_elements, fontsize=8)

# ── Plot 4: Top discriminating genes barplot (ratio)──
top20 = gene_df.tail(20).sort_values('MG_MAC_ratio', ascending=True)
colors = ['#4C72B0' if r > 1 else '#DD8452' 
          for r in top20['MG_MAC_ratio']]

axes[3].barh(top20.index, top20['MG_MAC_ratio'], color=colors)
axes[3].set_xlabel('|Microglia / Macrophage| expression difference')
axes[3].set_title('Top discriminating genes\n(blue=MG high, orange=MAC high)')
axes[3].axvline(0, color='black', linewidth=0.5)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='Higher in Microglia'),
                   Patch(facecolor='#DD8452', label='Higher in Macrophage')]
axes[3].legend(handles=legend_elements, fontsize=8)


plt.tight_layout()
plt.savefig(f"{save_dir}gene_discrimination_analysis.pdf", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Filter genes with at least 10% difference in ratio from 1
# ratio > 1.1 = higher in Microglia
# ratio < 0.9 = higher in Macrophage

mg_specific  = gene_df[gene_df['MG_MAC_ratio'] > 1.1].sort_values('MG_MAC_ratio', ascending=False)
mac_specific = gene_df[gene_df['MG_MAC_ratio'] < 0.9].sort_values('MG_MAC_ratio', ascending=True)

# Top 10 MG-specific and top 10 MAC-specific
top10_mg  = gene_df[gene_df['MG_MAC_ratio'] > 1.1].sort_values('MG_MAC_ratio', ascending=False).head(10)
top10_mac = gene_df[gene_df['MG_MAC_ratio'] < 0.9].sort_values('MG_MAC_ratio', ascending=True).head(10)

print(f"Top 10 Microglia-specific genes:")
print(top10_mg[['Microglia', 'Macrophage', 'MG_MAC_ratio']].round(3))

print(f"\nTop 10 Macrophage-specific genes:")
print(top10_mac[['Microglia', 'Macrophage', 'MG_MAC_ratio']].round(3))

# Genes with ratio between 0.9 and 1.1 — not discriminating
non_discriminating = gene_df[
    (gene_df['MG_MAC_ratio'] >= 0.9) & 
    (gene_df['MG_MAC_ratio'] <= 1.1)
].sort_values('MG_MAC_ratio')

print(f"Non-discriminating genes (ratio 0.9-1.1): {len(non_discriminating)}")
print(non_discriminating[['Microglia', 'Macrophage', 'MG_MAC_ratio']].round(3))
print("\nGene list:")
print(sorted(non_discriminating.index.tolist()))

# Combined discriminating gene list
discriminating_genes = mg_specific.index.tolist() + mac_specific.index.tolist()
print(f"\nTotal discriminating genes: {len(discriminating_genes)}")
print(sorted(discriminating_genes))


In [ ]:
# Lowercase good_genes to match MZ_sub
#discriminating_genes_lower = [g.lower() for g in discriminating_genes]

genes_in_sub = [g for g in discriminating_genes if g in MZ_sub.var_names]
print(f"Genes found: {len(genes_in_sub)}")

X = MZ_sub[:, genes_in_sub].X
if sp.issparse(X):
    X = X.toarray()

print(f"X shape: {X.shape}")
print(f"X range: {X.min():.3f} to {X.max():.3f}")

df = pd.DataFrame(X,
                  index=MZ_sub.obs_names,
                  columns=genes_in_sub)
df['celltype_coarse'] = MZ_sub.obs['celltype_coarse'].values

cluster_means = df.groupby('celltype_coarse').mean()
corr = cluster_means.T.corr()
print("\nCorrelation between cell types:")
print(corr.round(2))

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5)
ax.set_title('Cell type correlation\nacross 66 genes (scRNA-seq)')
plt.tight_layout()
plt.savefig(f"{save_dir}celltype_correlation.pdf", bbox_inches='tight', dpi=150)
plt.show()

#### Run Tangram

In [ ]:
# ─────────────────────────────────────────
# STEP 5 — pp_adatas ONCE
# Find shared genes between sc and spatial
# ─────────────────────────────────────────
tg.pp_adatas(MZ_sub, adata_good, genes=discriminating_genes)
training_genes = MZ_sub.uns['training_genes']
print(f"\nTraining genes confirmed: {len(training_genes)}")


In [ ]:
# ─────────────────────────────────────────
# STEP 6 — Build stratified batches
# Each batch proportionally represents
# all 5 cell types
# ─────────────────────────────────────────
batch_size = 500   # smaller batch since MZ_sub only has ~2010 cells
labels     = MZ_sub.obs['celltype_coarse'].values
cell_types = MZ_sub.obs['celltype_coarse'].unique()
n_cells    = MZ_sub.n_obs
n_batches  = int(np.ceil(n_cells / batch_size))

print(f"\nBuilding {n_batches} stratified batches of ~{batch_size} cells...")

# Distribute each cell type proportionally across batches
batches = [[] for _ in range(n_batches)]
for ct in cell_types:
    ct_idx = np.where(labels == ct)[0]
    np.random.shuffle(ct_idx)
    for i, chunk in enumerate(np.array_split(ct_idx, n_batches)):
        batches[i].extend(chunk.tolist())

# Shuffle within each batch
for i in range(n_batches):
    np.random.shuffle(batches[i])

print(f"Batch sizes: min={min(len(b) for b in batches)}, "
      f"max={max(len(b) for b in batches)}")

# Verify composition of all batches
print("\nComposition per batch:")
for i in range(n_batches):
    batch_labels = labels[batches[i]]
    unique, counts = np.unique(batch_labels, return_counts=True)
    print(f"  Batch {i+1}: " +
          ", ".join([f"{ct}={n}" for ct, n in zip(unique, counts)]))



In [ ]:
import numpy as np

# By copying connectivities into distances we're saying:
# "All neighbors are equally distant (distance = 1)"
# This is a simplification — in reality cells have different distances from each other. But for refined_tangram's spatial regularization it just needs to know who is a neighbor to enforce spatial smoothness, not the exact distance.

adata_good.obsm['spatial'] = np.vstack([
    adata_good.obs['x'],
    adata_good.obs['y']
]).T

print(f"spatial shape: {adata_good.obsm['spatial'].shape}")

# Then compute neighbors
sq.gr.spatial_neighbors(
    adata_good,
    coord_type="generic",
    spatial_key="spatial",
    n_neighs=10
)

print(adata_good.obsp.keys())

import squidpy as sq

# Compute spatial neighbors
# Use connectivities as distances — simplest fix
adata_good.obsp['spatial_distances'] = adata_good.obsp['spatial_connectivities'].copy()

print(f"connectivities nnz: {adata_good.obsp['spatial_connectivities'].nnz}")
print(f"distances nnz: {adata_good.obsp['spatial_distances'].nnz}")
# Should now match

In [ ]:
import traceback

# ─────────────────────────────────────────
# STEP 7 — Batch Tangram mapping
# Maps stratified batches of sc cells
# to good quality spatial cells
# ─────────────────────────────────────────

# Required by refined_tangram for ct_islands
MZ_immune_cells.obs['cell_subclass'] = MZ_immune_cells.obs['celltype_coarse']
MZ_sub.obs['cell_subclass']          = MZ_sub.obs['celltype_coarse']

print(f"\nRunning Tangram on {n_batches} stratified batches...")
all_scores = []

for i in range(n_batches):
    print(f"\n--- Batch {i+1}/{n_batches} ---")

    batch_idx = batches[i]
    MZ_batch  = MZ_sub[batch_idx].copy()

    MZ_batch.uns['training_genes'] = training_genes
    MZ_batch.uns['overlap_genes']  = training_genes

    try:
        ad_map_batch = tg.map_cells_to_space(
            adata_sc               = MZ_batch,
            adata_sp               = adata_good,
            mode                   = "cells",
            density_prior          = "rna_count_based",
            num_epochs             = 300,
            device                 = "cpu",
            verbose                = True,
            lambda_neighborhood_g1 = 0.99,
            lambda_ct_islands      = 0.5, 
            lambda_g1 = 
        )

        tg.project_cell_annotations(
            ad_map_batch,
            adata_good,
            annotation="celltype_coarse"
        )

        all_scores.append(adata_good.obsm['tangram_ct_pred'].copy())
        print(f"Batch {i+1} done ✓")

    except Exception as e:
        traceback.print_exc()
        print(f"Batch {i+1} failed: {e}")
        continue

    if (i + 1) % 3 == 0:
        pd.concat(all_scores).groupby(level=0).mean().to_csv(
            f"{save_dir}scores_checkpoint_batch{i+1}.csv")
        print(f"  → Checkpoint saved")

In [ ]:
# ─────────────────────────────────────────
# STEP 8 — Average scores across batches
# ─────────────────────────────────────────
print("\nAggregating scores...")
avg_scores = pd.concat(all_scores).groupby(level=0).mean()

ct_cols = avg_scores.columns.tolist()
adata_good.obs['predicted_celltype'] = avg_scores[ct_cols].idxmax(axis=1)
adata_good.obs['mapping_score']      = avg_scores[ct_cols].max(axis=1)

# Score distribution
scores = adata_good.obs['mapping_score']
print("\nScore distribution:")
print(scores.describe())

# Threshold at 5th percentile
threshold = scores.quantile(0.05)
print(f"\nThreshold (5th percentile): {threshold:.6f}")

adata_good.obs['predicted_celltype_filtered'] = adata_good.obs['predicted_celltype'].copy()
adata_good.obs.loc[
    adata_good.obs['mapping_score'] < threshold,
    'predicted_celltype_filtered'
] = 'low_confidence'

print("\nTangram predictions (good quality cells):")
print(adata_good.obs['predicted_celltype_filtered'].value_counts())

# Visualize score distribution with threshold
plt.figure(figsize=(7, 4))
plt.hist(adata_good.obs['mapping_score'], bins=100, 
         color='steelblue', edgecolor='white')
plt.axvline(adata_good.obs['mapping_score'].quantile(0.10), 
            color='red', linestyle='--', label='5th percentile threshold')
plt.xlabel('Max mapping score')
plt.ylabel('N cells')
plt.legend()
plt.title('Tangram score distribution')
plt.savefig(f"{save_dir}score_distribution.pdf", bbox_inches='tight')
plt.show()

#### Check the projection

In [ ]:
coarse_map = {
    'MG_Homeostatic'    : 'Microglia',
    'MG_Repair'         : 'Microglia',
    'MG_Inflammatory'   : 'Microglia',
    'MG_Ifn'            : 'Microglia',
    'MG_Lipid_metabo'   : 'Microglia',
    'MG_Prolif'         : 'Microglia',
    'MG_migrating'      : 'Microglia',
    'MG_High_mito'      : 'Microglia',
    'Macro_Inflammatory': 'Macrophage',
    'Macro_chemotaxis'  : 'Macrophage',
    'Monocytes'         : 'Monocyte',
    'T_cells'           : 'T_cell',
    'Dendritic_cells'   : 'DC'
}

MZ_immune_cells.obs['celltype_coarse'] = (
    MZ_immune_cells.obs['celltype_annotated']
    .map(coarse_map)
    .astype('category')
)

# Now check proportions
print("SC proportions (full MZ_immune_cells):")
print(MZ_immune_cells.obs['celltype_coarse'].value_counts(normalize=True).round(3) * 100)

# SC proportions — full object before downsampling
print("\nSC counts (full MZ_immune_cells):")
print(MZ_immune_cells.obs['celltype_coarse'].value_counts())

print("\nSpatial predictions (good quality cells):")
total = adata_good.obs['predicted_celltype_filtered'].value_counts()
pct   = adata_good.obs['predicted_celltype_filtered'].value_counts(normalize=True).round(3) * 100
print(pd.DataFrame({'count': total, 'percent': pct}))


In [ ]:
# ── 2. Dotplot validation ──
# Add predicted celltype to adata_good for plotting
sc.pl.dotplot(
    adata_good,
    var_names=["hexb", "itgam", 'cd3e', 'trem2', 'arg1', 'fn1', 'cd68', 'mertk', 'ccr2', 'ifng', 'mki67', 'cxcr4'],
    groupby='predicted_celltype_filtered',
    standard_scale='var',
    swap_axes=True,
    dot_min=0,
    figsize=(10, 6),
    title='Marker gene validation'
)

In [ ]:
sc.pl.umap(adata_good,
           color=['predicted_celltype_filtered'],
           frameon=False,
           legend_loc='right margin',
           ncols=3,
           size=20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cluster_col = "predicted_celltype_filtered"

categories = adata_good.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 6                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = adata_good.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = adata_good.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = adata_good.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sections to use: "uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2", 
#                  "myelin_d1_1", "myelin_d3_4", "myelin_d4_3", "myelin_d5_1", "myelin_d7_11", "myelin_d9_9", "myelin_d14_11", 
#                  "liver_d1_2", "liver_d3_4", "liver_d4_1", "liver_d5_6", "liver_d7_14", "liver_d9_11", "liver_d14_11"


# ── Choose your section ──
batch = 'myelin_d5_1'  # ← change this to any batch

# ── Check available batches ──
print("Available batches:")
print(adata_good.obs['batch'].unique().tolist())

# ── Subset ──
mask        = adata_good.obs['batch'] == batch
adata_batch = adata_good[mask]
print(f"\nCells in {batch}: {adata_batch.n_obs}")
print(adata_batch.obs['predicted_celltype_filtered'].value_counts())

# ── Plot ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

ct_colors = {
    'Microglia'     : '#4C72B0',
    'Macrophage'    : '#DD8452',
    'Monocyte'      : '#55A868',
    'DC'            : '#C44E52',
    'T_cell'        : '#8172B2',
    'low_confidence': 'lightgrey'
}

# Predicted cell types
for ct, color in ct_colors.items():
    mask_ct = adata_batch.obs['predicted_celltype_filtered'] == ct
    if mask_ct.sum() > 0:
        axes[0].scatter(
            adata_batch.obs.loc[mask_ct, 'x'],
            adata_batch.obs.loc[mask_ct, 'y'],
            c=color, s=3, label=f"{ct} ({mask_ct.sum()})", alpha=0.8
        )
axes[0].set_title(f'Predicted — {batch}', fontsize=10)
axes[0].legend(markerscale=3, fontsize=6, loc='upper right')
axes[0].invert_yaxis()
axes[0].axis('off')

# Marker genes
for ax, gene in zip(axes[1:], ['hexb', 'trem2', 'itgam']):
    if gene in adata_batch.var_names:
        expr = adata_batch[:, gene].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        sc1 = ax.scatter(
            adata_batch.obs['x'],
            adata_batch.obs['y'],
            c=expr, cmap='Reds', s=3, alpha=0.8
        )
        plt.colorbar(sc1, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f'{gene} — {batch}', fontsize=10)
    else:
        ax.set_title(f'{gene} — not in panel', fontsize=10)
    ax.invert_yaxis()
    ax.axis('off')

plt.suptitle(f'Spatial validation — {batch}', fontsize=14)
plt.tight_layout()
plt.savefig(f"{save_dir}spatial_validation_{batch}.pdf", bbox_inches='tight', dpi=150)
plt.show()

#### Fine tuning of hyperparameters

In [ ]:
# Small test — 5 epochs only to check it runs
MZ_tune = MZ_sub[batches[0]].copy()
MZ_tune.uns['training_genes'] = training_genes
MZ_tune.uns['overlap_genes']  = training_genes
MZ_tune.obs['cell_subclass']  = MZ_tune.obs['celltype_coarse']

metric = [
    "cell_map_consistency",
    "cell_map_agreement",
    "cell_map_certainty",
    "gene_expr_consistency",
    "gene_expr_correctness"
]

config = {
    "learning_rate"         : tune.loguniform(0.001, 1),
    "lambda_g1"             : tune.uniform(0, 1.0),
    "lambda_r"              : tune.loguniform(1e-20, 1e-3),
    "lambda_l2"             : tune.loguniform(1e-20, 1e-3),
    "lambda_neighborhood_g1": tune.uniform(0, 1.0),
    "lambda_ct_islands"     : tune.uniform(0, 1.0),
    "lambda_getis_ord"      : tune.uniform(0, 1.0),
}

# Create a short temp directory
os.makedirs('/tmp/ray_tg', exist_ok=True)

ray.init(_temp_dir='/tmp/ray_tg', ignore_reinit_error=True)

tuner = tg.map_cells_to_space_hyperparameter_tuning(
    MZ_tune,
    adata_good,
    metric,
    config
)

results = tuner.get_results()
print(results)


#### KNN projection

In [ ]:
# ── STEP 9 — KNN projection ── DOES NOT WORK 

immune_only_sp.var_names = immune_only_sp.var_names.str.lower()

# Convert good_genes to lowercase to match immune_only_sp var_names
# Use adata_good for training — it has the correct gene names from Tangram
knn_genes = [g for g in adata_good.var_names if g in immune_only_sp.var_names]
print(f"KNN genes available: {len(knn_genes)}")

confident_mask  = adata_good.obs['predicted_celltype_filtered'] != 'low_confidence'
confident_cells = adata_good.obs_names[confident_mask]
print(f"Confident cells for KNN training: {len(confident_cells)}")

# Train on adata_good (has Tangram gene names)
X_train = adata_good[confident_cells, knn_genes].X
y_train = adata_good.obs.loc[confident_cells, 'predicted_celltype_filtered']
if hasattr(X_train, 'toarray'):
    X_train = X_train.toarray()

knn = KNeighborsClassifier(n_neighbors=30, metric='cosine', n_jobs=-1)
knn.fit(X_train, y_train)
print("KNN trained ✓")

# Predict on immune_only_sp using same genes
X_all = immune_only_sp[:, knn_genes].X
if hasattr(X_all, 'toarray'):
    X_all = X_all.toarray()

immune_only_sp.obs['predicted_celltype'] = knn.predict(X_all)
immune_only_sp.obs['annotation_method']  = 'KNN_from_Tangram'
immune_only_sp.obs.loc[
    immune_only_sp.obs_names.isin(confident_cells),
    'annotation_method'
] = 'Tangram'

print("\nFinal predictions:")
print(immune_only_sp.obs['predicted_celltype'].value_counts())
print("\nAnnotation method:")
print(immune_only_sp.obs['annotation_method'].value_counts())

In [ ]:
sc.pl.dotplot(
    immune_only_sp,
    var_names=["hexb", "itgam", 'cd3e', 'trem2', 'arg1', 'fn1', 'cd68', 'mertk', 'ccr2', 'ifng', 'mki67', 'cxcr4'],
    groupby='predicted_celltype',  # ← use predicted_celltype not predicted_celltype_filtered
    standard_scale='var',
    swap_axes=True,
    dot_min=0,
    figsize=(10, 6),
    title='Marker gene validation',
    save='marker_validation_dotplot.pdf'
)

In [ ]:
# ─────────────────────────────────────────
# STEP 11 — Save
# ─────────────────────────────────────────
immune_only_sp.write_h5ad(f"{save_dir}immune_only_sp_annotated_v3.h5ad")
adata_good.write_h5ad(f"{save_dir}adata_good_tangram_scores_v3.h5ad")
avg_scores.to_csv(f"{save_dir}tangram_avg_scores_v3.csv")

print("\nAll files saved!")
print(f"  → {save_dir}immune_only_sp_annotated_v3.h5ad")
print(f"  → {save_dir}adata_good_tangram_scores.h5ad")
print(f"  → {save_dir}tangram_avg_scores.csv")

In [ ]:
sc.pl.umap(immune_only_sp,
           color=['predicted_celltype'],
           frameon=False,
           legend_loc='right margin',
           ncols=3,
           size=20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cluster_col = "predicted_celltype"

categories = immune_only_sp.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 6                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = immune_only_sp.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = immune_only_sp.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = immune_only_sp.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

### Save object for Seurat

In [ ]:
immune_only_sp.write_h5ad(f"{save_dir}immune_only_sp.h5ad")


In [ ]:
# Small test — 5 epochs only to check it runs
MZ_tune = MZ_sub[batches[0]].copy()
MZ_tune.uns['training_genes'] = training_genes
MZ_tune.uns['overlap_genes']  = training_genes
MZ_tune.obs['cell_subclass']  = MZ_tune.obs['celltype_coarse']

metric = [
    "cell_map_consistency",
    "cell_map_agreement",
    "cell_map_certainty",
    "gene_expr_consistency",
    "gene_expr_correctness"
]

config = {
    "learning_rate"         : tune.loguniform(0.001, 1),
    "lambda_g1"             : tune.uniform(0, 1.0),
    "lambda_r"              : tune.loguniform(1e-20, 1e-3),
    "lambda_l2"             : tune.loguniform(1e-20, 1e-3),
    "lambda_neighborhood_g1": tune.uniform(0, 1.0),
    "lambda_ct_islands"     : tune.uniform(0, 1.0),
    "lambda_getis_ord"      : tune.uniform(0, 1.0),
}

tuner = tg.map_cells_to_space_hyperparameter_tuning(
    MZ_tune,
    adata_good,
    metric,
    config
)

results = tuner.get_results()
print(results)

In [ ]:
import anndata
import scipy.sparse as sp
import numpy as np

save_dir = "/path/to/output/"

# ── Standardize gene names to lowercase ──
MZ_immune_cells.var_names     = MZ_immune_cells.var_names.str.lower()
immune_only_sp.var_names         = immune_only_sp.var_names.str.lower()

# ── Lowercase discriminating_genes too ──
discriminating_genes_lower = [g.lower() for g in discriminating_genes]

# ── Verify overlap ──
shared = [g for g in discriminating_genes_lower if g in immune_only_sp.var_names 
          and g in MZ_immune_cells.var_names]
print(f"Shared discriminating_genes: {len(shared)}")

# ── Create clean minimal objects ──
spatial_cols = ['x', 'y', 'batch', 'inj_type', 'day', 'n_genes', 'total_counts']
spatial_cols = [c for c in spatial_cols if c in immune_only_sp.obs.columns]

# Use normalized layer for spatial
X_spatial = immune_only_sp.layers['normalized']

spatial_clean = anndata.AnnData(
    X   = X_spatial,
    obs = immune_only_sp.obs[spatial_cols].copy(),
    var = immune_only_sp.var[[]].copy()
)
spatial_clean.var_names = immune_only_sp.var_names  # already lowercase

# SC object
MZ_clean = anndata.AnnData(
    X   = MZ_immune_cells.X,
    obs = MZ_immune_cells.obs[['celltype_coarse']].copy(),
    var = MZ_immune_cells.var[[]].copy()
)
MZ_clean.var_names = MZ_immune_cells.var_names  # already lowercase

# ── Verify ranges ──
X_sp = X_spatial.toarray() if sp.issparse(X_spatial) else X_spatial
X_sc = MZ_clean.X.toarray() if sp.issparse(MZ_clean.X) else MZ_clean.X
print(f"Spatial range: {X_sp.min():.3f} to {X_sp.max():.3f}")
print(f"SC range: {X_sc.min():.3f} to {X_sc.max():.3f}")

# ── Save ──
MZ_clean.write_h5ad(f"{save_dir}MZ_immune_cells_seurat.h5ad")
spatial_clean.write_h5ad(f"{save_dir}immune_only_sp_seurat.h5ad")
print("Saved!")
print(f"  → {save_dir}MZ_immune_cells_clean.h5ad")
print(f"  → {save_dir}immune_only_sp_seurat.h5ad")

In [ ]:
X = adata_immune.X
if sp.issparse(X):
    X = X.toarray()
print(f"adata_immune.X range: {X.min():.3f} to {X.max():.3f}")

In [ ]:
X = MZ_immune_cells.X
if sp.issparse(X):
    X = X.toarray()
print(f"MZ_immune_cells.X range: {X.min():.3f} to {X.max():.3f}")

In [ ]:
X = immune_only_sp.X
if sp.issparse(X):
    X = X.toarray()
print(f"immune_only_sp.X range: {X.min():.3f} to {X.max():.3f}")

## EdgerR to follow AUGR analyses

In [ ]:
import os
import subprocess
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

adata_immune = sc.read_h5ad("/path/to/your/file.h5ad")

proj_dir = "/path/to/project/"

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETERS — change only this block
# ══════════════════════════════════════════════════════════════════════════════
celltype_value       = "Macrophages"
inj_types            = ["myelin", "liver"]
phase_to_test        = "subacute"
use_day_covariate    = True
use_injury_covariate = False
MIN_CELLS_PER_REPLICATE = 10

use_padj      = False
pval_thresh   = 0.05
lfc_thresh    = 0.58
top_n_labels  = 10

OUT_DIR  = f"{proj_dir}DEG_spatial_raw_Macro_Mono/"
RSCRIPT  = "/usr/local/bin/Rscript"

COVARIATE_R_THRESHOLD            = 0.75
MIN_REPS_PER_GROUP_FOR_COVARIATE = 3
# ══════════════════════════════════════════════════════════════════════════════

analysis_label = (f"spatial_raw_{celltype_value}_"
                  f"{inj_types[0]}_vs_{inj_types[1]}_{phase_to_test}")
os.makedirs(OUT_DIR, exist_ok=True)

injury_col = "inj_type"
time_col   = "day"
sample_col = "manual_anno"
rep_col    = "replicate"

celltype_groups = {
    "Microglia":       ['MG_Homeostatic', 'MG_Activated', 'MG_Reactive', 'MG_Serpina', 'MG_Chronic'],
    "Macrophages":     ['Macro_early', 'Macro_middle', 'Macro_late', 'Macro_chronic'],
    "Dendritic_cells": ["Dendritic_cells"],
    "Monocytes":       ["Monocytes"],
}

phase_map = {
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

replicate_map_Macro = {
    # ── ACUTE ─────────────────────────────────────────────────────────────────
    # liver
    "liver_d1_1":  "rep_1",   "liver_d1_7":  "rep_2",   "liver_d1_6":  "rep_3",
    "liver_d1_5":  "rep_4",   "liver_d1_9":  "rep_4",   "liver_d1_8":  "rep_6",
    "liver_d1_4":  "rep_7",   "liver_d1_3":  "rep_8",   "liver_d1_2":  "rep_9",
    "liver_d2_4":  "rep_10",  "liver_d2_5":  "rep_11",  "liver_d2_2":  "rep_12",
    "liver_d2_3":  "rep_13",  "liver_d2_1":  "rep_11",  "liver_d2_6":  "rep_11",
    "liver_d2_7":  "rep_11",  "liver_d3_1":  "rep_17",  "liver_d3_6":  "rep_18",
    "liver_d3_7":  "rep_19",  "liver_d3_4":  "rep_19",  "liver_d3_8":  "rep_21",
    "liver_d3_5":  "rep_21",  "liver_d3_2":  "rep_23",  "liver_d3_3":  "rep_24",
    # myelin
    "myelin_d1_7": "rep_25",  "myelin_d1_6": "rep_25",  "myelin_d1_1": "rep_27",
    "myelin_d1_3": "rep_28",  "myelin_d1_2": "rep_25",  "myelin_d1_9": "rep_30",
    "myelin_d1_8": "rep_30",  "myelin_d1_4": "rep_30",  "myelin_d2_2": "rep_30",
    "myelin_d2_3": "rep_30",  "myelin_d2_4": "rep_35",  "myelin_d2_5": "rep_36",
    "myelin_d2_6": "rep_37",  "myelin_d2_7": "rep_38",  "myelin_d2_1": "rep_38",
    "myelin_d3_1": "rep_47",  "myelin_d3_2": "rep_41",  "myelin_d3_3": "rep_43",
    "myelin_d3_4": "rep_43",
    # ── SUBACUTE ──────────────────────────────────────────────────────────────
    # liver
    "liver_d4_2":  "rep_45",  "liver_d4_3":  "rep_45",  "liver_d4_8":  "rep_45",
    "liver_d4_9":  "rep_45",  "liver_d4_5":  "rep_45",  "liver_d4_6":  "rep_45",
    "liver_d4_7":  "rep_45",  "liver_d4_10": "rep_45",  "liver_d4_1":  "rep_45",
    "liver_d4_11": "rep_45",  "liver_d5_6":  "rep_55",  "liver_d5_7":  "rep_45",
    "liver_d5_2":  "rep_45",  "liver_d5_3":  "rep_58",  "liver_d5_8":  "rep_59",
    "liver_d5_4":  "rep_60",  "liver_d5_5":  "rep_61",  "liver_d5_10": "rep_62",
    "liver_d7_18": "rep_63",  "liver_d7_14": "rep_62",  "liver_d7_15": "rep_62",
    "liver_d7_19": "rep_66",  "liver_d7_12": "rep_62",  "liver_d7_13": "rep_62",
    "liver_d7_10": "rep_69",  "liver_d7_11": "rep_62",  "liver_d7_9":  "rep_71",
    "liver_d7_16": "rep_72",  "liver_d7_17": "rep_73",
    # myelin
    "myelin_d4_4":  "rep_74", "myelin_d4_14": "rep_75", "myelin_d4_8":  "rep_76",
    "myelin_d4_9":  "rep_77", "myelin_d4_5":  "rep_78", "myelin_d4_2":  "rep_79",
    "myelin_d4_3":  "rep_80", "myelin_d4_1":  "rep_81", "myelin_d4_6":  "rep_82",
    "myelin_d4_7":  "rep_83", "myelin_d5_1":  "rep_84", "myelin_d5_6":  "rep_85",
    "myelin_d5_7":  "rep_86", "myelin_d5_5":  "rep_87", "myelin_d5_2":  "rep_88",
    "myelin_d5_3":  "rep_88", "myelin_d7_12": "rep_90", "myelin_d7_13": "rep_91",
    "myelin_d7_14": "rep_92", "myelin_d7_9":  "rep_93", "myelin_d7_8":  "rep_93",
    "myelin_d7_10": "rep_93", "myelin_d7_11": "rep_96",
    # ── LATE ──────────────────────────────────────────────────────────────────
    # liver
    "liver_d9_10":  "rep_98",  "liver_d9_11":  "rep_99",  "liver_d9_8":   "rep_100",
    "liver_d9_9":   "rep_101", "liver_d14_1":  "rep_102", "liver_d14_13": "rep_103",
    "liver_d14_12": "rep_104", "liver_d14_11": "rep_105", "liver_d14_9":  "rep_106",
    "liver_d14_10": "rep_107", "liver_d14_2":  "rep_108", "liver_d14_3":  "rep_109",
    # myelin
    "myelin_d9_1":   "rep_110", "myelin_d9_6":   "rep_111", "myelin_d9_7":   "rep_112",
    "myelin_d9_8":   "rep_113", "myelin_d9_4":   "rep_114", "myelin_d9_5":   "rep_115",
    "myelin_d9_9":   "rep_116", "myelin_d9_2":   "rep_117", "myelin_d14_10": "rep_118",
    "myelin_d14_7":  "rep_119", "myelin_d14_11": "rep_120", "myelin_d14_1":  "rep_121",
    "myelin_d14_2":  "rep_122", "myelin_d14_3":  "rep_123", "myelin_d14_12": "rep_124",
    "myelin_d14_8":  "rep_125", "myelin_d14_9":  "rep_126", "myelin_d14_13": "rep_127",
}

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"Total batches in map: {len(replicate_map_Macro)}")
print(f"Unique replicates:    {len(set(replicate_map_Macro.values()))}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. Prepare metadata
# ══════════════════════════════════════════════════════════════════════════════
adata_immune.obs[injury_col] = adata_immune.obs[sample_col].str.split("_d").str[0]
adata_immune.obs[time_col]   = "d" + adata_immune.obs[sample_col].str.split("_d").str[1].str.split("_").str[0]
adata_immune.obs["phase"]    = adata_immune.obs[time_col].map(phase_map)

# ══════════════════════════════════════════════════════════════════════════════
# 2. Subset
# ══════════════════════════════════════════════════════════════════════════════
members       = celltype_groups[celltype_value]
celltype_mask = adata_immune.obs["immune_subset"].isin(members)
injury_mask   = adata_immune.obs[injury_col].isin(inj_types)
phase_mask    = adata_immune.obs["phase"] == phase_to_test

adata_deg = adata_immune[celltype_mask & injury_mask & phase_mask].copy()
print(f"Cells after subsetting: {adata_deg.n_obs}")

if adata_deg.n_obs == 0:
    raise ValueError(f"No cells found for {celltype_value}, {inj_types}, {phase_to_test}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Extract raw counts from layers["counts"] ← KEY DIFFERENCE
# ══════════════════════════════════════════════════════════════════════════════
X = adata_deg.layers["counts"]  # ← raw integer counts
if hasattr(X, "toarray"):
    X = X.toarray()
else:
    X = np.asarray(X)

X_scaled = np.round(X).astype(int)  # already integers, just ensure dtype
adata_deg.obs["total_counts"] = X_scaled.sum(axis=1)

print(f"\nRaw counts check (layers['counts']):")
print(f"  min:        {X_scaled.min()}")
print(f"  max:        {X_scaled.max()}")
print(f"  mean:       {X_scaled.mean():.4f}")
print(f"  % zeros:    {(X_scaled == 0).mean() * 100:.1f}%")
print(f"  % integers: {(X_scaled == np.round(X_scaled)).mean() * 100:.1f}%")
print("✅ Using raw counts from layers['counts']")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Apply replicate map
# ══════════════════════════════════════════════════════════════════════════════
adata_deg.obs[rep_col] = adata_deg.obs[sample_col].map(replicate_map_Macro)

unmapped = adata_deg.obs.loc[adata_deg.obs[rep_col].isna(), sample_col].unique()
if len(unmapped) > 0:
    print(f"\n⚠ Unmapped batches (will be dropped): {sorted(unmapped)}")
    keep_mask = adata_deg.obs[rep_col].notna().values
    adata_deg = adata_deg[keep_mask].copy()
    X_scaled  = X_scaled[keep_mask]
else:
    print(f"\n✅ All batches mapped successfully")

print(f"Unique replicates: {adata_deg.obs[rep_col].nunique()}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Pseudobulk
# ══════════════════════════════════════════════════════════════════════════════
def make_pseudobulk(adata, X_scaled, rep_col):
    replicates  = adata.obs[rep_col].astype(str).values
    unique_reps = pd.Index(pd.unique(replicates))

    pb_rows = []
    for r in unique_reps:
        idx    = np.where(replicates == r)[0]
        summed = X_scaled[idx].sum(axis=0)
        pb_rows.append(summed)

    pb_counts = pd.DataFrame(
        pb_rows,
        index=unique_reps,
        columns=adata.var_names,
        dtype=int,
    )

    pb_meta = (
        adata.obs
        .groupby(rep_col, observed=True)
        [[injury_col, "phase", time_col]]
        .agg("first")
        .copy()
    )
    pb_meta["n_cells"]     = adata.obs.groupby(rep_col).size()
    pb_meta["mean_counts"] = adata.obs.groupby(rep_col)["total_counts"].mean().round(4)

    return pb_counts, pb_meta

pb_counts, pb_meta = make_pseudobulk(adata_deg, X_scaled, rep_col)

print(f"\npb_counts shape: {pb_counts.shape}")
print(f"\nCell counts per replicate:")
print(pb_meta[[injury_col, time_col, "n_cells"]]
      .sort_values([injury_col, time_col, "n_cells"])
      .to_string())

print(f"\nReplicates below {MIN_CELLS_PER_REPLICATE} cells:")
low = pb_meta[pb_meta["n_cells"] < MIN_CELLS_PER_REPLICATE]
print(low[[injury_col, time_col, "n_cells"]].to_string() if len(low) > 0 else "None ✅")

print(f"\nDay balance per condition:")
print(pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0))

print(f"\nSummary per condition:")
print(pb_meta.groupby(injury_col)["n_cells"]
      .agg(["count", "sum", "min", "max", "mean"])
      .rename(columns={"count": "n_reps", "sum": "total_cells", "mean": "mean_cells"})
      .round(1))

# ══════════════════════════════════════════════════════════════════════════════
# 6. QC plot
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=adata_deg.obs, x=injury_col, y="total_counts",
            palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
            order=inj_types, ax=axes[0], showfliers=False)
sns.stripplot(data=adata_deg.obs, x=injury_col, y="total_counts",
              palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
              order=inj_types, ax=axes[0], size=2, alpha=0.3, jitter=True)
axes[0].set_title("Raw counts per cell (layers['counts'])")
axes[0].set_xlabel("")

day_counts = pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0)
day_counts.T.plot(kind="bar", ax=axes[1],
                  color={inj_types[0]: "steelblue", inj_types[1]: "orange"})
axes[1].set_title("Replicates per day per condition")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("N replicates")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}QC_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 7. Covariate decision
# ══════════════════════════════════════════════════════════════════════════════
def decide_covariate_use(meta_ph, inj_types):
    counts_by_group = meta_ph.groupby(injury_col).size().to_dict()
    n_0 = int(counts_by_group.get(inj_types[0], 0))
    n_1 = int(counts_by_group.get(inj_types[1], 0))

    injury_binary = (meta_ph[injury_col] == inj_types[0]).astype(int)

    if meta_ph["mean_counts"].nunique() <= 1:
        r, p = np.nan, np.nan
    else:
        r, p = stats.pointbiserialr(injury_binary, meta_ph["mean_counts"])

    enough_reps      = (n_0 >= MIN_REPS_PER_GROUP_FOR_COVARIATE) and \
                       (n_1 >= MIN_REPS_PER_GROUP_FOR_COVARIATE)
    low_collinearity = pd.notna(r) and (abs(r) <= COVARIATE_R_THRESHOLD)
    use_covariate    = bool(enough_reps and low_collinearity)

    print(f"\nCorrelation injury ~ mean_counts: r={r:.3f}, p={p:.3f}")
    print(f"n_{inj_types[0]}={n_0}, n_{inj_types[1]}={n_1}")
    if not enough_reps:
        print(f"  → DO NOT use mean_counts covariate: too few replicates")
    elif pd.isna(r):
        print(f"  → DO NOT use mean_counts covariate: no variation")
    elif abs(r) > COVARIATE_R_THRESHOLD:
        print(f"  → DO NOT use mean_counts covariate: collinearity too high (|r|={abs(r):.3f})")
    else:
        print(f"  → USE mean_counts covariate (|r|={abs(r):.3f})")
    print(f"  → Day covariate:    {'YES' if use_day_covariate else 'NO'}")
    print(f"  → Injury covariate: {'YES' if use_injury_covariate else 'NO'}")

    return use_covariate

use_mc_covariate = decide_covariate_use(pb_meta, inj_types)

terms = []
if use_day_covariate:    terms.append("day")
if use_mc_covariate:     terms.append("mc_scaled")
if use_injury_covariate: terms.append("injury_type")
terms.append("injury")
design_str = "~ " + " + ".join(terms)
print(f"\nFinal design: {design_str}")

# ══════════════════════════════════════════════════════════════════════════════
# 8. edgeR R script
# ══════════════════════════════════════════════════════════════════════════════
R_SCRIPT_TEMPLATE = r"""
suppressPackageStartupMessages({{ library(edgeR) }})

counts_path          <- "{counts_path}"
meta_path            <- "{meta_path}"
out_path             <- "{out_path}"
use_mc_covariate     <- {use_mc_covariate}
use_day_covariate    <- {use_day_covariate}
use_injury_covariate <- {use_injury_covariate}
ref_level            <- "{ref_level}"
test_level           <- "{test_level}"

counts_df <- read.csv(counts_path, row.names = 1, check.names = FALSE)
counts    <- t(as.matrix(counts_df))
meta      <- read.csv(meta_path, row.names = 1, check.names = FALSE)
meta      <- meta[colnames(counts), , drop = FALSE]

injury      <- factor(meta$inj_type, levels = c(ref_level, test_level))
day         <- factor(meta$day)
injury_type <- factor(meta$inj_type)

terms <- c()
if (use_day_covariate)    terms <- c(terms, "day")
if (use_mc_covariate) {{
    mc_scaled <- meta$mean_counts_scaled
    terms <- c(terms, "mc_scaled")
}}
if (use_injury_covariate) terms <- c(terms, "injury_type")
terms <- c(terms, "injury")

formula_str <- paste("~", paste(terms, collapse = " + "))
cat("Formula:", formula_str, "\n")

design <- model.matrix(as.formula(formula_str))
rownames(design) <- colnames(counts)
cat("Design:", paste(colnames(design), collapse=", "), "\n")
cat("Design dimensions:", nrow(design), "x", ncol(design), "\n")

dge  <- DGEList(counts = counts, group = injury)
keep <- rowSums(dge$counts) >= 5
cat("Genes passing rowSums filter:", sum(keep), "\n")
dge  <- dge[keep, , keep.lib.sizes = FALSE]
dge  <- calcNormFactors(dge, method = "TMM")
dge  <- estimateDisp(dge, design, robust = TRUE)
fit  <- glmQLFit(dge, design, robust = TRUE)
qlf  <- glmQLFTest(fit, coef = ncol(design))

result <- topTags(qlf, n = Inf, sort.by = "PValue")$table
colnames(result)[colnames(result) == "logFC"]  <- "log2FoldChange"
colnames(result)[colnames(result) == "PValue"] <- "pvalue"
colnames(result)[colnames(result) == "FDR"]    <- "padj"
result$gene <- rownames(result)
result <- result[, c("gene", "log2FoldChange", "logCPM", "F", "pvalue", "padj")]
write.csv(result, out_path, row.names = FALSE)
cat("edgeR done:", nrow(result), "genes tested\n")
"""

# ══════════════════════════════════════════════════════════════════════════════
# 9. Run edgeR
# ══════════════════════════════════════════════════════════════════════════════
def run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
              use_day_covariate, use_injury_covariate,
              min_cells=MIN_CELLS_PER_REPLICATE):

    meta_ph   = pb_meta.copy()
    counts_ph = pb_counts.loc[meta_ph.index].copy()

    valid   = meta_ph["n_cells"] >= min_cells
    dropped = meta_ph[~valid].index.tolist()
    if dropped:
        print(f"⚠ Dropping {len(dropped)} replicate(s) with <{min_cells} cells: {dropped}")
    meta_ph   = meta_ph[valid]
    counts_ph = counts_ph.loc[meta_ph.index]

    conditions = meta_ph[injury_col].unique()
    if not set(inj_types).issubset(set(conditions)):
        raise ValueError(f"Lost a condition after filtering. Found: {conditions}\n"
                         f"Try reducing MIN_CELLS_PER_REPLICATE (currently {min_cells})")

    keep      = (counts_ph > 0).sum(axis=0) >= 2
    counts_ph = counts_ph.loc[:, keep]
    print(f"Replicates: {len(meta_ph)} | Genes after pre-filter: {keep.sum()}")
    print(f"Day balance:\n"
          f"{meta_ph.groupby([injury_col, time_col]).size().unstack(fill_value=0)}")

    meta_ph = meta_ph.copy()
    meta_ph["mean_counts_scaled"] = meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()

    with tempfile.TemporaryDirectory() as tmpdir:
        counts_path = os.path.join(tmpdir, "counts.csv")
        meta_path   = os.path.join(tmpdir, "meta.csv")
        out_path    = os.path.join(tmpdir, "result.csv")
        r_path      = os.path.join(tmpdir, "run_edgeR.R")

        counts_ph.to_csv(counts_path)
        meta_ph[[injury_col, time_col, "mean_counts", "mean_counts_scaled"]].to_csv(meta_path)

        r_script = R_SCRIPT_TEMPLATE.format(
            counts_path=counts_path,
            meta_path=meta_path,
            out_path=out_path,
            use_mc_covariate="TRUE" if use_mc_covariate else "FALSE",
            use_day_covariate="TRUE" if use_day_covariate else "FALSE",
            use_injury_covariate="TRUE" if use_injury_covariate else "FALSE",
            ref_level=inj_types[0],
            test_level=inj_types[1],
        )
        with open(r_path, "w") as f:
            f.write(r_script)

        proc = subprocess.run([RSCRIPT, "--vanilla", r_path],
                              capture_output=True, text=True)

        if proc.returncode != 0:
            print(f"R stdout:\n{proc.stdout}")
            print(f"R stderr:\n{proc.stderr}")
            raise RuntimeError("edgeR failed")

        print(proc.stdout.strip())
        res = pd.read_csv(out_path, index_col="gene")

    return res.sort_values("pvalue")

res = run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
                use_day_covariate, use_injury_covariate)
out_csv = f"{OUT_DIR}DEG_{analysis_label}.csv"
res.to_csv(out_csv)

# ══════════════════════════════════════════════════════════════════════════════
# 10. Diagnostic
# ══════════════════════════════════════════════════════════════════════════════
pval_col = "padj" if use_padj else "pvalue"
n_sig = ((res[pval_col] < pval_thresh) & (res["log2FoldChange"].abs() > lfc_thresh)).sum()

print(f"\nResults saved → {out_csv}")
print(f"Significant genes ({pval_col}<{pval_thresh}, |lfc|>{lfc_thresh}): {n_sig}")
print(f"\nGenes with pvalue < 0.05: {(res['pvalue'] < 0.05).sum()}")
print(f"Genes with pvalue < 0.01: {(res['pvalue'] < 0.01).sum()}")
print(f"Genes with padj  < 0.05: {(res['padj']  < 0.05).sum()}")
print(f"Genes with padj  < 0.10: {(res['padj']  < 0.10).sum()}")
print(f"\nTop 20 genes by pvalue:")
print(res.head(20)[["log2FoldChange", "pvalue", "padj"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(res["pvalue"].dropna(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("pvalue")
axes[0].set_ylabel("Count")
axes[0].set_title("P-value distribution\n(enriched near 0 = real signal)")
axes[1].hist(res["padj"].dropna(), bins=50, color="salmon", edgecolor="white")
axes[1].set_xlabel("padj")
axes[1].set_title("Padj distribution")
plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}pval_distribution_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 11. Volcano
# ══════════════════════════════════════════════════════════════════════════════
def volcano(df, analysis_label, inj_types, celltype_value, phase_to_test,
            design_str, use_padj=True, pval_thresh=0.05, lfc_thresh=0.58, top_n=10):

    pval_col   = "padj" if use_padj else "pvalue"
    pval_label = "padj" if use_padj else "p-value"

    df = df.dropna(subset=["log2FoldChange", pval_col]).copy()
    df["neglog10_pvalue"] = -np.log10(df["pvalue"].clip(lower=1e-300))

    sig_test = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] >  lfc_thresh)
    sig_ref  = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    other    = ~(sig_test | sig_ref)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.scatter(df.loc[other,    "log2FoldChange"], df.loc[other,    "neglog10_pvalue"],
               s=10, alpha=0.35, color="grey",      label="Not significant")
    ax.scatter(df.loc[sig_ref,  "log2FoldChange"], df.loc[sig_ref,  "neglog10_pvalue"],
               s=20, alpha=0.85, color="steelblue",
               label=f"Higher in {inj_types[0]} (n={sig_ref.sum()})")
    ax.scatter(df.loc[sig_test, "log2FoldChange"], df.loc[sig_test, "neglog10_pvalue"],
               s=20, alpha=0.85, color="orange",
               label=f"Higher in {inj_types[1]} (n={sig_test.sum()})")

    ax.axvline(-lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)
    ax.axvline( lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)

    if use_padj:
        sig_boundary = df[df[pval_col] < pval_thresh]["pvalue"]
        if len(sig_boundary) > 0:
            ax.axhline(-np.log10(sig_boundary.max()), linestyle="--",
                       linewidth=0.9, color="black", alpha=0.5)
    else:
        ax.axhline(-np.log10(pval_thresh), linestyle="--",
                   linewidth=0.9, color="black", alpha=0.5)

    texts = []
    for gene, row in df[sig_test].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))
    for gene, row in df[sig_ref].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))

    ax.set_xlabel(f"log₂ Fold Change ({inj_types[0]} vs {inj_types[1]})", fontsize=12)
    ax.set_ylabel("−log₁₀ p-value", fontsize=12)
    ax.set_title(
        f"{celltype_value} — {phase_to_test} (raw spatial counts)\n"
        f"{inj_types[0]} vs {inj_types[1]} | "
        f"{pval_label}<{pval_thresh}, |lfc|>{lfc_thresh} | design: {design_str}",
        fontsize=10
    )
    ax.legend(frameon=False)
    plt.tight_layout()
    save = f"{OUT_DIR}volcano_{analysis_label}.pdf"
    plt.savefig(save, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Volcano saved → {save}")

volcano(res, analysis_label, inj_types, celltype_value, phase_to_test,
        design_str, use_padj=use_padj, pval_thresh=pval_thresh,
        lfc_thresh=lfc_thresh, top_n=top_n_labels)